In [2]:
%pip install -U sentence-transformers

   ---------------------------------------- 0.0/611.3 kB ? eta -:--:--
   ---------------------------------------- 611.3/611.3 kB 21.8 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [3]:
import sentence_transformers
from sentence_transformers import SentenceTransformer

print("sentence-transformers:", sentence_transformers.__version__)
print("Import: PASS")

sentence-transformers: 5.7.0
Import: PASS


In [4]:
# ==============================================================================
# TRACE THE ACE — R2 DENSE RETRIEVAL
# CELL 0 — ENVIRONMENT + R0/R1 FROZEN DEPENDENCY VERIFICATION
#
# IMPORTANT:
# This cell performs verification only.
# No embedding generation.
# No retrieval.
# No scoring.
# ==============================================================================

from pathlib import Path
import hashlib
import json
import platform
import sys

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


print("=" * 80)
print("TRACE THE ACE — R2 DENSE RETRIEVAL")
print("CELL 0 — ENVIRONMENT + R0/R1 FROZEN DEPENDENCY VERIFICATION")
print("=" * 80)


# ==============================================================================
# 1. ENVIRONMENT
# ==============================================================================

print("\n" + "=" * 80)
print("ENVIRONMENT")
print("=" * 80)

print("Python :", sys.version)
print("OS     :", platform.platform())
print("pandas :", pd.__version__)
print("numpy  :", np.__version__)
print("pyarrow:", pa.__version__)
print("CWD    :", Path.cwd())


# ==============================================================================
# 2. PROJECT PATHS
# ==============================================================================

NOTEBOOK_DIR = Path.cwd()

PROJECT_ROOT = NOTEBOOK_DIR.parent

SCRATCH_ROOT = (
    PROJECT_ROOT
    / "scratch_mastery_outputs"
)

R0_ROOT = (
    SCRATCH_ROOT
    / "02_retrieval"
    / "R0_input"
)

R1_ROOT = (
    SCRATCH_ROOT
    / "02_retrieval"
    / "R1_sparse"
)

R2_ROOT = (
    SCRATCH_ROOT
    / "02_retrieval"
    / "R2_dense"
)

R2_EMBEDDING_ROOT = (
    R2_ROOT
    / "embeddings"
)

R2_OBJECTIVE_EMBEDDING_ROOT = (
    R2_EMBEDDING_ROOT
    / "objective_embeddings"
)

R2_TURN_EMBEDDING_ROOT = (
    R2_EMBEDDING_ROOT
    / "turn_embeddings"
)

R2_FROZEN_ROOT = (
    R2_ROOT
    / "frozen"
)

R2_DIAGNOSTIC_ROOT = (
    R2_ROOT
    / "diagnostics"
)


print("\n" + "=" * 80)
print("PROJECT PATHS")
print("=" * 80)

print("PROJECT_ROOT :", PROJECT_ROOT)
print("SCRATCH_ROOT :", SCRATCH_ROOT)
print("R0_ROOT      :", R0_ROOT)
print("R1_ROOT      :", R1_ROOT)
print("R2_ROOT      :", R2_ROOT)


# ==============================================================================
# 3. R0 FROZEN ARTIFACTS
# ==============================================================================

R0_RETRIEVAL_QUERIES = (
    R0_ROOT
    / "retrieval_queries.parquet"
)

R0_SESSION_TURN_INDEX = (
    R0_ROOT
    / "session_turn_index.parquet"
)

R0_OBJECTIVE_CATALOGUE = (
    R0_ROOT
    / "objective_catalogue.parquet"
)

R0_MANIFEST = (
    R0_ROOT
    / "r0_manifest.json"
)


print("\n" + "=" * 80)
print("R0 DEPENDENCIES")
print("=" * 80)

r0_paths = {
    "retrieval_queries":
        R0_RETRIEVAL_QUERIES,

    "session_turn_index":
        R0_SESSION_TURN_INDEX,

    "objective_catalogue":
        R0_OBJECTIVE_CATALOGUE,

    "r0_manifest":
        R0_MANIFEST,
}

for name, path in r0_paths.items():

    print(
        f"{name:24s}: "
        f"{path.exists()} | {path}"
    )

assert all(
    p.exists()
    for p in r0_paths.values()
), (
    "R0 frozen dependency check failed."
)


# ==============================================================================
# 4. R0 MANIFEST VALIDATION
# ==============================================================================

with open(
    R0_MANIFEST,
    "r",
    encoding="utf-8",
) as f:

    r0_manifest = json.load(f)


assert isinstance(
    r0_manifest,
    dict,
)

print("\n" + "=" * 80)
print("R0 MANIFEST")
print("=" * 80)

print(
    "JSON valid : True"
)

print(
    "R0 ready   :",
    r0_manifest.get(
        "r0_retrieval_input_ready",
        r0_manifest.get(
            "R0_RETRIEVAL_INPUT_READY",
            "not recorded",
        ),
    ),
)

print(
    "R0 frozen  :",
    r0_manifest.get(
        "r0_frozen",
        r0_manifest.get(
            "R0_FROZEN",
            "not recorded",
        ),
    ),
)


# ==============================================================================
# 5. R1 FROZEN CANDIDATE
# ==============================================================================

R1_FROZEN_ROOT = (
    R1_ROOT
    / "frozen"
)

R1_CANDIDATES = (
    R1_FROZEN_ROOT
    / "r1_sparse_candidates.parquet"
)

R1_FREEZE_MANIFEST = (
    R1_FROZEN_ROOT
    / "r1_cell6_freeze_manifest.json"
)


print("\n" + "=" * 80)
print("R1 FROZEN DEPENDENCIES")
print("=" * 80)

print(
    "R1 candidate :",
    R1_CANDIDATES.exists(),
    "|",
    R1_CANDIDATES,
)

print(
    "R1 manifest  :",
    R1_FREEZE_MANIFEST.exists(),
    "|",
    R1_FREEZE_MANIFEST,
)

assert R1_CANDIDATES.exists(), (
    "Frozen R1 sparse candidate artifact is missing."
)

assert R1_FREEZE_MANIFEST.exists(), (
    "R1 freeze manifest is missing."
)


# ==============================================================================
# 6. R1 MANIFEST VALIDATION
# ==============================================================================

with open(
    R1_FREEZE_MANIFEST,
    "r",
    encoding="utf-8",
) as f:

    r1_manifest = json.load(f)


assert isinstance(
    r1_manifest,
    dict,
)

assert (
    r1_manifest.get("status")
    == "FROZEN"
), (
    "R1 candidate manifest does not report FROZEN."
)


print("\n" + "=" * 80)
print("R1 MANIFEST")
print("=" * 80)

print(
    "JSON valid : True"
)

print(
    "Status     :",
    r1_manifest.get("status"),
)

print(
    "Artifact   :",
    r1_manifest.get("artifact"),
)

print(
    "Rows       :",
    f"{r1_manifest.get('rows', 'unknown'):,}"
    if isinstance(
        r1_manifest.get("rows"),
        int,
    )
    else r1_manifest.get("rows"),
)


# ==============================================================================
# 7. LOAD R0 SCHEMAS ONLY
#
# We deliberately load only the R0 artifacts needed to establish the
# R2 contract. Full 6.1M-row turn data is NOT loaded here.
# ==============================================================================

retrieval_queries_schema = (
    pq.ParquetFile(
        R0_RETRIEVAL_QUERIES
    )
    .schema_arrow
)

session_turn_schema = (
    pq.ParquetFile(
        R0_SESSION_TURN_INDEX
    )
    .schema_arrow
)

objective_catalogue_schema = (
    pq.ParquetFile(
        R0_OBJECTIVE_CATALOGUE
    )
    .schema_arrow
)


print("\n" + "=" * 80)
print("R0 SCHEMA DISCOVERY")
print("=" * 80)

print(
    "retrieval_queries fields:",
    retrieval_queries_schema.names,
)

print(
    "session_turn_index fields:",
    session_turn_schema.names,
)

print(
    "objective_catalogue fields:",
    objective_catalogue_schema.names,
)


# ==============================================================================
# 8. EXACT R2 QUERY CONTRACT
# ==============================================================================

R2_QUERY_REQUIRED = {
    "response_id",
    "session_id",
    "objective_raw",
    "objective_uid",
    "fold",
}

R2_TURN_REQUIRED = {
    "session_id",
    "turn_uid",
    "turn_index",
    "role",
    "text_norm",
}

R2_OBJECTIVE_REQUIRED = {
    "objective_uid",
    "objective_raw",
}


query_missing = sorted(
    R2_QUERY_REQUIRED
    - set(
        retrieval_queries_schema.names
    )
)

turn_missing = sorted(
    R2_TURN_REQUIRED
    - set(
        session_turn_schema.names
    )
)

objective_missing = sorted(
    R2_OBJECTIVE_REQUIRED
    - set(
        objective_catalogue_schema.names
    )
)


assert query_missing == [], (
    f"R2 query contract missing: {query_missing}"
)

assert turn_missing == [], (
    f"R2 turn contract missing: {turn_missing}"
)

assert objective_missing == [], (
    f"R2 objective contract missing: {objective_missing}"
)


print("\n" + "=" * 80)
print("R2 INPUT CONTRACT")
print("=" * 80)

print(
    "Query fields   : PASS"
)

print(
    "Turn fields    : PASS"
)

print(
    "Objective fields: PASS"
)


# ==============================================================================
# 9. TARGET ISOLATION
# ==============================================================================

PROHIBITED_R2_FIELDS = {
    "target",
    "label",
    "y",
    "correct",
    "is_correct",
    "outcome",
    "positive_rate",
    "mean_target",
}

observed_prohibited = (
    PROHIBITED_R2_FIELDS
    &
    set(
        retrieval_queries_schema.names
    )
)

observed_prohibited |= (
    PROHIBITED_R2_FIELDS
    &
    set(
        objective_catalogue_schema.names
    )
)

observed_prohibited |= (
    PROHIBITED_R2_FIELDS
    &
    set(
        session_turn_schema.names
    )
)

assert observed_prohibited == set(), (
    "Target/evaluation fields detected in R2 input contract: "
    f"{sorted(observed_prohibited)}"
)


print("\n" + "=" * 80)
print("TARGET ISOLATION")
print("=" * 80)

print(
    "Prohibited fields:",
    sorted(observed_prohibited),
)

print(
    "Target leakage:",
    0,
)


# ==============================================================================
# 10. R2 CONFIGURATION
# ==============================================================================
#
# IMPORTANT:
# The exact pretrained SentenceTransformer model is deliberately declared
# here, before any embedding is generated.
#
# This makes the embedding space reproducible and allows the model/config
# to be frozen into the R2 manifest later.
# ==============================================================================

R2_MODEL_NAME = (
    "sentence-transformers/all-MiniLM-L6-v2"
)

R2_TOP_K = 50

R2_BATCH_SIZE = 256

R2_EMBEDDING_DTYPE = np.float32

R2_NORMALIZE_EMBEDDINGS = True

R2_DEVICE = "auto"


print("\n" + "=" * 80)
print("R2 DENSE RETRIEVAL CONFIGURATION")
print("=" * 80)

print(
    "Model:",
    R2_MODEL_NAME,
)

print(
    "Top-K:",
    R2_TOP_K,
)

print(
    "Batch size:",
    R2_BATCH_SIZE,
)

print(
    "Embedding dtype:",
    R2_EMBEDDING_DTYPE,
)

print(
    "L2 normalization:",
    R2_NORMALIZE_EMBEDDINGS,
)

print(
    "Device policy:",
    R2_DEVICE,
)


# ==============================================================================
# 11. VERIFY SENTENCE-TRANSFORMERS AVAILABILITY
# ==============================================================================

try:

    import sentence_transformers

    from sentence_transformers import (
        SentenceTransformer,
    )

    SENTENCE_TRANSFORMERS_AVAILABLE = True

except Exception as exc:

    SENTENCE_TRANSFORMERS_AVAILABLE = False

    print(
        "\nSentenceTransformers import failed:"
    )

    print(
        repr(exc)
    )


assert (
    SENTENCE_TRANSFORMERS_AVAILABLE
    is True
), (
    "sentence-transformers is not available "
    "in the active environment."
)


print("\n" + "=" * 80)
print("EMBEDDING LIBRARY")
print("=" * 80)

print(
    "sentence-transformers:",
    sentence_transformers.__version__,
)

print(
    "Import status:",
    "PASS",
)


# ==============================================================================
# 12. CREATE R2 DIRECTORIES
# ==============================================================================

for path in [
    R2_ROOT,
    R2_EMBEDDING_ROOT,
    R2_OBJECTIVE_EMBEDDING_ROOT,
    R2_TURN_EMBEDDING_ROOT,
    R2_FROZEN_ROOT,
    R2_DIAGNOSTIC_ROOT,
]:

    path.mkdir(
        parents=True,
        exist_ok=True,
    )


# ==============================================================================
# 13. FINAL CELL-0 STATUS
# ==============================================================================

R2_ENVIRONMENT_READY = True
R2_R0_DEPENDENCY_READY = True
R2_R1_DEPENDENCY_READY = True
R2_INPUT_SCHEMA_READY = True
R2_TARGET_FREE = True
R2_EMBEDDING_LIBRARY_READY = True


print("\n" + "=" * 80)
print("R2 CELL 0 STATUS")
print("=" * 80)

print(
    "Environment ready          :",
    R2_ENVIRONMENT_READY,
)

print(
    "R0 dependency ready        :",
    R2_R0_DEPENDENCY_READY,
)

print(
    "R1 dependency ready        :",
    R2_R1_DEPENDENCY_READY,
)

print(
    "Input schema ready         :",
    R2_INPUT_SCHEMA_READY,
)

print(
    "Target-free retrieval      :",
    R2_TARGET_FREE,
)

print(
    "Embedding library ready    :",
    R2_EMBEDDING_LIBRARY_READY,
)

print(
    "R2_CELL_0_READY            :",
    True,
)

print("=" * 80)
print("R2 CELL 0 VERIFICATION: PASS")
print("=" * 80)

TRACE THE ACE — R2 DENSE RETRIEVAL
CELL 0 — ENVIRONMENT + R0/R1 FROZEN DEPENDENCY VERIFICATION

ENVIRONMENT
Python : 3.10.20 | packaged by Anaconda, Inc. | (main, Jun 11 2026, 15:13:20) [MSC v.1942 64 bit (AMD64)]
OS     : Windows-10-10.0.26200-SP0
pandas : 2.3.3
numpy  : 2.2.5
pyarrow: 25.0.1
CWD    : d:\Competition\Trace-the-race-local\Notebooks

PROJECT PATHS
PROJECT_ROOT : d:\Competition\Trace-the-race-local
SCRATCH_ROOT : d:\Competition\Trace-the-race-local\scratch_mastery_outputs
R0_ROOT      : d:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R0_input
R1_ROOT      : d:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R1_sparse
R2_ROOT      : d:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R2_dense

R0 DEPENDENCIES
retrieval_queries       : True | d:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R0_input\retrieval_queries.parquet
session_turn_index      : True | d:\Competition\Trace-the-r

In [6]:
# ==============================================================================
# TRACE THE ACE — R2 DENSE RETRIEVAL
# CELL 1 — R0/R1 INPUT ALIGNMENT + DENSE RETRIEVAL CONTRACT
#
# PURPOSE
# -------
# Verify that the frozen R0/R1 artifacts form a valid, target-free input
# contract for R2 dense retrieval.
#
# IMPORTANT
# ---------
# This cell DOES NOT:
#   - load the 6.1M turn corpus into memory
#   - generate embeddings
#   - run retrieval
#   - recompute R1 sparse retrieval
#   - use target/labels
#
# ==============================================================================

from pathlib import Path
import json
import hashlib

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


print("=" * 80)
print("TRACE THE ACE — R2 DENSE RETRIEVAL")
print("CELL 1 — R0/R1 INPUT ALIGNMENT + DENSE RETRIEVAL CONTRACT")
print("=" * 80)


# ==============================================================================
# 1. DEPENDENCY GATE
# ==============================================================================

# ==============================================================================
# 1. DEPENDENCY GATE — RELOADABLE / KERNEL-SAFE
# ==============================================================================

# Cell 1 must remain runnable after a kernel restart.
# Therefore, do not depend on transient Python globals created by Cell 0.

from pathlib import Path
import json

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
SCRATCH_ROOT = PROJECT_ROOT / "scratch_mastery_outputs"

R0_ROOT = (
    SCRATCH_ROOT
    / "02_retrieval"
    / "R0_input"
)

R1_ROOT = (
    SCRATCH_ROOT
    / "02_retrieval"
    / "R1_sparse"
)

R2_ROOT = (
    SCRATCH_ROOT
    / "02_retrieval"
    / "R2_dense"
)

# --------------------------------------------------------------------------
# R0 artifacts
# --------------------------------------------------------------------------

R0_RETRIEVAL_QUERIES = (
    R0_ROOT / "retrieval_queries.parquet"
)

R0_SESSION_TURN_INDEX = (
    R0_ROOT / "session_turn_index.parquet"
)

R0_OBJECTIVE_CATALOGUE = (
    R0_ROOT / "objective_catalogue.parquet"
)

R0_MANIFEST = (
    R0_ROOT / "r0_manifest.json"
)

# --------------------------------------------------------------------------
# R1 frozen artifacts
# --------------------------------------------------------------------------

R1_FROZEN_ROOT = (
    R1_ROOT / "frozen"
)

R1_CANDIDATES = (
    R1_FROZEN_ROOT
    / "r1_sparse_candidates.parquet"
)

R1_FREEZE_MANIFEST = (
    R1_FROZEN_ROOT
    / "r1_cell6_freeze_manifest.json"
)

# --------------------------------------------------------------------------
# R2 roots
# --------------------------------------------------------------------------

R2_EMBEDDING_ROOT = (
    R2_ROOT / "embeddings"
)

R2_OBJECTIVE_EMBEDDING_ROOT = (
    R2_EMBEDDING_ROOT
    / "objective_embeddings"
)

R2_TURN_EMBEDDING_ROOT = (
    R2_EMBEDDING_ROOT
    / "turn_embeddings"
)

R2_FROZEN_ROOT = (
    R2_ROOT / "frozen"
)

R2_DIAGNOSTIC_ROOT = (
    R2_ROOT / "diagnostics"
)


# ==============================================================================
# R0 VERIFICATION
# ==============================================================================

R0_REQUIRED = {
    "retrieval_queries": R0_RETRIEVAL_QUERIES,
    "session_turn_index": R0_SESSION_TURN_INDEX,
    "objective_catalogue": R0_OBJECTIVE_CATALOGUE,
    "manifest": R0_MANIFEST,
}

missing_r0 = [
    name
    for name, path in R0_REQUIRED.items()
    if not path.exists()
]

assert missing_r0 == [], (
    "Missing R0 artifacts: "
    f"{missing_r0}"
)

with open(
    R0_MANIFEST,
    "r",
    encoding="utf-8",
) as f:
    r0_manifest = json.load(f)

assert isinstance(r0_manifest, dict), (
    "R0 manifest is not a JSON object."
)


# ==============================================================================
# R1 VERIFICATION
# ==============================================================================

assert R1_CANDIDATES.exists(), (
    "Frozen R1 candidate artifact is missing."
)

assert R1_FREEZE_MANIFEST.exists(), (
    "R1 freeze manifest is missing."
)

with open(
    R1_FREEZE_MANIFEST,
    "r",
    encoding="utf-8",
) as f:
    r1_manifest = json.load(f)

assert (
    r1_manifest.get("status") == "FROZEN"
), (
    "R1 candidate manifest does not report FROZEN."
)


# ==============================================================================
# R2 CONFIGURATION
# ==============================================================================

R2_MODEL_NAME = (
    "sentence-transformers/all-MiniLM-L6-v2"
)

R2_TOP_K = 50
R2_BATCH_SIZE = 256
R2_EMBEDDING_DTYPE = np.float32
R2_NORMALIZE_EMBEDDINGS = True
R2_DEVICE = "auto"


# ==============================================================================
# RECREATE CELL-0 FLAGS FROM PERSISTED DEPENDENCIES
# ==============================================================================

R2_ENVIRONMENT_READY = True
R2_R0_DEPENDENCY_READY = True
R2_R1_DEPENDENCY_READY = True
R2_INPUT_SCHEMA_READY = True
R2_TARGET_FREE = True


print("\n" + "=" * 80)
print("R2 DEPENDENCY RELOAD")
print("=" * 80)

print(
    "R0 frozen artifacts : PASS"
)

print(
    "R0 manifest         : PASS"
)

print(
    "R1 frozen candidate : PASS"
)

print(
    "R1 freeze manifest  : PASS"
)

print(
    "R2 configuration    : PASS"
)

print(
    "Kernel-safe reload  : PASS"
)

assert R2_R0_DEPENDENCY_READY is True, (
    "R0 dependency is not ready."
)

assert R2_R1_DEPENDENCY_READY is True, (
    "R1 dependency is not ready."
)

assert R2_INPUT_SCHEMA_READY is True, (
    "R2 input schema contract is not ready."
)

assert R2_TARGET_FREE is True, (
    "R2 target-isolation contract is not ready."
)

print("\nR2 Cell 0 dependency: PASS")


# ==============================================================================
# 2. R0 ARTIFACT PATHS
# ==============================================================================

print("\n" + "=" * 80)
print("R0 INPUT ARTIFACTS")
print("=" * 80)

R0_INPUTS = {
    "retrieval_queries":
        R0_RETRIEVAL_QUERIES,

    "session_turn_index":
        R0_SESSION_TURN_INDEX,

    "objective_catalogue":
        R0_OBJECTIVE_CATALOGUE,
}

for name, path in R0_INPUTS.items():

    print(
        f"{name:24s}: "
        f"{path.exists()} | {path}"
    )

assert all(
    p.exists()
    for p in R0_INPUTS.values()
), (
    "One or more R0 input artifacts are missing."
)


# ==============================================================================
# 3. R1 FROZEN ARTIFACT
# ==============================================================================

print("\n" + "=" * 80)
print("R1 FROZEN ARTIFACT")
print("=" * 80)

assert R1_CANDIDATES.exists(), (
    "Frozen R1 candidate artifact is missing."
)

print(
    "R1 sparse candidates:",
    R1_CANDIDATES
)

print(
    "Exists:",
    R1_CANDIDATES.exists()
)


# ==============================================================================
# 4. R0 ROW COUNTS
# ==============================================================================

print("\n" + "=" * 80)
print("R0 POPULATION CONTRACT")
print("=" * 80)

R0_QUERY_ROWS = pq.ParquetFile(
    R0_RETRIEVAL_QUERIES
).metadata.num_rows

R0_TURN_ROWS = pq.ParquetFile(
    R0_SESSION_TURN_INDEX
).metadata.num_rows

R0_OBJECTIVE_ROWS = pq.ParquetFile(
    R0_OBJECTIVE_CATALOGUE
).metadata.num_rows

print(
    "Retrieval queries :",
    f"{R0_QUERY_ROWS:,}"
)

print(
    "Session-turn rows:",
    f"{R0_TURN_ROWS:,}"
)

print(
    "Objectives        :",
    f"{R0_OBJECTIVE_ROWS:,}"
)

assert R0_QUERY_ROWS == 35_072, (
    f"Unexpected R0 query population: {R0_QUERY_ROWS:,}"
)

assert R0_TURN_ROWS == 6_139_854, (
    f"Unexpected R0 turn population: {R0_TURN_ROWS:,}"
)

assert R0_OBJECTIVE_ROWS == 398, (
    f"Unexpected R0 objective population: "
    f"{R0_OBJECTIVE_ROWS:,}"
)


# ==============================================================================
# 5. LOAD QUERY ARTIFACT
#
# 35k rows is small enough to load completely.
# ==============================================================================

retrieval_queries_r2 = pd.read_parquet(
    R0_RETRIEVAL_QUERIES,
    engine="pyarrow",
)

print("\n" + "=" * 80)
print("RETRIEVAL QUERY CONTRACT")
print("=" * 80)

print(
    "Rows:",
    f"{len(retrieval_queries_r2):,}"
)

print(
    "Columns:",
    list(retrieval_queries_r2.columns)
)


required_query_columns = [
    "response_id",
    "session_id",
    "objective_raw",
    "objective_uid",
    "fold",
]

assert list(
    retrieval_queries_r2.columns
) == required_query_columns, (
    "R2 retrieval query schema/order differs from "
    "the frozen R0 contract."
)

assert len(
    retrieval_queries_r2
) == 35_072

assert retrieval_queries_r2[
    "response_id"
].notna().all()

assert retrieval_queries_r2[
    "session_id"
].notna().all()

assert retrieval_queries_r2[
    "objective_uid"
].notna().all()

assert retrieval_queries_r2[
    "objective_raw"
].notna().all()

assert retrieval_queries_r2[
    "fold"
].notna().all()

assert retrieval_queries_r2[
    "response_id"
].is_unique, (
    "R2 retrieval query response_id is not unique."
)

print(
    "Response ID unique: PASS"
)


# ==============================================================================
# 6. FOLD CONTRACT
# ==============================================================================

observed_folds = sorted(
    retrieval_queries_r2[
        "fold"
    ].unique()
    .tolist()
)

print(
    "\nObserved folds:",
    observed_folds
)

assert observed_folds == [
    0, 1, 2, 3, 4
], (
    f"Unexpected fold set: {observed_folds}"
)


# ==============================================================================
# 7. LOAD OBJECTIVE CATALOGUE
# ==============================================================================

objective_catalogue_r2 = pd.read_parquet(
    R0_OBJECTIVE_CATALOGUE,
    engine="pyarrow",
)

print("\n" + "=" * 80)
print("OBJECTIVE CATALOGUE CONTRACT")
print("=" * 80)

print(
    "Rows:",
    f"{len(objective_catalogue_r2):,}"
)

print(
    "Columns:",
    list(objective_catalogue_r2.columns)
)


required_objective_columns = [
    "objective_uid",
    "objective_raw",
]

assert list(
    objective_catalogue_r2.columns
) == required_objective_columns, (
    "R2 objective catalogue schema/order differs "
    "from frozen R0 contract."
)

assert len(
    objective_catalogue_r2
) == 398

assert objective_catalogue_r2[
    "objective_uid"
].is_unique

assert objective_catalogue_r2[
    "objective_uid"
].notna().all()

assert objective_catalogue_r2[
    "objective_raw"
].notna().all()

print(
    "Objective identity: PASS"
)


# ==============================================================================
# 8. RESPONSE → OBJECTIVE ALIGNMENT
# ==============================================================================

objective_lookup = objective_catalogue_r2[
    [
        "objective_uid",
        "objective_raw",
    ]
].copy()

query_objective_check = retrieval_queries_r2[
    [
        "objective_uid",
        "objective_raw",
    ]
].merge(
    objective_lookup,
    on="objective_uid",
    how="left",
    suffixes=(
        "_query",
        "_catalogue",
    ),
)

unknown_objectives = int(
    query_objective_check[
        "objective_raw_catalogue"
    ].isna().sum()
)

identity_mismatches = int(
    (
        query_objective_check[
            "objective_raw_query"
        ]
        !=
        query_objective_check[
            "objective_raw_catalogue"
        ]
    ).sum()
)

print("\n" + "=" * 80)
print("OBJECTIVE ALIGNMENT")
print("=" * 80)

print(
    "Unknown objective IDs:",
    unknown_objectives
)

print(
    "Objective text mismatches:",
    identity_mismatches
)

assert unknown_objectives == 0, (
    "Retrieval queries contain unknown objective_uid values."
)

assert identity_mismatches == 0, (
    "Retrieval query objective text disagrees with "
    "the canonical R0 objective catalogue."
)

print(
    "Objective alignment: PASS"
)


# ==============================================================================
# 9. LOAD R0 SESSION-TURN INDEX IN COLUMNAR FORM
#
# We only need the relational columns for this contract.
# This is NOT the embedding corpus load.
# ==============================================================================

turn_columns = [
    "session_id",
    "turn_uid",
    "turn_index",
    "role",
    "text_norm",
]

session_turn_r2 = pd.read_parquet(
    R0_SESSION_TURN_INDEX,
    columns=turn_columns,
    engine="pyarrow",
)

print("\n" + "=" * 80)
print("SESSION-TURN INPUT CONTRACT")
print("=" * 80)

print(
    "Rows:",
    f"{len(session_turn_r2):,}"
)

print(
    "Columns:",
    list(session_turn_r2.columns)
)

assert len(
    session_turn_r2
) == 6_139_854

assert list(
    session_turn_r2.columns
) == turn_columns

assert session_turn_r2[
    "turn_uid"
].is_unique, (
    "turn_uid is not globally unique."
)

assert session_turn_r2[
    "session_id"
].notna().all()

assert session_turn_r2[
    "turn_uid"
].notna().all()

assert session_turn_r2[
    "text_norm"
].notna().all()

assert (
    session_turn_r2[
        "text_norm"
    ].str.strip().ne("").all()
), (
    "Blank normalized turn text detected."
)

print(
    "Turn identity/text contract: PASS"
)


# ==============================================================================
# 10. SESSION COVERAGE
# ==============================================================================

query_sessions = set(
    retrieval_queries_r2[
        "session_id"
    ].unique()
)

turn_sessions = set(
    session_turn_r2[
        "session_id"
    ].unique()
)

missing_query_sessions = (
    query_sessions
    - turn_sessions
)

print("\n" + "=" * 80)
print("SESSION COVERAGE")
print("=" * 80)

print(
    "Query sessions:",
    f"{len(query_sessions):,}"
)

print(
    "Turn sessions :",
    f"{len(turn_sessions):,}"
)

print(
    "Missing query sessions:",
    len(missing_query_sessions)
)

assert not missing_query_sessions, (
    "Some R0 retrieval-query sessions do not exist "
    "in the R0 session-turn index."
)

print(
    "Session coverage: PASS"
)


# ==============================================================================
# 11. SESSION TURN COUNTS
# ==============================================================================

query_session_counts = (
    retrieval_queries_r2[
        "session_id"
    ]
    .value_counts()
)

turn_session_counts = (
    session_turn_r2[
        "session_id"
    ]
    .value_counts()
)

invalid_query_session_counts = (
    query_session_counts[
        query_session_counts <= 0
    ]
)

invalid_turn_session_counts = (
    turn_session_counts[
        turn_session_counts <= 0
    ]
)

assert len(
    invalid_query_session_counts
) == 0

assert len(
    invalid_turn_session_counts
) == 0

print("\n" + "=" * 80)
print("SESSION CENSUS")
print("=" * 80)

print(
    "Sessions with queries:",
    f"{len(query_session_counts):,}"
)

print(
    "Sessions with turns :",
    f"{len(turn_session_counts):,}"
)

print(
    "Session census: PASS"
)


# ==============================================================================
# 12. R1 CANDIDATE SCHEMA
# ==============================================================================

r1_schema = (
    pq.ParquetFile(
        R1_CANDIDATES
    )
    .schema_arrow
)

print("\n" + "=" * 80)
print("R1 FROZEN CANDIDATE SCHEMA")
print("=" * 80)

print(
    r1_schema.names
)


required_r1_columns = {
    "response_id",
    "session_id",
    "turn_uid",
    "turn_index",
    "role",
    "word_score",
    "char_score",
    "math_score",
    "sparse_score",
}

missing_r1 = sorted(
    required_r1_columns
    - set(r1_schema.names)
)

assert missing_r1 == [], (
    f"R1 candidate artifact missing fields: {missing_r1}"
)

print(
    "R1 candidate schema: PASS"
)


# ==============================================================================
# 13. R1 CANDIDATE POPULATION
#
# Only metadata is read here.
# ==============================================================================

R1_CANDIDATE_ROWS = (
    pq.ParquetFile(
        R1_CANDIDATES
    )
    .metadata
    .num_rows
)

print(
    "R1 candidate rows:",
    f"{R1_CANDIDATE_ROWS:,}"
)

assert R1_CANDIDATE_ROWS == 1_752_048, (
    "Frozen R1 candidate population changed."
)


# ==============================================================================
# 14. TARGET ISOLATION
# ==============================================================================

all_r2_input_columns = set(
    retrieval_queries_r2.columns
) | set(
    objective_catalogue_r2.columns
) | set(
    session_turn_r2.columns
)

target_fields_found = (
    PROHIBITED_R2_FIELDS
    &
    all_r2_input_columns
)

assert target_fields_found == set(), (
    "Target/evaluation fields found in R2 input data: "
    f"{sorted(target_fields_found)}"
)

print("\n" + "=" * 80)
print("TARGET ISOLATION")
print("=" * 80)

print(
    "Target/evaluation fields:",
    sorted(target_fields_found)
)

print(
    "Target leakage: 0"
)


# ==============================================================================
# 15. R2 OUTPUT CONTRACT
# ==============================================================================

R2_OBJECTIVE_EMBEDDING_PATH = (
    R2_OBJECTIVE_EMBEDDING_ROOT
    / "objective_embeddings.npy"
)

R2_OBJECTIVE_INDEX_PATH = (
    R2_OBJECTIVE_EMBEDDING_ROOT
    / "objective_embedding_index.parquet"
)

R2_TURN_EMBEDDING_MANIFEST = (
    R2_TURN_EMBEDDING_ROOT
    / "turn_embedding_manifest.json"
)

R2_OBJECTIVE_EMBEDDING_MANIFEST = (
    R2_OBJECTIVE_EMBEDDING_ROOT
    / "objective_embedding_manifest.json"
)

R2_DENSE_CANDIDATES_PATH = (
    R2_FROZEN_ROOT
    / "r2_dense_candidates.parquet"
)

R2_MANIFEST_PATH = (
    R2_ROOT
    / "r2_manifest.json"
)


print("\n" + "=" * 80)
print("R2 OUTPUT CONTRACT")
print("=" * 80)

print(
    "Embedding root:",
    R2_EMBEDDING_ROOT
)

print(
    "Frozen root:",
    R2_FROZEN_ROOT
)

print(
    "Diagnostics root:",
    R2_DIAGNOSTIC_ROOT
)


# ==============================================================================
# 16. CREATE A SMALL CONTRACT SNAPSHOT
# ==============================================================================

R2_INPUT_CONTRACT = {
    "r0_query_rows": int(R0_QUERY_ROWS),
    "r0_turn_rows": int(R0_TURN_ROWS),
    "r0_objective_rows": int(R0_OBJECTIVE_ROWS),
    "r1_candidate_rows": int(R1_CANDIDATE_ROWS),
    "query_sessions": int(len(query_sessions)),
    "turn_sessions": int(len(turn_sessions)),
    "folds": [int(x) for x in observed_folds],
    "target_fields_found": sorted(
        target_fields_found
    ),
    "dense_model": R2_MODEL_NAME,
    "top_k": int(R2_TOP_K),
    "embedding_dtype": str(
        R2_EMBEDDING_DTYPE
    ),
    "normalize_embeddings": bool(
        R2_NORMALIZE_EMBEDDINGS
    ),
}

R2_INPUT_CONTRACT_PATH = (
    R2_ROOT
    / "r2_input_contract.json"
)

with open(
    R2_INPUT_CONTRACT_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        R2_INPUT_CONTRACT,
        f,
        indent=2,
        ensure_ascii=False,
    )


# ==============================================================================
# 17. FINAL STATUS
# ==============================================================================

R2_INPUT_ALIGNMENT_READY = True
R2_CELL_1_READY = True

print("\n" + "=" * 80)
print("R2 CELL 1 STATUS")
print("=" * 80)

print(
    "R0 population contract      : PASS"
)

print(
    "R0 objective alignment      : PASS"
)

print(
    "R0 session coverage         : PASS"
)

print(
    "R0 turn identity/text       : PASS"
)

print(
    "R1 frozen candidate         : PASS"
)

print(
    "Target isolation            : PASS"
)

print(
    "R2 input contract           : PASS"
)

print(
    "R2_CELL_1_READY             :",
    R2_CELL_1_READY,
)

print("=" * 80)
print("R2 CELL 1 — INPUT ALIGNMENT: PASS")
print("=" * 80)

TRACE THE ACE — R2 DENSE RETRIEVAL
CELL 1 — R0/R1 INPUT ALIGNMENT + DENSE RETRIEVAL CONTRACT

R2 DEPENDENCY RELOAD
R0 frozen artifacts : PASS
R0 manifest         : PASS
R1 frozen candidate : PASS
R1 freeze manifest  : PASS
R2 configuration    : PASS
Kernel-safe reload  : PASS

R2 Cell 0 dependency: PASS

R0 INPUT ARTIFACTS
retrieval_queries       : True | d:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R0_input\retrieval_queries.parquet
session_turn_index      : True | d:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R0_input\session_turn_index.parquet
objective_catalogue     : True | d:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R0_input\objective_catalogue.parquet

R1 FROZEN ARTIFACT
R1 sparse candidates: d:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R1_sparse\frozen\r1_sparse_candidates.parquet
Exists: True

R0 POPULATION CONTRACT
Retrieval queries : 35,072
Session-turn rows: 6,139

In [7]:
# ==============================================================================
# TRACE THE ACE — R2 DENSE RETRIEVAL
# CELL 2 — DENSE EMBEDDING MODEL CONTRACT + SMOKE TEST
#
# PURPOSE
# -------
# Establish and verify the exact SentenceTransformer model used by R2.
#
# THIS CELL DOES NOT:
#   - encode the full 6.1M-turn corpus
#   - perform retrieval
#   - modify R0/R1 artifacts
#   - use targets / labels
#
# It only loads the model and performs small deterministic smoke tests.
# ==============================================================================

from pathlib import Path
import json
import hashlib
import platform

import numpy as np
import pandas as pd


print("=" * 80)
print("TRACE THE ACE — R2 DENSE RETRIEVAL")
print("CELL 2 — DENSE EMBEDDING MODEL CONTRACT + SMOKE TEST")
print("=" * 80)


# ==============================================================================
# 1. DEPENDENCY GATE
# ==============================================================================

assert R2_CELL_1_READY is True, (
    "R2 Cell 1 must pass before Cell 2."
)

assert R2_INPUT_ALIGNMENT_READY is True, (
    "R2 input alignment is not ready."
)

print("\nR2 Cell 1 dependency: PASS")


# ==============================================================================
# 2. SENTENCE TRANSFORMERS IMPORT
# ==============================================================================

print("\n" + "=" * 80)
print("EMBEDDING LIBRARY")
print("=" * 80)

try:
    import sentence_transformers
    from sentence_transformers import SentenceTransformer

    SENTENCE_TRANSFORMERS_AVAILABLE = True

except Exception as exc:

    SENTENCE_TRANSFORMERS_AVAILABLE = False

    print(
        "SentenceTransformers import failed:"
    )

    print(
        repr(exc)
    )


assert SENTENCE_TRANSFORMERS_AVAILABLE is True, (
    "sentence-transformers is not available "
    "in the active environment."
)

print(
    "sentence-transformers:",
    sentence_transformers.__version__,
)

print(
    "Import status: PASS"
)


# ==============================================================================
# 3. MODEL CONTRACT
# ==============================================================================

R2_MODEL_NAME = (
    "sentence-transformers/all-MiniLM-L6-v2"
)

R2_MODEL_REVISION = None

R2_EXPECTED_EMBEDDING_DIM = 384

R2_EXPECTED_MAX_SEQUENCE_LENGTH = 256

R2_BATCH_SIZE = 256

R2_EMBEDDING_DTYPE = np.float32

R2_NORMALIZE_EMBEDDINGS = True


print("\n" + "=" * 80)
print("R2 MODEL CONTRACT")
print("=" * 80)

print(
    "Model:",
    R2_MODEL_NAME,
)

print(
    "Expected embedding dimension:",
    R2_EXPECTED_EMBEDDING_DIM,
)

print(
    "Expected max sequence length:",
    R2_EXPECTED_MAX_SEQUENCE_LENGTH,
)

print(
    "Batch size:",
    R2_BATCH_SIZE,
)

print(
    "Embedding dtype:",
    R2_EMBEDDING_DTYPE,
)

print(
    "Normalize embeddings:",
    R2_NORMALIZE_EMBEDDINGS,
)


# ==============================================================================
# 4. MODEL ROOT
#
# Keep model metadata under R2 so the exact model used by the embedding stage
# is recorded alongside the generated artifacts.
# ==============================================================================

R2_MODEL_ROOT = (
    R2_ROOT / "model"
)

R2_MODEL_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

R2_MODEL_CONTRACT_PATH = (
    R2_MODEL_ROOT
    / "r2_model_contract.json"
)


# ==============================================================================
# 5. LOAD MODEL
# ==============================================================================

print("\n" + "=" * 80)
print("LOADING SENTENCE TRANSFORMER")
print("=" * 80)

print(
    "Model:",
    R2_MODEL_NAME,
)

print(
    "This may download/load the model once."
)

r2_model = SentenceTransformer(
    R2_MODEL_NAME
)

print(
    "Model loaded: PASS"
)


# ==============================================================================
# 6. MODEL DIMENSION
# ==============================================================================

embedding_dimension = int(
    r2_model.get_sentence_embedding_dimension()
)

print("\n" + "=" * 80)
print("MODEL DIMENSION")
print("=" * 80)

print(
    "Observed dimension:",
    embedding_dimension,
)

print(
    "Expected dimension:",
    R2_EXPECTED_EMBEDDING_DIM,
)

assert embedding_dimension == (
    R2_EXPECTED_EMBEDDING_DIM
), (
    "Unexpected embedding dimension."
)

print(
    "Embedding dimension: PASS"
)


# ==============================================================================
# 7. TOKENIZER / MAX SEQUENCE LENGTH
# ==============================================================================

try:

    observed_max_seq_length = int(
        r2_model.max_seq_length
    )

except Exception:

    observed_max_seq_length = None


print("\n" + "=" * 80)
print("MODEL SEQUENCE CONTRACT")
print("=" * 80)

print(
    "Observed max sequence length:",
    observed_max_seq_length,
)

if observed_max_seq_length is not None:

    assert (
        observed_max_seq_length
        == R2_EXPECTED_MAX_SEQUENCE_LENGTH
    ), (
        "Unexpected model max sequence length."
    )

    print(
        "Sequence length: PASS"
    )

else:

    print(
        "Sequence length could not be read directly; "
        "continuing without asserting it."
    )


# ==============================================================================
# 8. MODEL DEVICE
# ==============================================================================

try:

    observed_device = str(
        r2_model.device
    )

except Exception:

    observed_device = "UNKNOWN"


print("\n" + "=" * 80)
print("MODEL DEVICE")
print("=" * 80)

print(
    "Device:",
    observed_device,
)


# ==============================================================================
# 9. SMOKE TEXTS
#
# These are intentionally small and deterministic.
# ==============================================================================

smoke_texts = [
    "Writing tenths as decimals.",
    "2 tenths.",
    "What is 3 plus 4?",
]


print("\n" + "=" * 80)
print("DENSE ENCODING SMOKE TEST")
print("=" * 80)

print(
    "Smoke texts:",
    len(smoke_texts),
)


# ==============================================================================
# 10. ENCODE WITHOUT NORMALIZATION FIRST
#
# We explicitly request normalize_embeddings=False so we can independently
# verify the normalization contract.
# ==============================================================================

smoke_raw = r2_model.encode(
    smoke_texts,
    batch_size=3,
    convert_to_numpy=True,
    normalize_embeddings=False,
    show_progress_bar=False,
)

smoke_raw = np.asarray(
    smoke_raw,
    dtype=np.float32,
)


assert smoke_raw.shape == (
    len(smoke_texts),
    R2_EXPECTED_EMBEDDING_DIM,
), (
    "Unexpected raw embedding shape."
)


assert np.isfinite(
    smoke_raw
).all(), (
    "Non-finite values found in raw embeddings."
)


print(
    "Raw shape:",
    smoke_raw.shape,
)

print(
    "Raw dtype:",
    smoke_raw.dtype,
)

print(
    "Finite values: PASS"
)


# ==============================================================================
# 11. EXPLICIT L2 NORMALIZATION
# ==============================================================================

smoke_norms = np.linalg.norm(
    smoke_raw,
    axis=1,
)

assert np.all(
    smoke_norms > 0
), (
    "Zero-norm embedding encountered."
)


smoke_embeddings = (
    smoke_raw
    /
    smoke_norms[:, None]
)


normalized_norms = np.linalg.norm(
    smoke_embeddings,
    axis=1,
)


assert np.allclose(
    normalized_norms,
    1.0,
    atol=1e-5,
), (
    "L2 normalization contract failed."
)


print(
    "Normalized norms:",
    normalized_norms,
)

print(
    "L2 normalization: PASS"
)


# ==============================================================================
# 12. DETERMINISM TEST
#
# Encode the same text twice and verify numerical equality within a tight
# tolerance. This is a smoke-level determinism check, not a claim that every
# hardware/backend combination is bit-identical.
# ==============================================================================

determinism_a = r2_model.encode(
    [smoke_texts[0]],
    batch_size=1,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

determinism_b = r2_model.encode(
    [smoke_texts[0]],
    batch_size=1,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)


determinism_a = np.asarray(
    determinism_a,
    dtype=np.float32,
)

determinism_b = np.asarray(
    determinism_b,
    dtype=np.float32,
)


deterministic_match = bool(
    np.allclose(
        determinism_a,
        determinism_b,
        atol=1e-6,
        rtol=1e-6,
    )
)


print("\n" + "=" * 80)
print("DETERMINISM SMOKE TEST")
print("=" * 80)

print(
    "Repeated encoding match:",
    deterministic_match,
)

assert deterministic_match is True, (
    "Repeated model encoding did not match."
)

print(
    "Determinism smoke test: PASS"
)


# ==============================================================================
# 13. SIMPLE COSINE SANITY CHECK
#
# Normalized vectors should produce cosine similarity via dot product.
# ==============================================================================

smoke_similarity = (
    smoke_embeddings
    @
    smoke_embeddings.T
)

assert smoke_similarity.shape == (
    len(smoke_texts),
    len(smoke_texts),
)

assert np.allclose(
    np.diag(smoke_similarity),
    1.0,
    atol=1e-5,
)

assert np.isfinite(
    smoke_similarity
).all()


print("\n" + "=" * 80)
print("COSINE SANITY TEST")
print("=" * 80)

print(
    "Similarity matrix shape:",
    smoke_similarity.shape,
)

print(
    "Diagonal ≈ 1.0:",
    bool(
        np.allclose(
            np.diag(smoke_similarity),
            1.0,
            atol=1e-5,
        )
    ),
)

print(
    "Cosine sanity: PASS"
)


# ==============================================================================
# 14. MODEL CONTRACT MANIFEST
# ==============================================================================

R2_MODEL_CONTRACT = {
    "model_name": R2_MODEL_NAME,
    "model_revision": R2_MODEL_REVISION,
    "embedding_dimension": int(
        embedding_dimension
    ),
    "max_sequence_length": (
        None
        if observed_max_seq_length is None
        else int(observed_max_seq_length)
    ),
    "batch_size": int(R2_BATCH_SIZE),
    "embedding_dtype": str(
        np.dtype(R2_EMBEDDING_DTYPE)
    ),
    "normalize_embeddings": bool(
        R2_NORMALIZE_EMBEDDINGS
    ),
    "device": observed_device,
    "sentence_transformers_version": (
        sentence_transformers.__version__
    ),
    "python_version": platform.python_version(),
}


with open(
    R2_MODEL_CONTRACT_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        R2_MODEL_CONTRACT,
        f,
        indent=2,
        ensure_ascii=False,
    )


assert R2_MODEL_CONTRACT_PATH.exists(), (
    "R2 model contract was not written."
)


# ==============================================================================
# 15. FINAL STATUS
# ==============================================================================

R2_MODEL_READY = True
R2_CELL_2_READY = True

print("\n" + "=" * 80)
print("R2 CELL 2 STATUS")
print("=" * 80)

print(
    "SentenceTransformers available : PASS"
)

print(
    "Model loaded                    : PASS"
)

print(
    "Embedding dimension             : PASS"
)

print(
    "Sequence contract               : PASS"
)

print(
    "L2 normalization                : PASS"
)

print(
    "Determinism smoke test          : PASS"
)

print(
    "Cosine sanity                   : PASS"
)

print(
    "Model contract persisted        : PASS"
)

print(
    "R2_MODEL_READY                  :",
    R2_MODEL_READY,
)

print(
    "R2_CELL_2_READY                 :",
    R2_CELL_2_READY,
)

print("=" * 80)
print("R2 CELL 2 — DENSE MODEL CONTRACT: PASS")
print("=" * 80)

TRACE THE ACE — R2 DENSE RETRIEVAL
CELL 2 — DENSE EMBEDDING MODEL CONTRACT + SMOKE TEST

R2 Cell 1 dependency: PASS

EMBEDDING LIBRARY
sentence-transformers: 5.7.0
Import status: PASS

R2 MODEL CONTRACT
Model: sentence-transformers/all-MiniLM-L6-v2
Expected embedding dimension: 384
Expected max sequence length: 256
Batch size: 256
Embedding dtype: <class 'numpy.float32'>
Normalize embeddings: True

LOADING SENTENCE TRANSFORMER
Model: sentence-transformers/all-MiniLM-L6-v2
This may download/load the model once.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded: PASS

MODEL DIMENSION
Observed dimension: 384
Expected dimension: 384
Embedding dimension: PASS

MODEL SEQUENCE CONTRACT
Observed max sequence length: 256
Sequence length: PASS

MODEL DEVICE
Device: cpu

DENSE ENCODING SMOKE TEST
Smoke texts: 3
Raw shape: (3, 384)
Raw dtype: float32
Finite values: PASS
Normalized norms: [1. 1. 1.]
L2 normalization: PASS

DETERMINISM SMOKE TEST
Repeated encoding match: True
Determinism smoke test: PASS


C:\Users\USER\AppData\Local\Temp\ipykernel_7412\1893857686.py:199: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  r2_model.get_sentence_embedding_dimension()



COSINE SANITY TEST
Similarity matrix shape: (3, 3)
Diagonal ≈ 1.0: True
Cosine sanity: PASS

R2 CELL 2 STATUS
SentenceTransformers available : PASS
Model loaded                    : PASS
Embedding dimension             : PASS
Sequence contract               : PASS
L2 normalization                : PASS
Determinism smoke test          : PASS
Cosine sanity                   : PASS
Model contract persisted        : PASS
R2_MODEL_READY                  : True
R2_CELL_2_READY                 : True
R2 CELL 2 — DENSE MODEL CONTRACT: PASS


In [8]:
# ==============================================================================
# TRACE THE ACE — R2 DENSE RETRIEVAL
# CELL 3 — RESUMABLE TURN EMBEDDING PRECOMPUTE
#
# PURPOSE
# -------
# Encode all 6,139,854 canonical R0 turns into a disk-backed dense embedding
# matrix without requiring the entire embedding matrix in RAM.
#
# DESIGN
# ------
# - Source order is the canonical R0 session_turn_index order.
# - One embedding row corresponds exactly to one source turn row.
# - Embeddings are L2-normalized float32 vectors.
# - Final embedding matrix is stored as NumPy memmap.
# - Progress is checkpointed only after a complete batch is written/flushed.
# - A kernel crash/restart resumes from the last committed row.
#
# IMPORTANT
# ---------
# This cell does NOT perform retrieval.
# This cell does NOT use targets/labels.
# This cell does NOT modify R0 or R1 artifacts.
# ==============================================================================

from pathlib import Path
import json
import time
import gc
import hashlib
import platform

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


print("=" * 80)
print("TRACE THE ACE — R2 DENSE RETRIEVAL")
print("CELL 3 — RESUMABLE TURN EMBEDDING PRECOMPUTE")
print("=" * 80)


# ==============================================================================
# 1. RELOAD-SAFE DEPENDENCY GATE
# ==============================================================================

assert R2_CELL_2_READY is True, (
    "R2 Cell 2 must pass before Cell 3."
)

assert R2_MODEL_READY is True, (
    "R2 embedding model is not ready."
)

print("\nR2 Cell 2 dependency: PASS")


# ==============================================================================
# 2. RELOAD MODEL IF KERNEL WAS RESTARTED
# ==============================================================================

try:
    r2_model
except NameError:

    print(
        "\nR2 model object not present in kernel."
    )

    print(
        "Reloading model from persisted contract..."
    )

    from sentence_transformers import SentenceTransformer

    r2_model = SentenceTransformer(
        R2_MODEL_NAME
    )

    print(
        "Model reloaded: PASS"
    )


# ==============================================================================
# 3. SOURCE ARTIFACT
# ==============================================================================

R2_TURN_SOURCE = (
    R0_SESSION_TURN_INDEX
)

assert R2_TURN_SOURCE.exists(), (
    "R0 session-turn index is missing."
)


turn_source_pf = pq.ParquetFile(
    R2_TURN_SOURCE
)

R2_TURN_ROWS = int(
    turn_source_pf.metadata.num_rows
)

assert R2_TURN_ROWS == 6_139_854, (
    f"Unexpected R0 turn population: "
    f"{R2_TURN_ROWS:,}"
)


# ==============================================================================
# 4. OUTPUT PATHS
# ==============================================================================

R2_TURN_EMBEDDING_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

R2_TURN_EMBEDDINGS_PATH = (
    R2_TURN_EMBEDDING_ROOT
    / "turn_embeddings.float32.memmap"
)

R2_TURN_EMBEDDING_CHECKPOINT = (
    R2_TURN_EMBEDDING_ROOT
    / "turn_embedding_checkpoint.json"
)

R2_TURN_EMBEDDING_MANIFEST = (
    R2_TURN_EMBEDDING_ROOT
    / "turn_embedding_manifest.json"
)

R2_TURN_EMBEDDING_INDEX = (
    R2_TURN_EMBEDDING_ROOT
    / "turn_embedding_index.parquet"
)


print("\n" + "=" * 80)
print("R2 TURN EMBEDDING PATHS")
print("=" * 80)

print(
    "Source:",
    R2_TURN_SOURCE,
)

print(
    "Embedding matrix:",
    R2_TURN_EMBEDDINGS_PATH,
)

print(
    "Checkpoint:",
    R2_TURN_EMBEDDING_CHECKPOINT,
)

print(
    "Manifest:",
    R2_TURN_EMBEDDING_MANIFEST,
)

print(
    "Index:",
    R2_TURN_EMBEDDING_INDEX,
)


# ==============================================================================
# 5. EMBEDDING CONTRACT
# ==============================================================================

R2_EMBEDDING_DIM = int(
    r2_model.get_sentence_embedding_dimension()
)

assert R2_EMBEDDING_DIM == 384, (
    f"Unexpected embedding dimension: "
    f"{R2_EMBEDDING_DIM}"
)

R2_EMBEDDING_DTYPE = np.float32

R2_EMBEDDING_BATCH_ROWS = 10_000

R2_MODEL_BATCH_SIZE = 256

R2_NORMALIZE_EMBEDDINGS = True


print("\n" + "=" * 80)
print("EMBEDDING CONTRACT")
print("=" * 80)

print(
    "Turn rows:",
    f"{R2_TURN_ROWS:,}",
)

print(
    "Embedding dimension:",
    R2_EMBEDDING_DIM,
)

print(
    "Embedding dtype:",
    R2_EMBEDDING_DTYPE,
)

print(
    "Outer checkpoint batch:",
    f"{R2_EMBEDDING_BATCH_ROWS:,}",
)

print(
    "Model batch size:",
    R2_MODEL_BATCH_SIZE,
)

print(
    "L2 normalized:",
    R2_NORMALIZE_EMBEDDINGS,
)


# ==============================================================================
# 6. EXPECTED DISK SIZE
# ==============================================================================

R2_EXPECTED_EMBEDDING_BYTES = (
    R2_TURN_ROWS
    * R2_EMBEDDING_DIM
    * np.dtype(
        R2_EMBEDDING_DTYPE
    ).itemsize
)

R2_EXPECTED_EMBEDDING_GB = (
    R2_EXPECTED_EMBEDDING_BYTES
    / (1024 ** 3)
)


print(
    "\nExpected embedding storage:",
    f"{R2_EXPECTED_EMBEDDING_GB:.2f} GiB",
)


# ==============================================================================
# 7. SOURCE FINGERPRINT
#
# We fingerprint the source artifact using file metadata rather than hashing
# the entire multi-million-row parquet file before every run.
# ==============================================================================

source_stat = R2_TURN_SOURCE.stat()

R2_SOURCE_FINGERPRINT = {
    "path": str(
        R2_TURN_SOURCE.resolve()
    ),
    "size_bytes": int(
        source_stat.st_size
    ),
    "mtime_ns": int(
        source_stat.st_mtime_ns
    ),
    "rows": int(
        R2_TURN_ROWS
    ),
}


# ==============================================================================
# 8. CHECKPOINT HELPERS
# ==============================================================================

def write_json_atomic(
    path,
    payload,
):
    """
    Atomically write a small JSON checkpoint/manifest.
    """

    path = Path(path)

    tmp = path.with_name(
        path.name + ".tmp"
    )

    with open(
        tmp,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            payload,
            f,
            indent=2,
            ensure_ascii=False,
        )

        f.flush()

    tmp.replace(path)


def load_checkpoint(
    path,
):
    path = Path(path)

    if not path.exists():
        return None

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as f:

        return json.load(f)


# ==============================================================================
# 9. DETERMINE RESUME STATE
# ==============================================================================

checkpoint = load_checkpoint(
    R2_TURN_EMBEDDING_CHECKPOINT
)


resume_row = 0


if checkpoint is not None:

    print("\n" + "=" * 80)
    print("EXISTING CHECKPOINT")
    print("=" * 80)

    print(
        json.dumps(
            checkpoint,
            indent=2,
            ensure_ascii=False,
        )
    )

    assert (
        checkpoint.get(
            "source_fingerprint"
        )
        ==
        R2_SOURCE_FINGERPRINT
    ), (
        "Existing checkpoint belongs to a different "
        "source artifact."
    )

    assert (
        checkpoint.get(
            "embedding_dimension"
        )
        == R2_EMBEDDING_DIM
    )

    assert (
        checkpoint.get(
            "dtype"
        )
        ==
        str(
            np.dtype(
                R2_EMBEDDING_DTYPE
            )
        )
    )

    resume_row = int(
        checkpoint.get(
            "completed_rows",
            0,
        )
    )

    assert (
        0
        <= resume_row
        <= R2_TURN_ROWS
    )

    print(
        "\nResume row:",
        f"{resume_row:,}",
    )

else:

    print("\n" + "=" * 80)
    print("NO EXISTING CHECKPOINT")
    print("=" * 80)

    print(
        "Starting from row 0."
    )


# ==============================================================================
# 10. MEMMAP CREATION / VALIDATION
# ==============================================================================

expected_memmap_size = (
    R2_EXPECTED_EMBEDDING_BYTES
)


if R2_TURN_EMBEDDINGS_PATH.exists():

    actual_size = (
        R2_TURN_EMBEDDINGS_PATH.stat()
        .st_size
    )

    print("\n" + "=" * 80)
    print("EXISTING EMBEDDING MATRIX")
    print("=" * 80)

    print(
        "Existing size:",
        f"{actual_size:,} bytes",
    )

    print(
        "Expected size:",
        f"{expected_memmap_size:,} bytes",
    )

    assert actual_size == (
        expected_memmap_size
    ), (
        "Existing embedding matrix has unexpected "
        "size. Refusing to resume."
    )

    turn_embeddings_r2 = np.memmap(
        R2_TURN_EMBEDDINGS_PATH,
        dtype=R2_EMBEDDING_DTYPE,
        mode="r+",
        shape=(
            R2_TURN_ROWS,
            R2_EMBEDDING_DIM,
        ),
    )

    print(
        "Existing memmap opened: PASS"
    )

else:

    assert resume_row == 0, (
        "Checkpoint exists but embedding matrix "
        "does not. Refusing unsafe resume."
    )

    turn_embeddings_r2 = np.memmap(
        R2_TURN_EMBEDDINGS_PATH,
        dtype=R2_EMBEDDING_DTYPE,
        mode="w+",
        shape=(
            R2_TURN_ROWS,
            R2_EMBEDDING_DIM,
        ),
    )

    turn_embeddings_r2.flush()

    print("\n" + "=" * 80)
    print("NEW EMBEDDING MATRIX")
    print("=" * 80)

    print(
        "Created memmap:",
        R2_TURN_EMBEDDINGS_PATH,
    )

    print(
        "Storage:",
        f"{R2_EXPECTED_EMBEDDING_GB:.2f} GiB",
    )


# ==============================================================================
# 11. WRITE INITIAL CHECKPOINT
# ==============================================================================

if checkpoint is None:

    initial_checkpoint = {
        "status": "RUNNING",
        "completed_rows": 0,
        "total_rows": int(
            R2_TURN_ROWS
        ),
        "embedding_dimension": int(
            R2_EMBEDDING_DIM
        ),
        "dtype": str(
            np.dtype(
                R2_EMBEDDING_DTYPE
            )
        ),
        "normalized": bool(
            R2_NORMALIZE_EMBEDDINGS
        ),
        "outer_batch_rows": int(
            R2_EMBEDDING_BATCH_ROWS
        ),
        "model_batch_size": int(
            R2_MODEL_BATCH_SIZE
        ),
        "model_name": R2_MODEL_NAME,
        "source_fingerprint":
            R2_SOURCE_FINGERPRINT,
        "started_at": time.strftime(
            "%Y-%m-%dT%H:%M:%S"
        ),
    }

    write_json_atomic(
        R2_TURN_EMBEDDING_CHECKPOINT,
        initial_checkpoint,
    )


# ==============================================================================
# 12. RESUMABLE BATCH ENCODING
#
# We iterate over the parquet file in Arrow batches.
#
# IMPORTANT:
# If resume_row > 0, earlier source batches are read but NOT encoded.
# This guarantees alignment between source row position and embedding row.
# ==============================================================================

print("\n" + "=" * 80)
print("TURN EMBEDDING GENERATION")
print("=" * 80)

print(
    "Starting row:",
    f"{resume_row:,}",
)

print(
    "Total rows:",
    f"{R2_TURN_ROWS:,}",
)

print(
    "Remaining rows:",
    f"{R2_TURN_ROWS - resume_row:,}",
)

print(
    "\nEmbedding generation has now started."
)

processed_rows = resume_row

batch_counter = 0

generation_started = time.time()

source_iterator = turn_source_pf.iter_batches(
    batch_size=R2_EMBEDDING_BATCH_ROWS,
    columns=[
        "turn_uid",
        "session_id",
        "turn_index",
        "role",
        "text_norm",
    ],
)


for record_batch in source_iterator:

    batch_df = record_batch.to_pandas(
        use_threads=True
    )

    batch_start = (
        batch_df.index[0]
        if len(batch_df)
        else processed_rows
    )

    # --------------------------------------------------------------------------
    # The Arrow iterator does not expose global row positions directly.
    # Therefore maintain our own cursor.
    # --------------------------------------------------------------------------

    batch_start = (
        0
        if batch_counter == 0
        else batch_start
    )

    # Reconstruct global source position from processed iterator state.
    #
    # This variable is maintained separately from checkpoint state.
    # --------------------------------------------------------------------------

    # We need a dedicated source cursor.
    # It is initialized on first batch and incremented thereafter.

    if batch_counter == 0:

        source_cursor = 0

    batch_rows = len(batch_df)

    global_start = source_cursor

    global_end = (
        global_start
        + batch_rows
    )

    source_cursor = global_end

    batch_counter += 1

    # --------------------------------------------------------------------------
    # Skip already committed rows.
    # --------------------------------------------------------------------------

    if global_end <= resume_row:

        del batch_df

        continue

    # --------------------------------------------------------------------------
    # Defensive alignment assertion.
    # --------------------------------------------------------------------------

    if global_start < resume_row:

        # This can only occur when checkpoint falls inside an outer batch.
        # For safety, encode only the uncommitted suffix.

        skip = (
            resume_row
            - global_start
        )

        batch_df = batch_df.iloc[
            skip:
        ].copy()

        global_start = resume_row

        batch_rows = len(
            batch_df
        )

        global_end = (
            global_start
            + batch_rows
        )

    # --------------------------------------------------------------------------
    # Text contract
    # --------------------------------------------------------------------------

    texts = (
        batch_df[
            "text_norm"
        ]
        .astype(str)
        .tolist()
    )

    assert len(texts) == batch_rows

    assert all(
        text.strip() != ""
        for text in texts
    ), (
        "Blank text encountered during embedding."
    )

    # --------------------------------------------------------------------------
    # Encode
    # --------------------------------------------------------------------------

    encode_started = time.time()

    embeddings = r2_model.encode(
        texts,
        batch_size=R2_MODEL_BATCH_SIZE,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )

    embeddings = np.asarray(
        embeddings,
        dtype=R2_EMBEDDING_DTYPE,
    )

    encode_elapsed = (
        time.time()
        - encode_started
    )

    # --------------------------------------------------------------------------
    # Shape / numerical validation
    # --------------------------------------------------------------------------

    assert embeddings.shape == (
        batch_rows,
        R2_EMBEDDING_DIM,
    ), (
        "Embedding batch shape mismatch."
    )

    assert np.isfinite(
        embeddings
    ).all(), (
        "Non-finite embedding values detected."
    )

    norms = np.linalg.norm(
        embeddings,
        axis=1,
    )

    assert np.allclose(
        norms,
        1.0,
        atol=1e-4,
    ), (
        "Embedding normalization contract failed."
    )

    # --------------------------------------------------------------------------
    # WRITE
    #
    # The checkpoint is NOT advanced until after this write and flush.
    # --------------------------------------------------------------------------

    turn_embeddings_r2[
        global_start:global_end,
        :,
    ] = embeddings

    turn_embeddings_r2.flush()

    # --------------------------------------------------------------------------
    # Verify the just-written boundary rows before committing checkpoint.
    # --------------------------------------------------------------------------

    boundary_rows = np.array(
        [
            turn_embeddings_r2[
                global_start
            ],
            turn_embeddings_r2[
                global_end - 1
            ],
        ]
    )

    boundary_norms = np.linalg.norm(
        boundary_rows,
        axis=1,
    )

    assert np.allclose(
        boundary_norms,
        1.0,
        atol=1e-4,
    ), (
        "Post-write embedding boundary verification failed."
    )

    # --------------------------------------------------------------------------
    # COMMIT CHECKPOINT
    # --------------------------------------------------------------------------

    processed_rows = global_end

    elapsed_total = (
        time.time()
        - generation_started
    )

    rows_per_second = (
        processed_rows
        / elapsed_total
        if elapsed_total > 0
        else 0.0
    )

    remaining_rows = (
        R2_TURN_ROWS
        - processed_rows
    )

    eta_seconds = (
        remaining_rows
        / rows_per_second
        if rows_per_second > 0
        else None
    )

    checkpoint_payload = {
        "status": (
            "COMPLETE"
            if processed_rows == R2_TURN_ROWS
            else "RUNNING"
        ),
        "completed_rows": int(
            processed_rows
        ),
        "total_rows": int(
            R2_TURN_ROWS
        ),
        "embedding_dimension": int(
            R2_EMBEDDING_DIM
        ),
        "dtype": str(
            np.dtype(
                R2_EMBEDDING_DTYPE
            )
        ),
        "normalized": bool(
            R2_NORMALIZE_EMBEDDINGS
        ),
        "outer_batch_rows": int(
            R2_EMBEDDING_BATCH_ROWS
        ),
        "model_batch_size": int(
            R2_MODEL_BATCH_SIZE
        ),
        "model_name": R2_MODEL_NAME,
        "source_fingerprint":
            R2_SOURCE_FINGERPRINT,
        "updated_at": time.strftime(
            "%Y-%m-%dT%H:%M:%S"
        ),
    }

    write_json_atomic(
        R2_TURN_EMBEDDING_CHECKPOINT,
        checkpoint_payload,
    )

    # --------------------------------------------------------------------------
    # PROGRESS
    # --------------------------------------------------------------------------

    percent = (
        processed_rows
        / R2_TURN_ROWS
        * 100.0
    )

    print(
        f"Processed "
        f"{processed_rows:,}/"
        f"{R2_TURN_ROWS:,} "
        f"({percent:.2f}%) | "
        f"batch_rows={batch_rows:,} | "
        f"encode={encode_elapsed:.1f}s | "
        f"rate={rows_per_second:,.1f} rows/s"
        + (
            f" | ETA={eta_seconds / 3600:.2f}h"
            if eta_seconds is not None
            else ""
        )
    )

    # --------------------------------------------------------------------------
    # RELEASE BATCH MEMORY
    # --------------------------------------------------------------------------

    del embeddings
    del texts
    del batch_df
    del record_batch

    gc.collect()


# ==============================================================================
# 13. FINAL FLUSH
# ==============================================================================

turn_embeddings_r2.flush()


# ==============================================================================
# 14. FINAL ROW COUNT CHECK
# ==============================================================================

assert processed_rows == (
    R2_TURN_ROWS
), (
    "Embedding generation ended before all source rows were processed."
)


# ==============================================================================
# 15. FINAL CHECKPOINT
# ==============================================================================

final_checkpoint = {
    "status": "COMPLETE",
    "completed_rows": int(
        processed_rows
    ),
    "total_rows": int(
        R2_TURN_ROWS
    ),
    "embedding_dimension": int(
        R2_EMBEDDING_DIM
    ),
    "dtype": str(
        np.dtype(
            R2_EMBEDDING_DTYPE
        )
    ),
    "normalized": bool(
        R2_NORMALIZE_EMBEDDINGS
    ),
    "outer_batch_rows": int(
        R2_EMBEDDING_BATCH_ROWS
    ),
    "model_batch_size": int(
        R2_MODEL_BATCH_SIZE
    ),
    "model_name": R2_MODEL_NAME,
    "source_fingerprint":
        R2_SOURCE_FINGERPRINT,
    "completed_at": time.strftime(
        "%Y-%m-%dT%H:%M:%S"
    ),
}

write_json_atomic(
    R2_TURN_EMBEDDING_CHECKPOINT,
    final_checkpoint,
)


# ==============================================================================
# 16. FINAL MEMMAP VALIDATION
# ==============================================================================

assert R2_TURN_EMBEDDINGS_PATH.exists()

actual_memmap_size = (
    R2_TURN_EMBEDDINGS_PATH.stat()
    .st_size
)

assert actual_memmap_size == (
    expected_memmap_size
), (
    "Final embedding matrix size mismatch."
)


# ==============================================================================
# 17. FINAL SAMPLE VALIDATION
# ==============================================================================

sample_positions = np.array(
    [
        0,
        R2_TURN_ROWS // 2,
        R2_TURN_ROWS - 1,
    ],
    dtype=np.int64,
)

sample_embeddings = np.asarray(
    turn_embeddings_r2[
        sample_positions
    ],
    dtype=np.float32,
)

assert sample_embeddings.shape == (
    3,
    R2_EMBEDDING_DIM,
)

assert np.isfinite(
    sample_embeddings
).all()

sample_norms = np.linalg.norm(
    sample_embeddings,
    axis=1,
)

assert np.allclose(
    sample_norms,
    1.0,
    atol=1e-4,
), (
    "Final embedding sample normalization failed."
)


# ==============================================================================
# 18. MANIFEST
# ==============================================================================

R2_TURN_EMBEDDING_MANIFEST_PAYLOAD = {
    "artifact": "r2_turn_embeddings",
    "status": "FROZEN_READY",
    "source_artifact": str(
        R2_TURN_SOURCE.resolve()
    ),
    "source_fingerprint":
        R2_SOURCE_FINGERPRINT,
    "rows": int(
        R2_TURN_ROWS
    ),
    "embedding_dimension": int(
        R2_EMBEDDING_DIM
    ),
    "dtype": str(
        np.dtype(
            R2_EMBEDDING_DTYPE
        )
    ),
    "normalized": bool(
        R2_NORMALIZE_EMBEDDINGS
    ),
    "model_name": R2_MODEL_NAME,
    "model_revision": R2_MODEL_REVISION,
    "model_batch_size": int(
        R2_MODEL_BATCH_SIZE
    ),
    "outer_batch_rows": int(
        R2_EMBEDDING_BATCH_ROWS
    ),
    "device": str(
        r2_model.device
    ),
    "embedding_file": str(
        R2_TURN_EMBEDDINGS_PATH.resolve()
    ),
    "embedding_file_size_bytes": int(
        actual_memmap_size
    ),
    "completed_rows": int(
        processed_rows
    ),
    "created_at": time.strftime(
        "%Y-%m-%dT%H:%M:%S"
    ),
}

write_json_atomic(
    R2_TURN_EMBEDDING_MANIFEST,
    R2_TURN_EMBEDDING_MANIFEST_PAYLOAD,
)


# ==============================================================================
# 19. READY FLAG
# ==============================================================================

R2_TURN_EMBEDDINGS_READY = True
R2_CELL_3_READY = True


print("\n" + "=" * 80)
print("R2 CELL 3 STATUS")
print("=" * 80)

print(
    "Turn rows encoded        :",
    f"{processed_rows:,}",
)

print(
    "Expected turn rows       :",
    f"{R2_TURN_ROWS:,}",
)

print(
    "Embedding dimension      :",
    R2_EMBEDDING_DIM,
)

print(
    "Embedding storage        :",
    f"{actual_memmap_size / (1024 ** 3):.2f} GiB",
)

print(
    "Checkpoint               :",
    R2_TURN_EMBEDDING_CHECKPOINT.exists(),
)

print(
    "Manifest                 :",
    R2_TURN_EMBEDDING_MANIFEST.exists(),
)

print(
    "Normalization validation : PASS",
)

print(
    "Serialization validation : PASS",
)

print(
    "R2_TURN_EMBEDDINGS_READY :",
    R2_TURN_EMBEDDINGS_READY,
)

print(
    "R2_CELL_3_READY          :",
    R2_CELL_3_READY,
)

print("=" * 80)
print("R2 CELL 3 — TURN EMBEDDING PRECOMPUTE: PASS")
print("=" * 80)

TRACE THE ACE — R2 DENSE RETRIEVAL
CELL 3 — RESUMABLE TURN EMBEDDING PRECOMPUTE

R2 Cell 2 dependency: PASS

R2 TURN EMBEDDING PATHS
Source: d:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R0_input\session_turn_index.parquet
Embedding matrix: d:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R2_dense\embeddings\turn_embeddings\turn_embeddings.float32.memmap
Checkpoint: d:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R2_dense\embeddings\turn_embeddings\turn_embedding_checkpoint.json
Manifest: d:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R2_dense\embeddings\turn_embeddings\turn_embedding_manifest.json
Index: d:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R2_dense\embeddings\turn_embeddings\turn_embedding_index.parquet

EMBEDDING CONTRACT
Turn rows: 6,139,854
Embedding dimension: 384
Embedding dtype: <class 'numpy.float32'>
Outer checkpoint batch: 10,000
Model bat

C:\Users\USER\AppData\Local\Temp\ipykernel_7412\2317649743.py:178: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  r2_model.get_sentence_embedding_dimension()


Processed 10,000/6,139,854 (0.16%) | batch_rows=10,000 | encode=22.8s | rate=434.9 rows/s | ETA=3.92h
Processed 20,000/6,139,854 (0.33%) | batch_rows=10,000 | encode=18.6s | rate=478.9 rows/s | ETA=3.55h
Processed 30,000/6,139,854 (0.49%) | batch_rows=10,000 | encode=19.3s | rate=489.6 rows/s | ETA=3.47h
Processed 40,000/6,139,854 (0.65%) | batch_rows=10,000 | encode=19.4s | rate=494.6 rows/s | ETA=3.43h
Processed 50,000/6,139,854 (0.81%) | batch_rows=10,000 | encode=19.2s | rate=498.3 rows/s | ETA=3.39h
Processed 60,000/6,139,854 (0.98%) | batch_rows=10,000 | encode=18.1s | rate=505.7 rows/s | ETA=3.34h
Processed 70,000/6,139,854 (1.14%) | batch_rows=10,000 | encode=18.6s | rate=509.1 rows/s | ETA=3.31h
Processed 80,000/6,139,854 (1.30%) | batch_rows=10,000 | encode=18.5s | rate=512.1 rows/s | ETA=3.29h
Processed 90,000/6,139,854 (1.47%) | batch_rows=10,000 | encode=18.8s | rate=513.4 rows/s | ETA=3.27h
Processed 100,000/6,139,854 (1.63%) | batch_rows=10,000 | encode=18.3s | rate=515.

In [9]:
# ==============================================================================
# TRACE THE ACE — R2 DENSE RETRIEVAL
# CELL 4 — OBJECTIVE EMBEDDING PRECOMPUTE + FREEZE
#
# PURPOSE
# -------
# Encode the canonical 398 R0 objectives into the exact same dense embedding
# space used for the turn embeddings.
#
# This cell:
#   - loads the frozen R0 objective catalogue
#   - verifies objective identity/text contract
#   - encodes each objective exactly once
#   - L2-normalizes embeddings
#   - writes a frozen objective embedding artifact
#   - writes an identity-aligned objective index
#   - writes a manifest + SHA256
#
# This cell DOES NOT:
#   - retrieve turns
#   - use R1 candidates for scoring
#   - use targets / labels
#   - modify R0/R1 artifacts
# ==============================================================================

from pathlib import Path
import json
import hashlib
import time
import platform

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


print("=" * 80)
print("TRACE THE ACE — R2 DENSE RETRIEVAL")
print("CELL 4 — OBJECTIVE EMBEDDING PRECOMPUTE + FREEZE")
print("=" * 80)


# ==============================================================================
# 1. DEPENDENCY GATE
# ==============================================================================

assert R2_CELL_2_READY is True, (
    "R2 Cell 2 must pass before Cell 4."
)

assert R2_MODEL_READY is True, (
    "R2 model is not ready."
)

assert R2_CELL_3_READY is True, (
    "R2 Cell 3 must pass before Cell 4."
)

assert R2_TURN_EMBEDDINGS_READY is True, (
    "Turn embeddings are not ready."
)

print("\nR2 Cell 2 dependency: PASS")
print("R2 Cell 3 dependency: PASS")


# ==============================================================================
# 2. RELOAD MODEL IF NECESSARY
# ==============================================================================

try:
    r2_model
except NameError:

    print(
        "\nR2 model object not present in kernel."
    )

    from sentence_transformers import SentenceTransformer

    r2_model = SentenceTransformer(
        R2_MODEL_NAME
    )

    print(
        "R2 model reloaded: PASS"
    )


# ==============================================================================
# 3. OBJECTIVE SOURCE
# ==============================================================================

assert R0_OBJECTIVE_CATALOGUE.exists(), (
    "R0 objective catalogue is missing."
)

objective_catalogue_r2 = pd.read_parquet(
    R0_OBJECTIVE_CATALOGUE,
    engine="pyarrow",
)


print("\n" + "=" * 80)
print("OBJECTIVE SOURCE")
print("=" * 80)

print(
    "Path:",
    R0_OBJECTIVE_CATALOGUE,
)

print(
    "Rows:",
    f"{len(objective_catalogue_r2):,}",
)

print(
    "Columns:",
    list(objective_catalogue_r2.columns),
)


# ==============================================================================
# 4. EXACT OBJECTIVE CONTRACT
# ==============================================================================

EXPECTED_OBJECTIVE_COLUMNS = [
    "objective_uid",
    "objective_raw",
]

assert list(
    objective_catalogue_r2.columns
) == EXPECTED_OBJECTIVE_COLUMNS, (
    "Objective catalogue schema/order differs from "
    "the frozen R0 contract."
)

assert len(
    objective_catalogue_r2
) == 398, (
    "Objective population changed."
)

assert objective_catalogue_r2[
    "objective_uid"
].is_unique, (
    "objective_uid is not unique."
)

assert objective_catalogue_r2[
    "objective_uid"
].notna().all(), (
    "Null objective_uid detected."
)

assert objective_catalogue_r2[
    "objective_raw"
].notna().all(), (
    "Null objective_raw detected."
)

assert (
    objective_catalogue_r2[
        "objective_raw"
    ]
    .astype(str)
    .str.strip()
    .ne("")
    .all()
), (
    "Blank objective text detected."
)


print("\n" + "=" * 80)
print("OBJECTIVE CONTRACT")
print("=" * 80)

print(
    "Exact row count :",
    len(objective_catalogue_r2) == 398,
)

print(
    "UID unique      :",
    objective_catalogue_r2[
        "objective_uid"
    ].is_unique,
)

print(
    "UID non-null    :",
    objective_catalogue_r2[
        "objective_uid"
    ].notna().all(),
)

print(
    "Text non-blank  :",
    objective_catalogue_r2[
        "objective_raw"
    ]
    .astype(str)
    .str.strip()
    .ne("")
    .all(),
)


# ==============================================================================
# 5. OUTPUT PATHS
# ==============================================================================

R2_OBJECTIVE_EMBEDDING_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

R2_OBJECTIVE_EMBEDDINGS_PATH = (
    R2_OBJECTIVE_EMBEDDING_ROOT
    / "objective_embeddings.float32.npy"
)

R2_OBJECTIVE_INDEX_PATH = (
    R2_OBJECTIVE_EMBEDDING_ROOT
    / "objective_embedding_index.parquet"
)

R2_OBJECTIVE_EMBEDDING_MANIFEST = (
    R2_OBJECTIVE_EMBEDDING_ROOT
    / "objective_embedding_manifest.json"
)


print("\n" + "=" * 80)
print("OBJECTIVE EMBEDDING PATHS")
print("=" * 80)

print(
    "Embeddings:",
    R2_OBJECTIVE_EMBEDDINGS_PATH,
)

print(
    "Index:",
    R2_OBJECTIVE_INDEX_PATH,
)

print(
    "Manifest:",
    R2_OBJECTIVE_EMBEDDING_MANIFEST,
)


# ==============================================================================
# 6. EXACT EMBEDDING CONTRACT
# ==============================================================================

R2_OBJECTIVE_EMBEDDING_DIM = int(
    r2_model.get_sentence_embedding_dimension()
)

assert R2_OBJECTIVE_EMBEDDING_DIM == (
    R2_EMBEDDING_DIM
), (
    "Objective and turn embedding dimensions differ."
)

R2_OBJECTIVE_EMBEDDING_DTYPE = np.float32

R2_OBJECTIVE_BATCH_SIZE = 128

R2_OBJECTIVE_NORMALIZE = True


print("\n" + "=" * 80)
print("OBJECTIVE EMBEDDING CONTRACT")
print("=" * 80)

print(
    "Rows:",
    len(objective_catalogue_r2),
)

print(
    "Dimension:",
    R2_OBJECTIVE_EMBEDDING_DIM,
)

print(
    "Dtype:",
    R2_OBJECTIVE_EMBEDDING_DTYPE,
)

print(
    "Batch size:",
    R2_OBJECTIVE_BATCH_SIZE,
)

print(
    "L2 normalized:",
    R2_OBJECTIVE_NORMALIZE,
)


# ==============================================================================
# 7. OBJECTIVE TEXTS
# ==============================================================================

objective_uids = (
    objective_catalogue_r2[
        "objective_uid"
    ]
    .astype(str)
    .tolist()
)

objective_texts = (
    objective_catalogue_r2[
        "objective_raw"
    ]
    .astype(str)
    .tolist()
)

assert len(
    objective_uids
) == 398

assert len(
    objective_texts
) == 398


# ==============================================================================
# 8. ENCODE OBJECTIVES
# ==============================================================================

print("\n" + "=" * 80)
print("OBJECTIVE EMBEDDING GENERATION")
print("=" * 80)

encode_started = time.time()

objective_embeddings_r2 = r2_model.encode(
    objective_texts,
    batch_size=R2_OBJECTIVE_BATCH_SIZE,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

objective_embeddings_r2 = np.asarray(
    objective_embeddings_r2,
    dtype=R2_OBJECTIVE_EMBEDDING_DTYPE,
)

encode_elapsed = (
    time.time()
    - encode_started
)


# ==============================================================================
# 9. EMBEDDING SHAPE / NUMERICAL VALIDATION
# ==============================================================================

assert objective_embeddings_r2.shape == (
    398,
    R2_OBJECTIVE_EMBEDDING_DIM,
), (
    "Unexpected objective embedding shape."
)

assert np.isfinite(
    objective_embeddings_r2
).all(), (
    "Non-finite objective embedding values detected."
)

objective_norms = np.linalg.norm(
    objective_embeddings_r2,
    axis=1,
)

assert np.allclose(
    objective_norms,
    1.0,
    atol=1e-4,
), (
    "Objective embedding L2 normalization failed."
)


print(
    "Shape:",
    objective_embeddings_r2.shape,
)

print(
    "Dtype:",
    objective_embeddings_r2.dtype,
)

print(
    "Finite values: PASS"
)

print(
    "L2 normalization: PASS"
)

print(
    "Encoding time:",
    f"{encode_elapsed:.2f}s",
)


# ==============================================================================
# 10. SAVE OBJECTIVE EMBEDDINGS
#
# We refuse to silently overwrite an existing artifact with a potentially
# different model output.
# ==============================================================================

if R2_OBJECTIVE_EMBEDDINGS_PATH.exists():

    existing = np.load(
        R2_OBJECTIVE_EMBEDDINGS_PATH,
        mmap_mode="r",
    )

    assert existing.shape == (
        398,
        R2_OBJECTIVE_EMBEDDING_DIM,
    ), (
        "Existing objective embedding artifact has "
        "unexpected shape."
    )

    existing_array = np.asarray(
        existing
    )

    same_existing = np.allclose(
        existing_array,
        objective_embeddings_r2,
        atol=1e-6,
        rtol=1e-6,
    )

    assert same_existing, (
        "Existing objective embedding artifact differs "
        "from the current model output. Refusing overwrite."
    )

    del existing
    del existing_array

    print(
        "\nExisting objective embeddings verified: PASS"
    )

else:

    tmp_path = (
        R2_OBJECTIVE_EMBEDDINGS_PATH
        .with_name(
            R2_OBJECTIVE_EMBEDDINGS_PATH.name
            + ".tmp"
        )
    )

    if tmp_path.exists():
        tmp_path.unlink()

    with open(
        tmp_path,
        "wb",
    ) as f:

        np.save(
            f,
            objective_embeddings_r2,
            allow_pickle=False,
        )

        f.flush()

    tmp_path.replace(
        R2_OBJECTIVE_EMBEDDINGS_PATH
    )

    print(
        "\nObjective embeddings written: PASS"
    )


# ==============================================================================
# 11. OBJECTIVE INDEX
#
# The row position is the immutable alignment key between:
#
# objective_embedding_index.parquet
#          and
# objective_embeddings.float32.npy
#
# ==============================================================================

objective_embedding_index = (
    objective_catalogue_r2[
        [
            "objective_uid",
            "objective_raw",
        ]
    ]
    .copy()
)

objective_embedding_index.insert(
    0,
    "embedding_row",
    np.arange(
        len(
            objective_embedding_index
        ),
        dtype=np.int32,
    ),
)


assert objective_embedding_index[
    "embedding_row"
].is_unique

assert (
    objective_embedding_index[
        "embedding_row"
    ].min()
    == 0
)

assert (
    objective_embedding_index[
        "embedding_row"
    ].max()
    == 397
)


objective_embedding_index.to_parquet(
    R2_OBJECTIVE_INDEX_PATH,
    engine="pyarrow",
    index=False,
)


print(
    "Objective embedding index written: PASS"
)


# ==============================================================================
# 12. RELOAD VERIFICATION
# ==============================================================================

objective_reload = np.load(
    R2_OBJECTIVE_EMBEDDINGS_PATH,
    mmap_mode="r",
)

assert objective_reload.shape == (
    398,
    R2_OBJECTIVE_EMBEDDING_DIM,
)

assert objective_reload.dtype == (
    R2_OBJECTIVE_EMBEDDING_DTYPE
)

objective_index_reload = pd.read_parquet(
    R2_OBJECTIVE_INDEX_PATH,
    engine="pyarrow",
)

assert len(
    objective_index_reload
) == 398

assert list(
    objective_index_reload.columns
) == [
    "embedding_row",
    "objective_uid",
    "objective_raw",
]


assert objective_index_reload[
    "objective_uid"
].is_unique


# ==============================================================================
# 13. IDENTITY ALIGNMENT
# ==============================================================================

assert (
    objective_index_reload[
        "objective_uid"
    ].tolist()
    ==
    objective_uids
), (
    "Objective embedding index UID order differs "
    "from the embedding source order."
)

assert (
    objective_index_reload[
        "objective_raw"
    ].tolist()
    ==
    objective_texts
), (
    "Objective embedding index text order differs "
    "from the embedding source order."
)


# ==============================================================================
# 14. RELOADED EMBEDDING SAMPLE
# ==============================================================================

objective_sample_positions = np.array(
    [
        0,
        197,
        397,
    ],
    dtype=np.int64,
)

objective_sample = np.asarray(
    objective_reload[
        objective_sample_positions
    ],
    dtype=np.float32,
)

objective_sample_norms = np.linalg.norm(
    objective_sample,
    axis=1,
)

assert np.allclose(
    objective_sample_norms,
    1.0,
    atol=1e-4,
), (
    "Reloaded objective embedding normalization failed."
)


# ==============================================================================
# 15. OBJECTIVE SELF-SIMILARITY
# ==============================================================================

objective_self_similarity = (
    objective_sample
    @
    objective_sample.T
)

assert np.allclose(
    np.diag(
        objective_self_similarity
    ),
    1.0,
    atol=1e-5,
)

assert np.isfinite(
    objective_self_similarity
).all()


# ==============================================================================
# 16. SHA256 HELPERS
# ==============================================================================

def sha256_file_r2(
    path,
    chunk_size=8 * 1024 * 1024,
):

    digest = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


objective_embedding_sha256 = (
    sha256_file_r2(
        R2_OBJECTIVE_EMBEDDINGS_PATH
    )
)

objective_index_sha256 = (
    sha256_file_r2(
        R2_OBJECTIVE_INDEX_PATH
    )
)


# ==============================================================================
# 17. MANIFEST
# ==============================================================================

R2_OBJECTIVE_EMBEDDING_MANIFEST_PAYLOAD = {
    "artifact": "r2_objective_embeddings",
    "status": "FROZEN_READY",
    "source_artifact": str(
        R0_OBJECTIVE_CATALOGUE.resolve()
    ),
    "rows": 398,
    "embedding_dimension": int(
        R2_OBJECTIVE_EMBEDDING_DIM
    ),
    "dtype": str(
        np.dtype(
            R2_OBJECTIVE_EMBEDDING_DTYPE
        )
    ),
    "normalized": True,
    "model_name": R2_MODEL_NAME,
    "model_revision": R2_MODEL_REVISION,
    "model_device": str(
        r2_model.device
    ),
    "model_batch_size": int(
        R2_OBJECTIVE_BATCH_SIZE
    ),
    "sentence_transformers_version":
        sentence_transformers.__version__,
    "embedding_file": str(
        R2_OBJECTIVE_EMBEDDINGS_PATH.resolve()
    ),
    "embedding_sha256":
        objective_embedding_sha256,
    "index_file": str(
        R2_OBJECTIVE_INDEX_PATH.resolve()
    ),
    "index_sha256":
        objective_index_sha256,
    "created_at": time.strftime(
        "%Y-%m-%dT%H:%M:%S"
    ),
}


with open(
    R2_OBJECTIVE_EMBEDDING_MANIFEST,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        R2_OBJECTIVE_EMBEDDING_MANIFEST_PAYLOAD,
        f,
        indent=2,
        ensure_ascii=False,
    )


# ==============================================================================
# 18. FINAL MANIFEST VALIDATION
# ==============================================================================

with open(
    R2_OBJECTIVE_EMBEDDING_MANIFEST,
    "r",
    encoding="utf-8",
) as f:

    manifest_reload = json.load(f)

assert (
    manifest_reload["status"]
    == "FROZEN_READY"
)

assert (
    manifest_reload["rows"]
    == 398
)

assert (
    manifest_reload["embedding_dimension"]
    == R2_OBJECTIVE_EMBEDDING_DIM
)

assert (
    manifest_reload["embedding_sha256"]
    == objective_embedding_sha256
)

assert (
    manifest_reload["index_sha256"]
    == objective_index_sha256
)


# ==============================================================================
# 19. READY FLAGS
# ==============================================================================

R2_OBJECTIVE_EMBEDDINGS_READY = True
R2_CELL_4_READY = True


print("\n" + "=" * 80)
print("R2 CELL 4 STATUS")
print("=" * 80)

print(
    "Objective rows              :",
    f"{len(objective_catalogue_r2):,}",
)

print(
    "Embedding dimension         :",
    R2_OBJECTIVE_EMBEDDING_DIM,
)

print(
    "Embedding shape             :",
    objective_reload.shape,
)

print(
    "Embedding dtype             :",
    objective_reload.dtype,
)

print(
    "L2 normalization            : PASS",
)

print(
    "Objective identity alignment: PASS",
)

print(
    "Serialization verified      : PASS",
)

print(
    "SHA256 verified             : PASS",
)

print(
    "Manifest                    :",
    R2_OBJECTIVE_EMBEDDING_MANIFEST.exists(),
)

print(
    "R2_OBJECTIVE_EMBEDDINGS_READY:",
    R2_OBJECTIVE_EMBEDDINGS_READY,
)

print(
    "R2_CELL_4_READY             :",
    R2_CELL_4_READY,
)

print("=" * 80)
print("R2 CELL 4 — OBJECTIVE EMBEDDING FREEZE: PASS")
print("=" * 80)

TRACE THE ACE — R2 DENSE RETRIEVAL
CELL 4 — OBJECTIVE EMBEDDING PRECOMPUTE + FREEZE

R2 Cell 2 dependency: PASS
R2 Cell 3 dependency: PASS

OBJECTIVE SOURCE
Path: d:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R0_input\objective_catalogue.parquet
Rows: 398
Columns: ['objective_uid', 'objective_raw']

OBJECTIVE CONTRACT
Exact row count : True
UID unique      : True
UID non-null    : True
Text non-blank  : True

OBJECTIVE EMBEDDING PATHS
Embeddings: d:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R2_dense\embeddings\objective_embeddings\objective_embeddings.float32.npy
Index: d:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R2_dense\embeddings\objective_embeddings\objective_embedding_index.parquet
Manifest: d:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R2_dense\embeddings\objective_embeddings\objective_embedding_manifest.json

OBJECTIVE EMBEDDING CONTRACT
Rows: 398
Dimension: 384
Dtype: 

C:\Users\USER\AppData\Local\Temp\ipykernel_7412\3525147526.py:262: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  r2_model.get_sentence_embedding_dimension()


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Shape: (398, 384)
Dtype: float32
Finite values: PASS
L2 normalization: PASS
Encoding time: 0.39s

Objective embeddings written: PASS
Objective embedding index written: PASS

R2 CELL 4 STATUS
Objective rows              : 398
Embedding dimension         : 384
Embedding shape             : (398, 384)
Embedding dtype             : float32
L2 normalization            : PASS
Objective identity alignment: PASS
Serialization verified      : PASS
SHA256 verified             : PASS
Manifest                    : True
R2_OBJECTIVE_EMBEDDINGS_READY: True
R2_CELL_4_READY             : True
R2 CELL 4 — OBJECTIVE EMBEDDING FREEZE: PASS


In [18]:
# ==============================================================================
# TRACE THE ACE — R2 DENSE RETRIEVAL
# CELL 5 — RESUMABLE SESSION-LOCAL DENSE RETRIEVAL
# ==============================================================================

from pathlib import Path
import json
import gc
import time
import hashlib

import numpy as np
import pandas as pd


print("\n" + "=" * 80)
print("TRACE THE ACE — R2 DENSE RETRIEVAL")
print("CELL 5 — RESUMABLE SESSION-LOCAL DENSE RETRIEVAL")
print("=" * 80)


# ==============================================================================
# 1. DISK-BASED DEPENDENCY RELOAD
# ==============================================================================

print("\n" + "=" * 80)
print("R2 DISK-BASED DEPENDENCY VERIFICATION")
print("=" * 80)


# ------------------------------------------------------------------------------
# R2 ROOT
# ------------------------------------------------------------------------------

R2_ROOT = Path(
    r"D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R2_dense"
)


# ------------------------------------------------------------------------------
# R0 / R1 ROOTS
# ------------------------------------------------------------------------------

R0_ROOT = Path(
    r"D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R0_input"
)

R1_ROOT = Path(
    r"D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R1_sparse"
)


# ------------------------------------------------------------------------------
# R0 ARTIFACTS
# ------------------------------------------------------------------------------

R0_RETRIEVAL_QUERIES = (
    R0_ROOT
    / "retrieval_queries.parquet"
)

R0_SESSION_TURN_INDEX = (
    R0_ROOT
    / "session_turn_index.parquet"
)

R0_OBJECTIVE_CATALOGUE = (
    R0_ROOT
    / "objective_catalogue.parquet"
)


# ------------------------------------------------------------------------------
# R2 TURN EMBEDDINGS
# ------------------------------------------------------------------------------

R2_TURN_EMBEDDINGS_PATH = (
    R2_ROOT
    / "embeddings"
    / "turn_embeddings"
    / "turn_embeddings.float32.memmap"
)

R2_TURN_EMBEDDING_MANIFEST = (
    R2_ROOT
    / "embeddings"
    / "turn_embeddings"
    / "turn_embedding_manifest.json"
)


# ==============================================================================
# 2. OBJECTIVE EMBEDDING ARTIFACT DISCOVERY + VALIDATION
# ==============================================================================

print("\n" + "=" * 80)
print("R2 OBJECTIVE EMBEDDING ARTIFACT DISCOVERY")
print("=" * 80)


R2_OBJECTIVE_EMBEDDING_ROOT = (
    R2_ROOT / "embeddings"
)


assert R2_OBJECTIVE_EMBEDDING_ROOT.exists(), (
    f"R2 embedding root is missing:\n"
    f"{R2_OBJECTIVE_EMBEDDING_ROOT}"
)


# ------------------------------------------------------------------------------
# Cell 4 contract says objective embeddings are persisted under the objective
# embedding artifact directory.
#
# Do not validate .npy using raw payload byte size because .npy contains a
# NumPy header.
# ------------------------------------------------------------------------------

objective_embedding_candidates = sorted(
    R2_OBJECTIVE_EMBEDDING_ROOT.glob(
        "**/*objective*.npy"
    )
)


print(
    "Objective .npy candidates:"
)

for path in objective_embedding_candidates:

    print(
        " -",
        path,
        "|",
        f"{path.stat().st_size:,}",
        "bytes",
    )


assert len(
    objective_embedding_candidates
) >= 1, (
    "No objective .npy embedding artifact was found."
)


# ------------------------------------------------------------------------------
# Prefer the canonical expected filename if it exists.
# ------------------------------------------------------------------------------

canonical_objective_embedding = (
    R2_OBJECTIVE_EMBEDDING_ROOT
    / "objective_embeddings"
    / "objective_embeddings.npy"
)


if canonical_objective_embedding.exists():

    R2_OBJECTIVE_EMBEDDINGS_PATH = (
        canonical_objective_embedding
    )

else:

    if len(
        objective_embedding_candidates
    ) == 1:

        R2_OBJECTIVE_EMBEDDINGS_PATH = (
            objective_embedding_candidates[0]
        )

    else:

        raise AssertionError(
            "Multiple objective .npy embedding artifacts "
            "were found and the canonical artifact cannot "
            "be identified automatically."
        )


print(
    "\nSelected objective embedding:",
    R2_OBJECTIVE_EMBEDDINGS_PATH,
)


# ==============================================================================
# LOAD NPY WITHOUT COPYING THE WHOLE ARRAY
# ==============================================================================

objective_embeddings_r2 = np.load(
    R2_OBJECTIVE_EMBEDDINGS_PATH,
    mmap_mode="r",
)


# ------------------------------------------------------------------------------
# Exact Cell 4 embedding contract
# ------------------------------------------------------------------------------

assert objective_embeddings_r2.shape == (
    398,
    384,
), (
    "Objective embedding shape mismatch: "
    f"observed={objective_embeddings_r2.shape}, "
    f"expected=(398, 384)"
)


assert objective_embeddings_r2.dtype == (
    np.float32
), (
    "Objective embedding dtype mismatch: "
    f"observed={objective_embeddings_r2.dtype}, "
    f"expected=float32"
)


# ------------------------------------------------------------------------------
# Numeric integrity
# ------------------------------------------------------------------------------

objective_sample = np.asarray(
    objective_embeddings_r2[
        :5
    ],
    dtype=np.float32,
)


assert np.isfinite(
    objective_sample
).all(), (
    "Objective embedding sample contains "
    "non-finite values."
)


objective_sample_norms = np.linalg.norm(
    objective_sample,
    axis=1,
)


assert np.allclose(
    objective_sample_norms,
    1.0,
    atol=1e-4,
), (
    "Objective embeddings are not L2-normalized."
)


print(
    "\nShape:",
    objective_embeddings_r2.shape,
)

print(
    "Dtype:",
    objective_embeddings_r2.dtype,
)

print(
    "Finite values: PASS"
)

print(
    "L2 normalization: PASS"
)


# ==============================================================================
# 3. OBJECTIVE INDEX DISCOVERY
# ==============================================================================

objective_index_candidates = sorted(
    R2_OBJECTIVE_EMBEDDING_ROOT.glob(
        "**/*objective*index*.parquet"
    )
)


print(
    "\nObjective index candidates:"
)

for path in objective_index_candidates:

    print(
        " -",
        path,
    )


assert len(
    objective_index_candidates
) >= 1, (
    "No objective embedding index parquet was found."
)


canonical_objective_index = (
    R2_OBJECTIVE_EMBEDDING_ROOT
    / "objective_embeddings"
    / "objective_embedding_index.parquet"
)


if canonical_objective_index.exists():

    R2_OBJECTIVE_INDEX_PATH = (
        canonical_objective_index
    )

elif len(
    objective_index_candidates
) == 1:

    R2_OBJECTIVE_INDEX_PATH = (
        objective_index_candidates[0]
    )

else:

    raise AssertionError(
        "Multiple objective embedding index artifacts "
        "were found and the canonical index cannot "
        "be identified automatically."
    )


print(
    "Selected objective index:",
    R2_OBJECTIVE_INDEX_PATH,
)


# ==============================================================================
# OBJECTIVE ARTIFACT CONTRACT
# ==============================================================================

objective_embedding_index_r2 = (
    pd.read_parquet(
        R2_OBJECTIVE_INDEX_PATH,
        engine="pyarrow",
    )
)


assert len(
    objective_embedding_index_r2
) == 398, (
    "Objective embedding index row count mismatch."
)


assert (
    "objective_uid"
    in objective_embedding_index_r2.columns
), (
    "Objective embedding index missing objective_uid."
)


assert (
    "embedding_row"
    in objective_embedding_index_r2.columns
), (
    "Objective embedding index missing embedding_row."
)


assert (
    objective_embedding_index_r2[
        "objective_uid"
    ].astype(str).is_unique
), (
    "Objective embedding index contains duplicate objective_uid."
)


assert (
    objective_embedding_index_r2[
        "embedding_row"
    ].between(
        0,
        397,
    ).all()
), (
    "Objective embedding index contains invalid embedding rows."
)


assert (
    objective_embedding_index_r2[
        "embedding_row"
    ].nunique()
    == 398
), (
    "Objective embedding rows are not one-to-one."
)


print(
    "\nObjective index rows:",
    len(
        objective_embedding_index_r2
    ),
)

print(
    "Objective UID unique: PASS"
)

print(
    "Embedding row range: PASS"
)

print(
    "Objective embedding alignment: PASS"
)


print("\n" + "=" * 80)
print("OBJECTIVE EMBEDDING DISCOVERY: PASS")
print("=" * 80)


# ==============================================================================
# 3. OBJECTIVE INDEX DISCOVERY
# ==============================================================================

objective_index_candidates = list(
    R2_OBJECTIVE_EMBEDDING_ROOT.glob(
        "**/*objective*index*.parquet"
    )
)


if not objective_index_candidates:

    objective_index_candidates = list(
        R2_OBJECTIVE_EMBEDDING_ROOT.glob(
            "**/*index*.parquet"
        )
    )


print(
    "\nObjective index candidates:"
)

for path in objective_index_candidates:

    print(
        " -",
        path,
    )


assert len(
    objective_index_candidates
) >= 1, (
    "No objective embedding index parquet "
    "was found."
)


if len(
    objective_index_candidates
) == 1:

    R2_OBJECTIVE_INDEX_PATH = (
        objective_index_candidates[0]
    )

else:

    exact_name_matches = [
        path
        for path in objective_index_candidates
        if path.name
        == "objective_embedding_index.parquet"
    ]

    assert len(
        exact_name_matches
    ) == 1, (
        "Multiple objective embedding index artifacts "
        "were found and the correct one cannot be "
        "identified automatically."
    )

    R2_OBJECTIVE_INDEX_PATH = (
        exact_name_matches[0]
    )


print(
    "Selected objective index:",
    R2_OBJECTIVE_INDEX_PATH,
)


# ==============================================================================
# 4. ARTIFACT EXISTENCE GATE
# ==============================================================================

assert R0_RETRIEVAL_QUERIES.exists(), (
    f"Missing R0 retrieval queries:\n"
    f"{R0_RETRIEVAL_QUERIES}"
)

assert R0_SESSION_TURN_INDEX.exists(), (
    f"Missing R0 session-turn index:\n"
    f"{R0_SESSION_TURN_INDEX}"
)

assert R0_OBJECTIVE_CATALOGUE.exists(), (
    f"Missing R0 objective catalogue:\n"
    f"{R0_OBJECTIVE_CATALOGUE}"
)

assert R2_TURN_EMBEDDINGS_PATH.exists(), (
    f"Missing turn embeddings:\n"
    f"{R2_TURN_EMBEDDINGS_PATH}"
)

assert R2_TURN_EMBEDDING_MANIFEST.exists(), (
    f"Missing turn embedding manifest:\n"
    f"{R2_TURN_EMBEDDING_MANIFEST}"
)

assert R2_OBJECTIVE_EMBEDDINGS_PATH.exists(), (
    f"Missing objective embeddings:\n"
    f"{R2_OBJECTIVE_EMBEDDINGS_PATH}"
)

assert R2_OBJECTIVE_INDEX_PATH.exists(), (
    f"Missing objective embedding index:\n"
    f"{R2_OBJECTIVE_INDEX_PATH}"
)


print("\nArtifact existence:")
print("R0 retrieval queries       : PASS")
print("R0 session-turn index      : PASS")
print("R0 objective catalogue     : PASS")
print("R2 turn embedding memmap   : PASS")
print("R2 turn embedding manifest : PASS")
print("R2 objective embeddings    : PASS")
print("R2 objective index         : PASS")


# ==============================================================================
# 5. TURN EMBEDDING MANIFEST VERIFICATION
# ==============================================================================

with open(
    R2_TURN_EMBEDDING_MANIFEST,
    "r",
    encoding="utf-8",
) as f:

    turn_embedding_manifest = json.load(
        f
    )


assert turn_embedding_manifest.get(
    "status"
) in {
    "COMPLETE",
    "FROZEN_READY",
    "READY",
}, (
    "Turn embedding manifest does not indicate "
    "a completed/frozen artifact."
)


print(
    "Turn embedding manifest status:",
    turn_embedding_manifest.get(
        "status"
    ),
)


# ==============================================================================
# 6. LOCAL DEPENDENCY FLAGS
# ==============================================================================

R2_CELL_0_READY = True
R2_CELL_1_READY = True
R2_CELL_2_READY = True
R2_CELL_3_READY = True
R2_CELL_4_READY = True

R2_TURN_EMBEDDINGS_READY = True
R2_OBJECTIVE_EMBEDDINGS_READY = True

R2_EMBEDDING_DIM = 384
R2_TURN_ROWS = 6_139_854


print("\n" + "=" * 80)
print("R2 DISK DEPENDENCY RELOAD: PASS")
print("=" * 80)


# ==============================================================================
# 7. LOAD OBJECTIVE EMBEDDINGS
# ==============================================================================

print("\n" + "=" * 80)
print("OBJECTIVE EMBEDDING LOAD")
print("=" * 80)


if (
    R2_OBJECTIVE_EMBEDDINGS_PATH.suffix.lower()
    == ".npy"
):

    objective_embeddings_r2 = np.load(
        R2_OBJECTIVE_EMBEDDINGS_PATH,
        mmap_mode="r",
    )

else:

    objective_embeddings_r2 = np.memmap(
        R2_OBJECTIVE_EMBEDDINGS_PATH,
        dtype=np.float32,
        mode="r",
        shape=(
            EXPECTED_OBJECTIVE_ROWS,
            EXPECTED_EMBEDDING_DIM,
        ),
    )


assert objective_embeddings_r2.shape == (
    EXPECTED_OBJECTIVE_ROWS,
    EXPECTED_EMBEDDING_DIM,
), (
    "Objective embedding shape mismatch."
)

assert (
    objective_embeddings_r2.dtype
    == np.float32
), (
    "Objective embedding dtype must be float32."
)


objective_sample = np.asarray(
    objective_embeddings_r2[
        :min(
            5,
            EXPECTED_OBJECTIVE_ROWS,
        )
    ],
    dtype=np.float32,
)


assert np.isfinite(
    objective_sample
).all(), (
    "Objective embedding sample contains "
    "non-finite values."
)


objective_norms = np.linalg.norm(
    objective_sample,
    axis=1,
)


assert np.allclose(
    objective_norms,
    1.0,
    atol=1e-4,
), (
    "Objective embeddings are not L2-normalized."
)


print(
    "Path:",
    R2_OBJECTIVE_EMBEDDINGS_PATH,
)

print(
    "Shape:",
    objective_embeddings_r2.shape,
)

print(
    "Dtype:",
    objective_embeddings_r2.dtype,
)

print(
    "Finite values: PASS"
)

print(
    "L2 normalization: PASS"
)


# ==============================================================================
# 8. PATHS FOR DENSE RETRIEVAL
# ==============================================================================

R2_DENSE_ROOT = (
    R2_ROOT
    / "dense_retrieval"
)

R2_DENSE_PARTS_ROOT = (
    R2_DENSE_ROOT
    / "parts"
)

R2_DENSE_CHECKPOINT = (
    R2_DENSE_ROOT
    / "r2_dense_retrieval_checkpoint.json"
)

R2_DENSE_MANIFEST = (
    R2_DENSE_ROOT
    / "r2_dense_retrieval_manifest.json"
)


R2_DENSE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

R2_DENSE_PARTS_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ==============================================================================
# 9. CONFIGURATION
# ==============================================================================

R2_DENSE_TOP_K = 50

R2_DENSE_RESPONSE_BATCH = 100

R2_DENSE_TURN_BLOCK = 10_000


print("\n" + "=" * 80)
print("DENSE RETRIEVAL CONFIGURATION")
print("=" * 80)

print(
    "Top-K per response:",
    R2_DENSE_TOP_K,
)

print(
    "Response batch:",
    R2_DENSE_RESPONSE_BATCH,
)

print(
    "Turn scoring block:",
    f"{R2_DENSE_TURN_BLOCK:,}",
)


# ==============================================================================
# 10. LOAD R0 RETRIEVAL QUERIES
# ==============================================================================

retrieval_queries_r2 = pd.read_parquet(
    R0_RETRIEVAL_QUERIES,
    engine="pyarrow",
)


assert list(
    retrieval_queries_r2.columns
) == [
    "response_id",
    "session_id",
    "objective_raw",
    "objective_uid",
    "fold",
]


assert len(
    retrieval_queries_r2
) == 35_072


assert retrieval_queries_r2[
    "response_id"
].is_unique


# ==============================================================================
# 11. LOAD R0 SESSION-TURN INDEX
# ==============================================================================

session_turn_r2 = pd.read_parquet(
    R0_SESSION_TURN_INDEX,
    columns=[
        "session_id",
        "turn_uid",
        "turn_index",
        "role",
        "text_norm",
    ],
    engine="pyarrow",
)


assert len(
    session_turn_r2
) == 6_139_854


assert session_turn_r2[
    "turn_uid"
].is_unique


print("\n" + "=" * 80)
print("R0 INPUTS")
print("=" * 80)

print(
    "Responses:",
    f"{len(retrieval_queries_r2):,}",
)

print(
    "Turns:",
    f"{len(session_turn_r2):,}",
)

print(
    "Sessions:",
    f"{session_turn_r2['session_id'].nunique():,}",
)

print(
    "Objectives:",
    f"{retrieval_queries_r2['objective_uid'].nunique():,}",
)


# ==============================================================================
# 12. LOAD FROZEN TURN EMBEDDINGS
# ==============================================================================

turn_embeddings_r2 = np.memmap(
    R2_TURN_EMBEDDINGS_PATH,
    dtype=np.float32,
    mode="r",
    shape=(
        6_139_854,
        384,
    ),
)


assert turn_embeddings_r2.shape == (
    6_139_854,
    384,
)


# ==============================================================================
# 13. PREPARE ARRAYS
# ==============================================================================

session_ids = (
    session_turn_r2[
        "session_id"
    ]
    .astype(str)
    .to_numpy()
)


turn_uids = (
    session_turn_r2[
        "turn_uid"
    ]
    .astype(str)
    .to_numpy()
)


turn_indices = (
    session_turn_r2[
        "turn_index"
    ]
    .to_numpy(
        dtype=np.int32
    )
)


roles = (
    session_turn_r2[
        "role"
    ]
    .astype(str)
    .to_numpy()
)


texts = (
    session_turn_r2[
        "text_norm"
    ]
    .astype(str)
    .to_numpy()
)


# ==============================================================================
# 14. BUILD SESSION POSITION RANGES
# ==============================================================================

session_ranges = {}

start = 0

for i in range(
    1,
    len(session_ids),
):

    if (
        session_ids[i]
        != session_ids[start]
    ):

        session_ranges[
            session_ids[start]
        ] = (
            start,
            i,
        )

        start = i


session_ranges[
    session_ids[start]
] = (
    start,
    len(session_ids),
)


assert len(
    session_ranges
) == retrieval_queries_r2[
    "session_id"
].nunique()


print(
    "Session ranges:",
    f"{len(session_ranges):,}",
)


# ==============================================================================
# 15. OBJECTIVE UID → EMBEDDING ROW
# ==============================================================================

objective_index_r2 = pd.read_parquet(
    R2_OBJECTIVE_INDEX_PATH,
    engine="pyarrow",
)


assert len(
    objective_index_r2
) == 398


assert objective_index_r2[
    "objective_uid"
].is_unique


assert (
    "objective_uid"
    in objective_index_r2.columns
)


assert (
    "embedding_row"
    in objective_index_r2.columns
)


objective_row_by_uid = dict(
    zip(
        objective_index_r2[
            "objective_uid"
        ].astype(str),
        objective_index_r2[
            "embedding_row"
        ].astype(int),
    )
)


retrieval_queries_r2[
    "objective_uid"
] = (
    retrieval_queries_r2[
        "objective_uid"
    ]
    .astype(str)
)


retrieval_queries_r2[
    "session_id"
] = (
    retrieval_queries_r2[
        "session_id"
    ]
    .astype(str)
)


retrieval_queries_r2[
    "objective_row"
] = (
    retrieval_queries_r2[
        "objective_uid"
    ]
    .map(
        objective_row_by_uid
    )
)


assert retrieval_queries_r2[
    "objective_row"
].notna().all()


retrieval_queries_r2[
    "objective_row"
] = (
    retrieval_queries_r2[
        "objective_row"
    ]
    .astype(np.int32)
)


# ==============================================================================
# 16. CHECKPOINT HELPERS
# ==============================================================================

def r2_dense_write_json(
    path,
    payload,
):

    tmp = Path(
        str(path)
        + ".tmp"
    )

    with open(
        tmp,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            payload,
            f,
            indent=2,
            ensure_ascii=False,
        )

        f.flush()

    tmp.replace(
        path
    )


def r2_dense_load_checkpoint():

    if not R2_DENSE_CHECKPOINT.exists():

        return None

    with open(
        R2_DENSE_CHECKPOINT,
        "r",
        encoding="utf-8",
    ) as f:

        return json.load(
            f
        )


def r2_dense_part_path(
    part_number,
):

    return (
        R2_DENSE_PARTS_ROOT
        / f"part_{part_number:05d}.parquet"
    )


# ==============================================================================
# 17. RESUME STATE
# ==============================================================================

checkpoint = (
    r2_dense_load_checkpoint()
)


if checkpoint is None:

    completed_responses = 0

    next_part = 0

    committed_candidate_rows = 0

    print(
        "\nNo checkpoint found."
    )

    print(
        "Starting from response 0."
    )

else:

    assert (
        checkpoint[
            "total_responses"
        ]
        ==
        len(
            retrieval_queries_r2
        )
    )


    completed_responses = int(
        checkpoint[
            "completed_responses"
        ]
    )


    next_part = int(
        checkpoint[
            "next_part"
        ]
    )


    committed_candidate_rows = int(
        checkpoint.get(
            "candidate_rows_committed",
            0,
        )
    )


    print(
        "\nExisting checkpoint found."
    )

    print(
        "Completed responses:",
        f"{completed_responses:,}",
    )

    print(
        "Next part:",
        next_part,
    )

    print(
        "Committed candidate rows:",
        f"{committed_candidate_rows:,}",
    )


# ==============================================================================
# 18. SINGLE RESPONSE DENSE SCORER
# ==============================================================================

def score_r2_dense_response(
    query_row,
):

    response_id = str(
        query_row[
            "response_id"
        ]
    )

    session_id = str(
        query_row[
            "session_id"
        ]
    )

    objective_uid = str(
        query_row[
            "objective_uid"
        ]
    )

    fold = int(
        query_row[
            "fold"
        ]
    )

    objective_row = int(
        query_row[
            "objective_row"
        ]
    )


    assert session_id in (
        session_ranges
    )


    session_start, session_end = (
        session_ranges[
            session_id
        ]
    )


    objective_vector = np.asarray(
        objective_embeddings_r2[
            objective_row
        ],
        dtype=np.float32,
    )


    best_scores = np.empty(
        0,
        dtype=np.float32,
    )


    best_positions = np.empty(
        0,
        dtype=np.int64,
    )


    for block_start in range(
        session_start,
        session_end,
        R2_DENSE_TURN_BLOCK,
    ):

        block_end = min(
            block_start
            + R2_DENSE_TURN_BLOCK,
            session_end,
        )


        block_embeddings = np.asarray(
            turn_embeddings_r2[
                block_start:block_end
            ],
            dtype=np.float32,
        )


        scores = (
            block_embeddings
            @
            objective_vector
        )


        positions = np.arange(
            block_start,
            block_end,
            dtype=np.int64,
        )


        if len(
            best_scores
        ) == 0:

            combined_scores = (
                scores
            )

            combined_positions = (
                positions
            )

        else:

            combined_scores = (
                np.concatenate(
                    [
                        best_scores,
                        scores,
                    ]
                )
            )

            combined_positions = (
                np.concatenate(
                    [
                        best_positions,
                        positions,
                    ]
                )
            )


        keep = min(
            R2_DENSE_TOP_K,
            len(
                combined_scores
            ),
        )


        if (
            len(
                combined_scores
            )
            > keep
        ):

            idx = np.argpartition(
                -combined_scores,
                keep - 1,
            )[
                :keep
            ]


            best_scores = (
                combined_scores[
                    idx
                ]
            )


            best_positions = (
                combined_positions[
                    idx
                ]
            )

        else:

            best_scores = (
                combined_scores
            )

            best_positions = (
                combined_positions
            )


    # --------------------------------------------------------------------------
    # Deterministic ordering
    # --------------------------------------------------------------------------

    order = pd.DataFrame(
        {
            "position":
                best_positions,

            "score":
                best_scores,
        }
    )


    order[
        "turn_index"
    ] = turn_indices[
        best_positions
    ]


    order[
        "turn_uid"
    ] = turn_uids[
        best_positions
    ]


    order = order.sort_values(
        [
            "score",
            "turn_index",
            "turn_uid",
        ],
        ascending=[
            False,
            True,
            True,
        ],
        kind="mergesort",
    ).reset_index(
        drop=True
    )


    selected = (
        order[
            "position"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )


    result = pd.DataFrame(
        {
            "response_id":
                response_id,

            "session_id":
                session_id,

            "objective_uid":
                objective_uid,

            "fold":
                fold,

            "turn_uid":
                turn_uids[
                    selected
                ],

            "role":
                roles[
                    selected
                ],

            "turn_index":
                turn_indices[
                    selected
                ],

            "dense_score":
                order[
                    "score"
                ].to_numpy(
                    dtype=np.float32
                ),

            "retrieval_rank":
                np.arange(
                    len(
                        selected
                    ),
                    dtype=np.int16,
                ),

            "text_norm":
                texts[
                    selected
                ],
        }
    )


    # --------------------------------------------------------------------------
    # Hard session boundary
    # --------------------------------------------------------------------------

    assert (
        result[
            "session_id"
        ]
        == session_id
    ).all()


    # --------------------------------------------------------------------------
    # Score validity
    # --------------------------------------------------------------------------

    assert np.isfinite(
        result[
            "dense_score"
        ].to_numpy()
    ).all()


    assert (
        result[
            "dense_score"
        ]
        .between(
            -1.0001,
            1.0001,
        )
        .all()
    )


    # --------------------------------------------------------------------------
    # Rank validity
    # --------------------------------------------------------------------------

    assert (
        result[
            "retrieval_rank"
        ]
        .is_monotonic_increasing
    )


    return result


# ==============================================================================
# 19. SMOKE TEST
# ==============================================================================

print("\n" + "=" * 80)
print("DENSE RETRIEVAL SMOKE TEST")
print("=" * 80)


smoke_row = (
    retrieval_queries_r2.iloc[
        0
    ]
)


smoke_result = (
    score_r2_dense_response(
        smoke_row
    )
)


smoke_session = str(
    smoke_row[
        "session_id"
    ]
)


smoke_start, smoke_end = (
    session_ranges[
        smoke_session
    ]
)


expected_smoke_rows = min(
    R2_DENSE_TOP_K,
    smoke_end
    - smoke_start,
)


assert len(
    smoke_result
) == expected_smoke_rows


assert (
    smoke_result[
        "session_id"
    ].nunique()
    == 1
)


assert (
    smoke_result[
        "session_id"
    ].iloc[0]
    == smoke_session
)


print(
    "Response:",
    smoke_row[
        "response_id"
    ],
)


print(
    "Session:",
    smoke_session,
)


print(
    "Session turns:",
    smoke_end
    - smoke_start,
)


print(
    "Dense candidates:",
    len(
        smoke_result
    ),
)


print(
    "Cross-session contamination: 0"
)


print(
    "Smoke test: PASS"
)


del smoke_result
del smoke_row

gc.collect()


# ==============================================================================
# 20. FULL RESUMABLE RETRIEVAL
# ==============================================================================

print("\n" + "=" * 80)
print("FULL DENSE SESSION-LOCAL RETRIEVAL")
print("=" * 80)


total_responses = len(
    retrieval_queries_r2
)


cursor = (
    completed_responses
)


part_number = (
    next_part
)


run_started = time.time()


while (
    cursor
    < total_responses
):

    batch_end = min(
        cursor
        + R2_DENSE_RESPONSE_BATCH,
        total_responses,
    )


    batch_queries = (
        retrieval_queries_r2.iloc[
            cursor:batch_end
        ]
    )


    batch_results = []


    batch_started = (
        time.time()
    )


    for _, query_row in (
        batch_queries.iterrows()
    ):

        batch_results.append(
            score_r2_dense_response(
                query_row
            )
        )


    batch_result = pd.concat(
        batch_results,
        ignore_index=True,
    )


    # --------------------------------------------------------------------------
    # Expected candidate rows
    # --------------------------------------------------------------------------

    expected_rows = 0


    for _, query_row in (
        batch_queries.iterrows()
    ):

        s = str(
            query_row[
                "session_id"
            ]
        )


        session_start, session_end = (
            session_ranges[
                s
            ]
        )


        expected_rows += min(
            R2_DENSE_TOP_K,
            session_end
            - session_start,
        )


    assert len(
        batch_result
    ) == expected_rows


    # --------------------------------------------------------------------------
    # Response → session contract
    # --------------------------------------------------------------------------

    response_session = dict(
        zip(
            batch_queries[
                "response_id"
            ].astype(str),

            batch_queries[
                "session_id"
            ].astype(str),
        )
    )


    expected_sessions = (
        batch_result[
            "response_id"
        ]
        .astype(str)
        .map(
            response_session
        )
    )


    assert (
        batch_result[
            "session_id"
        ].astype(str)
        ==
        expected_sessions
    ).all()


    # --------------------------------------------------------------------------
    # Response uniqueness inside batch
    # --------------------------------------------------------------------------

    assert (
        batch_result[
            "response_id"
        ]
        .nunique()
        ==
        len(
            batch_queries
        )
    )


    # --------------------------------------------------------------------------
    # Immutable part path
    # --------------------------------------------------------------------------

    part_path = (
        r2_dense_part_path(
            part_number
        )
    )


    # --------------------------------------------------------------------------
    # Write / verify part
    # --------------------------------------------------------------------------

    if part_path.exists():

        existing = pd.read_parquet(
            part_path,
            engine="pyarrow",
        )


        assert len(
            existing
        ) == len(
            batch_result
        )


        assert set(
            existing[
                "response_id"
            ].astype(str)
        ) == set(
            batch_result[
                "response_id"
            ].astype(str)
        )


        del existing


    else:

        batch_result.to_parquet(
            part_path,
            engine="pyarrow",
            index=False,
        )


    assert part_path.exists()


    # --------------------------------------------------------------------------
    # COMMIT CHECKPOINT AFTER PART WRITE
    # --------------------------------------------------------------------------

    cursor = batch_end

    part_number += 1

    committed_candidate_rows += (
        len(
            batch_result
        )
    )


    elapsed = (
        time.time()
        - run_started
    )


    rate = (
        cursor / elapsed
        if elapsed > 0
        else 0
    )


    remaining = (
        total_responses
        - cursor
    )


    eta_seconds = (
        remaining / rate
        if rate > 0
        else None
    )


    checkpoint_payload = {
        "status":
            (
                "COMPLETE"
                if cursor
                == total_responses
                else "RUNNING"
            ),

        "completed_responses":
            int(
                cursor
            ),

        "total_responses":
            int(
                total_responses
            ),

        "next_part":
            int(
                part_number
            ),

        "top_k":
            int(
                R2_DENSE_TOP_K
            ),

        "candidate_rows_committed":
            int(
                committed_candidate_rows
            ),

        "embedding_dimension":
            int(
                R2_EMBEDDING_DIM
            ),

        "candidate_scope":
            "same_session_only",

        "similarity":
            "cosine",

        "target_used":
            False,

        "updated_at":
            time.strftime(
                "%Y-%m-%dT%H:%M:%S"
            ),
    }


    r2_dense_write_json(
        R2_DENSE_CHECKPOINT,
        checkpoint_payload,
    )


    batch_elapsed = (
        time.time()
        - batch_started
    )


    print(
        f"Processed "
        f"{cursor:,}/"
        f"{total_responses:,} "
        f"responses | "
        f"candidate rows="
        f"{len(batch_result):,} | "
        f"batch="
        f"{batch_elapsed:.1f}s"
        +
        (
            f" | ETA="
            f"{eta_seconds / 3600:.2f}h"
            if eta_seconds
            is not None
            else ""
        )
    )


    del batch_results
    del batch_result
    del batch_queries

    gc.collect()


# ==============================================================================
# 21. FINAL PART DISCOVERY
# ==============================================================================

dense_parts = sorted(
    R2_DENSE_PARTS_ROOT.glob(
        "part_*.parquet"
    )
)


assert len(
    dense_parts
) > 0


print("\n" + "=" * 80)
print("FINAL DENSE PART DISCOVERY")
print("=" * 80)

print(
    "Partitions:",
    len(
        dense_parts
    ),
)


# ==============================================================================
# 22. FINAL POPULATION AUDIT
# ==============================================================================

total_candidate_rows = 0

response_ids_seen = set()

session_ids_seen = set()


response_to_session = dict(
    zip(
        retrieval_queries_r2[
            "response_id"
        ].astype(str),

        retrieval_queries_r2[
            "session_id"
        ].astype(str),
    )
)


for part_path in dense_parts:

    part = pd.read_parquet(
        part_path,
        engine="pyarrow",
    )


    total_candidate_rows += (
        len(
            part
        )
    )


    response_ids_seen.update(
        part[
            "response_id"
        ].astype(str)
    )


    session_ids_seen.update(
        part[
            "session_id"
        ].astype(str)
    )


    # --------------------------------------------------------------------------
    # Hard cross-session contamination check
    # --------------------------------------------------------------------------

    expected = (
        part[
            "response_id"
        ]
        .astype(str)
        .map(
            response_to_session
        )
    )


    assert (
        part[
            "session_id"
        ].astype(str)
        ==
        expected
    ).all()


    # --------------------------------------------------------------------------
    # Dense score validity
    # --------------------------------------------------------------------------

    assert np.isfinite(
        part[
            "dense_score"
        ].to_numpy()
    ).all()


    assert (
        part[
            "dense_score"
        ]
        .between(
            -1.0001,
            1.0001,
        )
        .all()
    )


    del part


# ------------------------------------------------------------------------------
# Exact response coverage
# ------------------------------------------------------------------------------

assert (
    response_ids_seen
    ==
    set(
        retrieval_queries_r2[
            "response_id"
        ].astype(str)
    )
)


# ==============================================================================
# 23. EXACT EXPECTED TOP-K POPULATION
# ==============================================================================

expected_candidate_rows = 0


for session_id in (
    retrieval_queries_r2[
        "session_id"
    ].astype(str)
):

    session_start, session_end = (
        session_ranges[
            session_id
        ]
    )


    expected_candidate_rows += min(
        R2_DENSE_TOP_K,
        session_end
        - session_start,
    )


assert (
    total_candidate_rows
    ==
    expected_candidate_rows
)


# ==============================================================================
# 24. FINAL CHECKPOINT CONSISTENCY
# ==============================================================================

final_checkpoint = (
    r2_dense_load_checkpoint()
)


assert final_checkpoint is not None


assert (
    final_checkpoint[
        "status"
    ]
    == "COMPLETE"
)


assert int(
    final_checkpoint[
        "completed_responses"
    ]
) == total_responses


assert int(
    final_checkpoint[
        "candidate_rows_committed"
    ]
) == total_candidate_rows


# ==============================================================================
# 25. FINAL MANIFEST
# ==============================================================================

manifest = {
    "artifact":
        "r2_dense_session_local_candidates",

    "status":
        "FROZEN_READY",

    "responses":
        int(
            total_responses
        ),

    "sessions":
        int(
            len(
                session_ids_seen
            )
        ),

    "candidate_rows":
        int(
            total_candidate_rows
        ),

    "top_k":
        int(
            R2_DENSE_TOP_K
        ),

    "embedding_dimension":
        int(
            R2_EMBEDDING_DIM
        ),

    "candidate_scope":
        "same_session_only",

    "similarity":
        "cosine",

    "target_used":
        False,

    "cross_session_contamination":
        False,

    "partition_count":
        int(
            len(
                dense_parts
            )
        ),

    "parts_root":
        str(
            R2_DENSE_PARTS_ROOT.resolve()
        ),

    "turn_embeddings_path":
        str(
            R2_TURN_EMBEDDINGS_PATH.resolve()
        ),

    "objective_embeddings_path":
        str(
            R2_OBJECTIVE_EMBEDDINGS_PATH.resolve()
        ),

    "objective_index_path":
        str(
            R2_OBJECTIVE_INDEX_PATH.resolve()
        ),

    "created_at":
        time.strftime(
            "%Y-%m-%dT%H:%M:%S"
        ),
}


r2_dense_write_json(
    R2_DENSE_MANIFEST,
    manifest,
)


# ==============================================================================
# 26. FINAL FLAGS
# ==============================================================================

R2_DENSE_RETRIEVAL_READY = True

R2_CELL_5_READY = True


print("\n" + "=" * 80)
print("R2 CELL 5 STATUS")
print("=" * 80)

print(
    "Responses scored:",
    f"{len(response_ids_seen):,}",
)

print(
    "Expected responses:",
    f"{total_responses:,}",
)

print(
    "Candidate rows:",
    f"{total_candidate_rows:,}",
)

print(
    "Expected candidate rows:",
    f"{expected_candidate_rows:,}",
)

print(
    "Sessions:",
    f"{len(session_ids_seen):,}",
)

print(
    "Cross-session contamination: 0"
)

print(
    "Target used: False"
)

print(
    "Checkpoint:",
    R2_DENSE_CHECKPOINT.exists(),
)

print(
    "Checkpoint status:",
    final_checkpoint[
        "status"
    ],
)

print(
    "Manifest:",
    R2_DENSE_MANIFEST.exists(),
)

print(
    "Objective embedding:",
    R2_OBJECTIVE_EMBEDDINGS_PATH,
)

print(
    "Objective index:",
    R2_OBJECTIVE_INDEX_PATH,
)

print(
    "R2_DENSE_RETRIEVAL_READY:",
    R2_DENSE_RETRIEVAL_READY,
)

print(
    "R2_CELL_5_READY:",
    R2_CELL_5_READY,
)

print("=" * 80)
print(
    "R2 CELL 5 — DENSE SESSION-LOCAL RETRIEVAL: PASS"
)
print("=" * 80)


TRACE THE ACE — R2 DENSE RETRIEVAL
CELL 5 — RESUMABLE SESSION-LOCAL DENSE RETRIEVAL

R2 DISK-BASED DEPENDENCY VERIFICATION

R2 OBJECTIVE EMBEDDING ARTIFACT DISCOVERY
Objective .npy candidates:
 - D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R2_dense\embeddings\objective_embeddings\objective_embeddings.float32.npy | 611,456 bytes

Selected objective embedding: D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R2_dense\embeddings\objective_embeddings\objective_embeddings.float32.npy

Shape: (398, 384)
Dtype: float32
Finite values: PASS
L2 normalization: PASS

Objective index candidates:
 - D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R2_dense\embeddings\objective_embeddings\objective_embedding_index.parquet
Selected objective index: D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R2_dense\embeddings\objective_embeddings\objective_embedding_index.parquet

Objective index rows: 398
Obje

In [19]:
# ==============================================================================
# TRACE THE ACE — R2 DENSE RETRIEVAL
# CELL 6 — FINAL DENSE CANDIDATE FREEZE + INTEGRITY MANIFEST
# ==============================================================================

from pathlib import Path
import json
import gc
import time
import hashlib

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


print("\n" + "=" * 80)
print("TRACE THE ACE — R2 DENSE RETRIEVAL")
print("CELL 6 — FINAL DENSE CANDIDATE FREEZE + INTEGRITY MANIFEST")
print("=" * 80)


# ==============================================================================
# 1. PROJECT PATHS
# ==============================================================================

PROJECT_ROOT = Path(
    r"D:\Competition\Trace-the-race-local"
)

SCRATCH_ROOT = (
    PROJECT_ROOT
    / "scratch_mastery_outputs"
)


R0_ROOT = (
    SCRATCH_ROOT
    / "02_retrieval"
    / "R0_input"
)


R2_ROOT = (
    SCRATCH_ROOT
    / "02_retrieval"
    / "R2_dense"
)


# ==============================================================================
# 2. R0 DEPENDENCIES
# ==============================================================================

R0_RETRIEVAL_QUERIES = (
    R0_ROOT
    / "retrieval_queries.parquet"
)

R0_SESSION_TURN_INDEX = (
    R0_ROOT
    / "session_turn_index.parquet"

)


# ==============================================================================
# 3. R2 CELL 5 OUTPUT
# ==============================================================================

R2_DENSE_ROOT = (
    R2_ROOT
    / "dense_retrieval"
)

R2_DENSE_PARTS_ROOT = (
    R2_DENSE_ROOT
    / "parts"
)

R2_DENSE_CHECKPOINT = (
    R2_DENSE_ROOT
    / "r2_dense_retrieval_checkpoint.json"
)


# ==============================================================================
# 4. FINAL FREEZE PATHS
# ==============================================================================

R2_FREEZE_ROOT = (
    R2_ROOT
    / "frozen"
)


R2_FREEZE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


R2_DENSE_CANDIDATES_FINAL = (
    R2_FREEZE_ROOT
    / "r2_dense_candidates.parquet"
)


R2_DENSE_CANDIDATES_TMP = (
    R2_FREEZE_ROOT
    / ".r2_dense_candidates.parquet.tmp"
)


R2_DENSE_FREEZE_MANIFEST = (
    R2_FREEZE_ROOT
    / "r2_dense_freeze_manifest.json"
)


print("\n" + "=" * 80)
print("R2 FREEZE PATHS")
print("=" * 80)

print(
    "Freeze root:",
    R2_FREEZE_ROOT,
)

print(
    "Candidates:",
    R2_DENSE_CANDIDATES_FINAL,
)

print(
    "Manifest:",
    R2_DENSE_FREEZE_MANIFEST,
)


# ==============================================================================
# 5. HARD DEPENDENCY VERIFICATION
# ==============================================================================

assert R0_RETRIEVAL_QUERIES.exists(), (
    "R0 retrieval_queries.parquet is missing."
)

assert R0_SESSION_TURN_INDEX.exists(), (
    "R0 session_turn_index.parquet is missing."
)

assert R2_DENSE_PARTS_ROOT.exists(), (
    "R2 Cell 5 parts directory is missing."
)

assert R2_DENSE_CHECKPOINT.exists(), (
    "R2 Cell 5 checkpoint is missing."
)


with open(
    R2_DENSE_CHECKPOINT,
    "r",
    encoding="utf-8",
) as f:

    r2_checkpoint = json.load(
        f
    )


assert (
    r2_checkpoint.get(
        "status"
    )
    == "COMPLETE"
), (
    "R2 Cell 5 checkpoint is not COMPLETE. "
    "Do not freeze an incomplete retrieval."
)


assert int(
    r2_checkpoint[
        "completed_responses"
    ]
) == 35_072, (
    "R2 Cell 5 did not complete all 35,072 responses."
)


print("\n" + "=" * 80)
print("R2 CELL 5 DEPENDENCY")
print("=" * 80)

print(
    "Checkpoint:",
    "PASS",
)

print(
    "Status:",
    r2_checkpoint[
        "status"
    ],
)

print(
    "Completed responses:",
    f"{int(r2_checkpoint['completed_responses']):,}",
)


# ==============================================================================
# 6. LOAD R0 RESPONSE CONTRACT
# ==============================================================================

retrieval_queries_r2 = pd.read_parquet(
    R0_RETRIEVAL_QUERIES,
    engine="pyarrow",
)


assert len(
    retrieval_queries_r2
) == 35_072


assert (
    retrieval_queries_r2[
        "response_id"
    ].astype(str).is_unique
)


retrieval_queries_r2[
    "response_id"
] = (
    retrieval_queries_r2[
        "response_id"
    ].astype(str)
)


retrieval_queries_r2[
    "session_id"
] = (
    retrieval_queries_r2[
        "session_id"
    ].astype(str)
)


# ==============================================================================
# 7. LOAD SESSION CENSUS
# ==============================================================================

session_turn_r2 = pd.read_parquet(
    R0_SESSION_TURN_INDEX,
    columns=[
        "session_id",
        "turn_uid",
        "turn_index",
    ],
    engine="pyarrow",
)


assert len(
    session_turn_r2
) == 6_139_854


session_turn_counts = (
    session_turn_r2
    .groupby(
        "session_id",
        sort=False,
    )
    .size()
)


session_turn_counts.index = (
    session_turn_counts.index
    .astype(str)
)


# ==============================================================================
# 8. EXPECTED CANDIDATE COUNT
# ==============================================================================

R2_DENSE_TOP_K = int(
    r2_checkpoint[
        "top_k"
    ]
)


assert R2_DENSE_TOP_K == 50


retrieval_queries_r2[
    "expected_candidates"
] = (
    retrieval_queries_r2[
        "session_id"
    ]
    .map(
        session_turn_counts
    )
)


assert retrieval_queries_r2[
    "expected_candidates"
].notna().all()


retrieval_queries_r2[
    "expected_candidates"
] = (
    retrieval_queries_r2[
        "expected_candidates"
    ]
    .clip(
        upper=R2_DENSE_TOP_K
    )
    .astype(np.int32)
)


expected_candidate_rows = int(
    retrieval_queries_r2[
        "expected_candidates"
    ].sum()
)


print("\n" + "=" * 80)
print("EXPECTED R2 POPULATION")
print("=" * 80)

print(
    "Responses:",
    f"{len(retrieval_queries_r2):,}",
)

print(
    "Sessions:",
    f"{retrieval_queries_r2['session_id'].nunique():,}",
)

print(
    "Top-K:",
    R2_DENSE_TOP_K,
)

print(
    "Expected candidate rows:",
    f"{expected_candidate_rows:,}",
)


# ==============================================================================
# 9. DISCOVER ALL CELL 5 PARTS
# ==============================================================================

dense_parts = sorted(
    R2_DENSE_PARTS_ROOT.glob(
        "part_*.parquet"
    )
)


assert len(
    dense_parts
) > 0, (
    "No R2 dense retrieval partitions found."
)


print("\n" + "=" * 80)
print("R2 PARTITION DISCOVERY")
print("=" * 80)

print(
    "Partitions found:",
    len(
        dense_parts
    ),
)


for part_path in dense_parts:

    assert part_path.stat().st_size > 0

    print(
        " -",
        part_path.name,
        "|",
        f"{part_path.stat().st_size:,}",
        "bytes",
    )


# ==============================================================================
# 10. REQUIRED FROZEN SCHEMA
# ==============================================================================

EXPECTED_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "turn_uid",
    "role",
    "turn_index",
    "dense_score",
    "retrieval_rank",
    "text_norm",
]


# ==============================================================================
# 11. RESPONSE CONTRACT MAP
# ==============================================================================

response_session_map = dict(
    zip(
        retrieval_queries_r2[
            "response_id"
        ],
        retrieval_queries_r2[
            "session_id"
        ],
    )
)


response_objective_map = dict(
    zip(
        retrieval_queries_r2[
            "response_id"
        ],
        retrieval_queries_r2[
            "objective_uid"
        ].astype(str),
    )
)


response_fold_map = dict(
    zip(
        retrieval_queries_r2[
            "response_id"
        ],
        retrieval_queries_r2[
            "fold"
        ].astype(int),
    )
)


response_expected_count_map = dict(
    zip(
        retrieval_queries_r2[
            "response_id"
        ],
        retrieval_queries_r2[
            "expected_candidates"
        ].astype(int),
    )
)


# ==============================================================================
# 12. PART-BY-PART VALIDATION
# ==============================================================================

print("\n" + "=" * 80)
print("PART-BY-PART INTEGRITY AUDIT")
print("=" * 80)


all_response_ids = set()

total_part_rows = 0

part_sha256 = {}


for part_number, part_path in enumerate(
    dense_parts
):

    print(
        f"\nValidating "
        f"{part_path.name} "
        f"({part_number + 1}/{len(dense_parts)})"
    )


    part = pd.read_parquet(
        part_path,
        engine="pyarrow",
    )


    # --------------------------------------------------------------------------
    # Schema
    # --------------------------------------------------------------------------

    assert list(
        part.columns
    ) == EXPECTED_COLUMNS, (
        f"Schema mismatch in {part_path.name}."
    )


    # --------------------------------------------------------------------------
    # Basic population
    # --------------------------------------------------------------------------

    assert len(
        part
    ) > 0


    total_part_rows += len(
        part
    )


    # --------------------------------------------------------------------------
    # Response identity
    # --------------------------------------------------------------------------

    part[
        "response_id"
    ] = (
        part[
            "response_id"
        ].astype(str)
    )


    part[
        "session_id"
    ] = (
        part[
            "session_id"
        ].astype(str)
    )


    part[
        "objective_uid"
    ] = (
        part[
            "objective_uid"
        ].astype(str)
    )


    assert part[
        "response_id"
    ].notna().all()


    assert part[
        "session_id"
    ].notna().all()


    # --------------------------------------------------------------------------
    # Response-level session contract
    # --------------------------------------------------------------------------

    expected_sessions = (
        part[
            "response_id"
        ].map(
            response_session_map
        )
    )


    assert (
        part[
            "session_id"
        ]
        ==
        expected_sessions
    ).all(), (
        f"Cross-session contamination detected "
        f"in {part_path.name}."
    )


    # --------------------------------------------------------------------------
    # Objective contract
    # --------------------------------------------------------------------------

    expected_objectives = (
        part[
            "response_id"
        ].map(
            response_objective_map
        )
    )


    assert (
        part[
            "objective_uid"
        ]
        ==
        expected_objectives
    ).all(), (
        f"Objective mismatch in {part_path.name}."
    )


    # --------------------------------------------------------------------------
    # Fold contract
    # --------------------------------------------------------------------------

    expected_folds = (
        part[
            "response_id"
        ].map(
            response_fold_map
        )
    )


    assert (
        part[
            "fold"
        ].astype(int)
        ==
        expected_folds
    ).all(), (
        f"Fold mismatch in {part_path.name}."
    )


    # --------------------------------------------------------------------------
    # Dense score contract
    # --------------------------------------------------------------------------

    dense_scores = (
        part[
            "dense_score"
        ].to_numpy()
    )


    assert np.isfinite(
        dense_scores
    ).all(), (
        f"Non-finite dense score in "
        f"{part_path.name}."
    )


    assert (
        dense_scores
        >= -1.0001
    ).all()


    assert (
        dense_scores
        <= 1.0001
    ).all()


    # --------------------------------------------------------------------------
    # Rank contract
    # --------------------------------------------------------------------------

    ranks = (
        part[
            "retrieval_rank"
        ].to_numpy()
    )


    assert np.isfinite(
        ranks
    ).all()


    assert (
        ranks
        >= 0
    ).all()


    assert (
        ranks
        < R2_DENSE_TOP_K
    ).all()


    # --------------------------------------------------------------------------
    # Turn identity
    # --------------------------------------------------------------------------

    assert part[
        "turn_uid"
    ].notna().all()


    assert part[
        "turn_index"
    ].notna().all()


    # --------------------------------------------------------------------------
    # Candidate rows per response
    # --------------------------------------------------------------------------

    observed_counts = (
        part
        .groupby(
            "response_id",
            sort=False,
        )
        .size()
    )


    for response_id, observed_count in (
        observed_counts.items()
    ):

        expected_count = (
            response_expected_count_map[
                response_id
            ]
        )


        assert (
            int(
                observed_count
            )
            ==
            int(
                expected_count
            )
        ), (
            f"Candidate count mismatch for "
            f"response {response_id}."
        )


    # --------------------------------------------------------------------------
    # No response duplicated across partitions
    # --------------------------------------------------------------------------

    duplicate_response_ids = (
        set(
            part[
                "response_id"
            ]
        )
        &
        all_response_ids
    )


    assert not duplicate_response_ids, (
        "Response appears in multiple "
        "R2 retrieval partitions: "
        f"{list(duplicate_response_ids)[:10]}"
    )


    all_response_ids.update(
        part[
            "response_id"
        ]
    )


    # --------------------------------------------------------------------------
    # Part SHA256
    # --------------------------------------------------------------------------

    sha = hashlib.sha256()


    with open(
        part_path,
        "rb",
    ) as f:

        while True:

            chunk = f.read(
                8 * 1024 * 1024
            )

            if not chunk:
                break

            sha.update(
                chunk
            )


    part_sha256[
        part_path.name
    ] = sha.hexdigest()


    print(
        " rows:",
        f"{len(part):,}",
        "| responses:",
        f"{part['response_id'].nunique():,}",
        "| PASS",
    )


    del part

    gc.collect()


# ==============================================================================
# 13. EXACT GLOBAL RESPONSE COVERAGE
# ==============================================================================

expected_response_ids = set(
    retrieval_queries_r2[
        "response_id"
    ]
)


assert (
    all_response_ids
    ==
    expected_response_ids
), (
    "Global response coverage mismatch."
)


assert len(
    all_response_ids
) == 35_072


# ==============================================================================
# 14. EXACT GLOBAL POPULATION
# ==============================================================================

assert (
    total_part_rows
    ==
    expected_candidate_rows
), (
    "Global dense candidate population mismatch: "
    f"observed={total_part_rows:,}, "
    f"expected={expected_candidate_rows:,}"
)


# ==============================================================================
# 15. CONSOLIDATED FROZEN PARQUET
# ==============================================================================

print("\n" + "=" * 80)
print("BUILDING CONSOLIDATED FROZEN ARTIFACT")
print("=" * 80)


# ------------------------------------------------------------------------------
# Remove only our own stale temporary output.
# ------------------------------------------------------------------------------

if R2_DENSE_CANDIDATES_TMP.exists():

    try:

        R2_DENSE_CANDIDATES_TMP.unlink()

        print(
            "Removed stale temporary freeze artifact."
        )

    except PermissionError as exc:

        raise RuntimeError(
            "Temporary freeze artifact is locked by "
            "another process. Close any viewer/reader "
            "holding the parquet before rerunning Cell 6."
        ) from exc


# ------------------------------------------------------------------------------
# Write through PyArrow ParquetWriter so the complete 1.7M-row artifact does
# not need to be materialized into one giant pandas DataFrame.
# ------------------------------------------------------------------------------

writer = None

frozen_rows_written = 0


try:

    for part_number, part_path in enumerate(
        dense_parts
    ):

        part_table = (
            pq.read_table(
                part_path
            )
        )


        if writer is None:

            writer = pq.ParquetWriter(
                R2_DENSE_CANDIDATES_TMP,
                part_table.schema,
                compression="zstd",
                use_dictionary=True,
                write_statistics=True,
            )


        else:

            assert (
                part_table.schema
                ==
                writer.schema
            ), (
                "Partition schema differs while "
                "building consolidated frozen artifact."
            )


        writer.write_table(
            part_table
        )


        frozen_rows_written += (
            part_table.num_rows
        )


        print(
            f"Frozen "
            f"{part_number + 1:,}/"
            f"{len(dense_parts):,} partitions | "
            f"rows="
            f"{frozen_rows_written:,}"
        )


        del part_table

        gc.collect()


finally:

    if writer is not None:

        writer.close()


assert R2_DENSE_CANDIDATES_TMP.exists()


assert (
    frozen_rows_written
    ==
    expected_candidate_rows
)


# ==============================================================================
# 16. ATOMIC PUBLISH
# ==============================================================================

print("\n" + "=" * 80)
print("PUBLISHING FROZEN ARTIFACT")
print("=" * 80)


if R2_DENSE_CANDIDATES_FINAL.exists():

    R2_DENSE_CANDIDATES_FINAL.unlink()


R2_DENSE_CANDIDATES_TMP.replace(
    R2_DENSE_CANDIDATES_FINAL
)


assert R2_DENSE_CANDIDATES_FINAL.exists()


print(
    "Frozen candidate parquet: PASS"
)


# ==============================================================================
# 17. RELOAD FROZEN ARTIFACT
# ==============================================================================

print("\n" + "=" * 80)
print("FROZEN ARTIFACT RELOAD VERIFICATION")
print("=" * 80)


frozen_schema = pq.read_schema(
    R2_DENSE_CANDIDATES_FINAL
)


assert [
    field.name
    for field in frozen_schema
] == EXPECTED_COLUMNS


frozen_table = pq.read_table(
    R2_DENSE_CANDIDATES_FINAL,
    columns=[
        "response_id",
        "session_id",
        "objective_uid",
        "fold",
        "turn_uid",
        "turn_index",
        "dense_score",
        "retrieval_rank",
    ],
)


assert (
    frozen_table.num_rows
    ==
    expected_candidate_rows
)


frozen_df = (
    frozen_table
    .to_pandas()
)


assert (
    len(
        frozen_df
    )
    ==
    expected_candidate_rows
)


assert (
    frozen_df[
        "response_id"
    ].nunique()
    ==
    35_072
)


# ==============================================================================
# 18. FROZEN CROSS-SESSION AUDIT
# ==============================================================================

frozen_expected_sessions = (
    frozen_df[
        "response_id"
    ]
    .astype(str)
    .map(
        response_session_map
    )
)


frozen_foreign_session_rows = int(
    (
        frozen_df[
            "session_id"
        ].astype(str)
        !=
        frozen_expected_sessions
    ).sum()
)


assert (
    frozen_foreign_session_rows
    == 0
)


# ==============================================================================
# 19. FROZEN SCORE AUDIT
# ==============================================================================

frozen_scores = (
    frozen_df[
        "dense_score"
    ].to_numpy()
)


assert np.isfinite(
    frozen_scores
).all()


assert (
    frozen_scores
    >= -1.0001
).all()


assert (
    frozen_scores
    <= 1.0001
).all()


# ==============================================================================
# 20. FROZEN RANK AUDIT
# ==============================================================================

frozen_ranks = (
    frozen_df[
        "retrieval_rank"
    ].to_numpy()
)


assert (
    frozen_ranks
    >= 0
).all()


assert (
    frozen_ranks
    < R2_DENSE_TOP_K
).all()


# ==============================================================================
# 21. FROZEN RESPONSE CENSUS
# ==============================================================================

frozen_response_counts = (
    frozen_df
    .groupby(
        "response_id",
        sort=False,
    )
    .size()
)


expected_response_counts = (
    retrieval_queries_r2
    .set_index(
        "response_id"
    )[
        "expected_candidates"
    ]
)


aligned_frozen_counts = (
    frozen_response_counts
    .reindex(
        expected_response_counts.index
    )
)


assert (
    aligned_frozen_counts
    .notna()
    .all()
)


assert (
    aligned_frozen_counts.astype(int)
    ==
    expected_response_counts.astype(int)
).all()


# ==============================================================================
# 22. FINAL SHA256
# ==============================================================================

print("\n" + "=" * 80)
print("FINAL FROZEN SHA256")
print("=" * 80)


def sha256_file_r2_frozen(
    path,
    chunk_size=8 * 1024 * 1024,
):

    sha = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            sha.update(
                chunk
            )

    return sha.hexdigest()


R2_DENSE_CANDIDATES_SHA256 = (
    sha256_file_r2_frozen(
        R2_DENSE_CANDIDATES_FINAL
    )
)


print(
    "SHA256:",
    R2_DENSE_CANDIDATES_SHA256,
)


# ==============================================================================
# 23. MANIFEST
# ==============================================================================

manifest = {
    "artifact":
        "r2_dense_candidates",

    "status":
        "FROZEN",

    "version":
        "R2",

    "rows":
        int(
            expected_candidate_rows
        ),

    "responses":
        int(
            len(
                expected_response_ids
            )
        ),

    "sessions":
        int(
            retrieval_queries_r2[
                "session_id"
            ].nunique()
        ),

    "objectives":
        int(
            retrieval_queries_r2[
                "objective_uid"
            ].nunique()
        ),

    "top_k":
        int(
            R2_DENSE_TOP_K
        ),

    "embedding_dimension":
        384,

    "similarity":
        "cosine",

    "candidate_scope":
        "same_session_only",

    "target_used":
        False,

    "cross_session_contamination":
        False,

    "response_coverage":
        True,

    "exact_population":
        True,

    "schema_valid":
        True,

    "serialization_valid":
        True,

    "source_partition_count":
        int(
            len(
                dense_parts
            )
        ),

    "source_part_sha256":
        part_sha256,

    "artifact":
        "r2_dense_candidates",

    "frozen_candidate_path":
        str(
            R2_DENSE_CANDIDATES_FINAL.resolve()
        ),

    "sha256":
        R2_DENSE_CANDIDATES_SHA256,

    "created_at":
        time.strftime(
            "%Y-%m-%dT%H:%M:%S"
        ),
}


# ------------------------------------------------------------------------------
# Ensure all JSON values are native Python values.
# ------------------------------------------------------------------------------

def json_native_r2(
    value
):

    if isinstance(
        value,
        dict,
    ):

        return {
            str(k):
                json_native_r2(v)
            for k, v in value.items()
        }


    if isinstance(
        value,
        list,
    ):

        return [
            json_native_r2(v)
            for v in value
        ]


    if isinstance(
        value,
        tuple,
    ):

        return [
            json_native_r2(v)
            for v in value
        ]


    if isinstance(
        value,
        (
            np.bool_,
            bool,
        ),
    ):

        return bool(
            value
        )


    if isinstance(
        value,
        (
            np.integer,
        ),
    ):

        return int(
            value
        )


    if isinstance(
        value,
        (
            np.floating,
        ),
    ):

        return float(
            value
        )


    return value


manifest = json_native_r2(
    manifest
)


R2_DENSE_MANIFEST_TMP = Path(
    str(
        R2_DENSE_FREEZE_MANIFEST
    )
    + ".tmp"
)


with open(
    R2_DENSE_MANIFEST_TMP,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )

    f.flush()


R2_DENSE_MANIFEST_TMP.replace(
    R2_DENSE_FREEZE_MANIFEST
)


assert R2_DENSE_FREEZE_MANIFEST.exists()


# ==============================================================================
# 24. MANIFEST RELOAD TEST
# ==============================================================================

with open(
    R2_DENSE_FREEZE_MANIFEST,
    "r",
    encoding="utf-8",
) as f:

    manifest_reload = json.load(
        f
    )


assert (
    manifest_reload[
        "status"
    ]
    == "FROZEN"
)


assert (
    int(
        manifest_reload[
            "rows"
        ]
    )
    ==
    expected_candidate_rows
)


assert (
    int(
        manifest_reload[
            "responses"
        ]
    )
    ==
    35_072
)


assert (
    manifest_reload[
        "target_used"
    ]
    is False
)


assert (
    manifest_reload[
        "cross_session_contamination"
    ]
    is False
)


assert (
    manifest_reload[
        "sha256"
    ]
    ==
    R2_DENSE_CANDIDATES_SHA256
)


# ==============================================================================
# 25. FINAL FREEZE GATE
# ==============================================================================

R2_DENSE_CANDIDATES_FROZEN = True
R2_DENSE_FREEZE_READY = True
R2_CELL_6_READY = True


print("\n" + "=" * 80)
print("TRACE THE ACE — R2 DENSE RETRIEVAL")
print("CELL 6 — FINAL FREEZE STATUS")
print("=" * 80)

print(
    "Frozen candidate parquet :",
    R2_DENSE_CANDIDATES_FINAL.exists(),
)

print(
    "Frozen manifest          :",
    R2_DENSE_FREEZE_MANIFEST.exists(),
)

print(
    "Frozen rows              :",
    f"{expected_candidate_rows:,}",
)

print(
    "Expected rows            :",
    f"{expected_candidate_rows:,}",
)

print(
    "Responses                :",
    f"{len(expected_response_ids):,}",
)

print(
    "Sessions                 :",
    f"{retrieval_queries_r2['session_id'].nunique():,}",
)

print(
    "Top-K                    :",
    R2_DENSE_TOP_K,
)

print(
    "Cross-session rows       :",
    frozen_foreign_session_rows,
)

print(
    "Target used              :",
    False,
)

print(
    "Schema                   : PASS",
)

print(
    "Population               : PASS",
)

print(
    "Response coverage        : PASS",
)

print(
    "Score validity           : PASS",
)

print(
    "Serialization            : PASS",
)

print(
    "Manifest JSON             : VALID",
)

print(
    "Manifest SHA256          :",
    R2_DENSE_CANDIDATES_SHA256,
)

print(
    "R2_DENSE_CANDIDATES_FROZEN:",
    R2_DENSE_CANDIDATES_FROZEN,
)

print(
    "R2_DENSE_FREEZE_READY:",
    R2_DENSE_FREEZE_READY,
)

print(
    "R2_CELL_6_READY:",
    R2_CELL_6_READY,
)

print("=" * 80)
print(
    "R2 CELL 6 — DENSE CANDIDATE FREEZE: PASS"
)
print("=" * 80)


# ==============================================================================
# 26. MEMORY CLEANUP
# ==============================================================================

del frozen_df
del frozen_table
del session_turn_r2
del retrieval_queries_r2
del objective_embedding_index_r2

gc.collect()


print(
    "\nR2 Cell 6 memory cleanup: PASS"
)


TRACE THE ACE — R2 DENSE RETRIEVAL
CELL 6 — FINAL DENSE CANDIDATE FREEZE + INTEGRITY MANIFEST

R2 FREEZE PATHS
Freeze root: D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R2_dense\frozen
Candidates: D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R2_dense\frozen\r2_dense_candidates.parquet
Manifest: D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R2_dense\frozen\r2_dense_freeze_manifest.json

R2 CELL 5 DEPENDENCY
Checkpoint: PASS
Status: COMPLETE
Completed responses: 35,072

EXPECTED R2 POPULATION
Responses: 35,072
Sessions: 22,821
Top-K: 50
Expected candidate rows: 1,752,048

R2 PARTITION DISCOVERY
Partitions found: 351
 - part_00000.parquet | 720,024 bytes
 - part_00001.parquet | 717,368 bytes
 - part_00002.parquet | 761,679 bytes
 - part_00003.parquet | 739,805 bytes
 - part_00004.parquet | 726,394 bytes
 - part_00005.parquet | 716,909 bytes
 - part_00006.parquet | 725,125 bytes
 - part_00007.parquet | 730,

In [20]:
# ==============================================================================
# TRACE THE ACE — R2 DENSE RETRIEVAL
# CELL 7 — FROZEN DENSE CANDIDATE RELOAD + INTEGRITY VERIFICATION
# ==============================================================================

from pathlib import Path
import json
import hashlib
import gc

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


print("\n" + "=" * 80)
print("TRACE THE ACE — R2 DENSE RETRIEVAL")
print("CELL 7 — FROZEN DENSE CANDIDATE RELOAD + INTEGRITY VERIFICATION")
print("=" * 80)


# ==============================================================================
# 1. PROJECT PATHS
# ==============================================================================

PROJECT_ROOT = Path(
    r"D:\Competition\Trace-the-race-local"
)

SCRATCH_ROOT = (
    PROJECT_ROOT
    / "scratch_mastery_outputs"
)

R0_ROOT = (
    SCRATCH_ROOT
    / "02_retrieval"
    / "R0_input"
)

R2_ROOT = (
    SCRATCH_ROOT
    / "02_retrieval"
    / "R2_dense"
)

R2_FREEZE_ROOT = (
    R2_ROOT
    / "frozen"
)


R2_DENSE_CANDIDATES_FINAL = (
    R2_FREEZE_ROOT
    / "r2_dense_candidates.parquet"
)

R2_DENSE_FREEZE_MANIFEST = (
    R2_FREEZE_ROOT
    / "r2_dense_freeze_manifest.json"
)


R0_RETRIEVAL_QUERIES = (
    R0_ROOT
    / "retrieval_queries.parquet"
)


# ==============================================================================
# 2. EXPECTED CONTRACT
# ==============================================================================

EXPECTED_ROWS = 1_752_048
EXPECTED_RESPONSES = 35_072
EXPECTED_SESSIONS = 22_821
EXPECTED_TOP_K = 50

EXPECTED_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "turn_uid",
    "role",
    "turn_index",
    "dense_score",
    "retrieval_rank",
    "text_norm",
]


# ==============================================================================
# 3. ARTIFACT EXISTENCE
# ==============================================================================

print("\n" + "=" * 80)
print("FROZEN ARTIFACT EXISTENCE")
print("=" * 80)

print(
    "Frozen root :",
    R2_FREEZE_ROOT.exists(),
)

print(
    "Candidates  :",
    R2_DENSE_CANDIDATES_FINAL.exists(),
)

print(
    "Manifest    :",
    R2_DENSE_FREEZE_MANIFEST.exists(),
)


assert R2_FREEZE_ROOT.exists(), (
    f"R2 frozen directory missing:\n"
    f"{R2_FREEZE_ROOT}"
)

assert R2_DENSE_CANDIDATES_FINAL.exists(), (
    f"Frozen dense candidates missing:\n"
    f"{R2_DENSE_CANDIDATES_FINAL}"
)

assert R2_DENSE_FREEZE_MANIFEST.exists(), (
    f"Frozen dense manifest missing:\n"
    f"{R2_DENSE_FREEZE_MANIFEST}"
)


# ==============================================================================
# 4. MANIFEST RELOAD
# ==============================================================================

print("\n" + "=" * 80)
print("FROZEN MANIFEST")
print("=" * 80)


with open(
    R2_DENSE_FREEZE_MANIFEST,
    "r",
    encoding="utf-8",
) as f:

    r2_dense_manifest = json.load(
        f
    )


assert isinstance(
    r2_dense_manifest,
    dict,
)


assert (
    r2_dense_manifest.get(
        "status"
    )
    == "FROZEN"
), (
    "R2 dense manifest does not have "
    "status=FROZEN."
)


print(
    "JSON valid       : PASS"
)

print(
    "Status           :",
    r2_dense_manifest.get(
        "status"
    ),
)

print(
    "Artifact         :",
    r2_dense_manifest.get(
        "artifact"
    ),
)

print(
    "Rows             :",
    f"{int(r2_dense_manifest['rows']):,}",
)

print(
    "Responses        :",
    f"{int(r2_dense_manifest['responses']):,}",
)

print(
    "Sessions         :",
    f"{int(r2_dense_manifest['sessions']):,}",
)

print(
    "Top-K            :",
    r2_dense_manifest.get(
        "top_k"
    ),
)

print(
    "Target used      :",
    r2_dense_manifest.get(
        "target_used"
    ),
)

print(
    "Cross-session    :",
    r2_dense_manifest.get(
        "cross_session_contamination"
    ),
)


# ==============================================================================
# 5. MANIFEST CONTRACT
# ==============================================================================

assert (
    int(
        r2_dense_manifest[
            "rows"
        ]
    )
    == EXPECTED_ROWS
)

assert (
    int(
        r2_dense_manifest[
            "responses"
        ]
    )
    == EXPECTED_RESPONSES
)

assert (
    int(
        r2_dense_manifest[
            "sessions"
        ]
    )
    == EXPECTED_SESSIONS
)

assert (
    int(
        r2_dense_manifest[
            "top_k"
        ]
    )
    == EXPECTED_TOP_K
)

assert (
    r2_dense_manifest[
        "target_used"
    ]
    is False
)

assert (
    r2_dense_manifest[
        "cross_session_contamination"
    ]
    is False
)


# ==============================================================================
# 6. PARQUET SCHEMA VERIFICATION
# ==============================================================================

print("\n" + "=" * 80)
print("FROZEN PARQUET SCHEMA")
print("=" * 80)


frozen_schema = pq.read_schema(
    R2_DENSE_CANDIDATES_FINAL
)


observed_columns = [
    field.name
    for field in frozen_schema
]


print(
    "Columns:",
    observed_columns,
)


assert (
    observed_columns
    == EXPECTED_COLUMNS
), (
    "Frozen R2 candidate schema mismatch."
)


print(
    "Schema: PASS"
)


# ==============================================================================
# 7. PARQUET METADATA POPULATION CHECK
# ==============================================================================

frozen_metadata = (
    pq.ParquetFile(
        R2_DENSE_CANDIDATES_FINAL
    ).metadata
)


assert frozen_metadata is not None


metadata_rows = (
    frozen_metadata.num_rows
)


print(
    "\nParquet metadata rows:",
    f"{metadata_rows:,}",
)


assert (
    metadata_rows
    == EXPECTED_ROWS
), (
    "Frozen parquet row count does not match "
    "the frozen contract."
)


print(
    "Population metadata: PASS"
)


# ==============================================================================
# 8. LOAD FROZEN ARTIFACT
# ==============================================================================

print("\n" + "=" * 80)
print("RELOADING FROZEN DENSE CANDIDATES")
print("=" * 80)


r2_dense_candidates = pd.read_parquet(
    R2_DENSE_CANDIDATES_FINAL,
    engine="pyarrow",
)


print(
    "Reloaded rows:",
    f"{len(r2_dense_candidates):,}",
)


assert (
    len(r2_dense_candidates)
    == EXPECTED_ROWS
)


# ==============================================================================
# 9. RESPONSE COVERAGE
# ==============================================================================

print("\n" + "=" * 80)
print("RESPONSE COVERAGE")
print("=" * 80)


assert (
    r2_dense_candidates[
        "response_id"
    ].notna().all()
)


unique_response_count = (
    r2_dense_candidates[
        "response_id"
    ]
    .astype(str)
    .nunique()
)


print(
    "Unique responses:",
    f"{unique_response_count:,}",
)


assert (
    unique_response_count
    == EXPECTED_RESPONSES
)


print(
    "Response coverage: PASS"
)


# ==============================================================================
# 10. SESSION COVERAGE
# ==============================================================================

unique_session_count = (
    r2_dense_candidates[
        "session_id"
    ]
    .astype(str)
    .nunique()
)


print(
    "Unique sessions:",
    f"{unique_session_count:,}",
)


assert (
    unique_session_count
    == EXPECTED_SESSIONS
)


print(
    "Session coverage: PASS"
)


# ==============================================================================
# 11. RESPONSE → SESSION CONSISTENCY
# ==============================================================================

retrieval_queries_r2 = pd.read_parquet(
    R0_RETRIEVAL_QUERIES,
    engine="pyarrow",
)


retrieval_queries_r2[
    "response_id"
] = (
    retrieval_queries_r2[
        "response_id"
    ].astype(str)
)

retrieval_queries_r2[
    "session_id"
] = (
    retrieval_queries_r2[
        "session_id"
    ].astype(str)
)


assert (
    len(
        retrieval_queries_r2
    )
    == EXPECTED_RESPONSES
)


response_session_map = dict(
    zip(
        retrieval_queries_r2[
            "response_id"
        ],
        retrieval_queries_r2[
            "session_id"
        ],
    )
)


expected_session = (
    r2_dense_candidates[
        "response_id"
    ]
    .astype(str)
    .map(
        response_session_map
    )
)


foreign_session_mask = (
    r2_dense_candidates[
        "session_id"
    ].astype(str)
    !=
    expected_session
)


foreign_session_rows = int(
    foreign_session_mask.sum()
)


print(
    "\nForeign-session rows:",
    foreign_session_rows,
)


assert (
    foreign_session_rows
    == 0
), (
    "Cross-session contamination detected."
)


print(
    "Cross-session contamination: PASS"
)


# ==============================================================================
# 12. SCORE INTEGRITY
# ==============================================================================

dense_scores = (
    r2_dense_candidates[
        "dense_score"
    ].to_numpy(
        dtype=np.float32
    )
)


assert np.isfinite(
    dense_scores
).all()


assert (
    dense_scores
    >= -1.0001
).all()


assert (
    dense_scores
    <= 1.0001
).all()


print(
    "\nDense scores finite: PASS"
)

print(
    "Dense score range: PASS"
)


# ==============================================================================
# 13. RANK INTEGRITY
# ==============================================================================

ranks = (
    r2_dense_candidates[
        "retrieval_rank"
    ].to_numpy()
)


assert np.isfinite(
    ranks
).all()


assert (
    ranks
    >= 0
).all()


assert (
    ranks
    < EXPECTED_TOP_K
).all()


print(
    "Retrieval rank range: PASS"
)


# ==============================================================================
# 14. EXACT PER-RESPONSE TOP-K CENSUS
# ==============================================================================

candidate_counts = (
    r2_dense_candidates
    .groupby(
        "response_id",
        sort=False,
    )
    .size()
)


assert (
    len(
        candidate_counts
    )
    == EXPECTED_RESPONSES
)


expected_counts = (
    retrieval_queries_r2
    .set_index(
        "response_id"
    )
)


expected_counts[
    "session_id"
] = expected_counts[
    "session_id"
].astype(str)


session_sizes = (
    expected_counts[
        "session_id"
    ]
    .map(
        (
            r2_dense_candidates[
                [
                    "session_id",
                    "turn_uid",
                ]
            ]
            .groupby(
                "session_id"
            )
            .size()
        )
    )
)


# ------------------------------------------------------------------------------
# Cell 6 manifest already froze the exact population.
# Therefore verify every response has <= 50 candidates and the global
# population matches exactly.
# ------------------------------------------------------------------------------

assert (
    candidate_counts
    <= EXPECTED_TOP_K
).all()


assert (
    candidate_counts.sum()
    == EXPECTED_ROWS
)


print(
    "\nPer-response candidate census: PASS"
)

print(
    "Maximum candidates/response:",
    int(
        candidate_counts.max()
    ),
)


# ==============================================================================
# 15. DUPLICATE TURN CHECK WITHIN RESPONSE
# ==============================================================================

duplicate_pairs = (
    r2_dense_candidates
    .duplicated(
        subset=[
            "response_id",
            "turn_uid",
        ]
    )
)


duplicate_pair_count = int(
    duplicate_pairs.sum()
)


print(
    "\nDuplicate response/turn pairs:",
    duplicate_pair_count,
)


assert (
    duplicate_pair_count
    == 0
), (
    "Duplicate candidate turn detected "
    "within a response."
)


print(
    "Candidate identity uniqueness: PASS"
)


# ==============================================================================
# 16. MANIFEST SHA256 VERIFICATION
# ==============================================================================

print("\n" + "=" * 80)
print("SHA256 VERIFICATION")
print("=" * 80)


def sha256_file_r2(
    path,
    chunk_size=8 * 1024 * 1024,
):

    sha = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            sha.update(
                chunk
            )

    return sha.hexdigest()


reloaded_sha256 = sha256_file_r2(
    R2_DENSE_CANDIDATES_FINAL
)


manifest_sha256 = (
    r2_dense_manifest[
        "sha256"
    ]
)


print(
    "Manifest SHA256:",
    manifest_sha256,
)

print(
    "Reloaded SHA256:",
    reloaded_sha256,
)


assert (
    reloaded_sha256
    == manifest_sha256
), (
    "Frozen candidate SHA256 does not match "
    "the freeze manifest."
)


print(
    "SHA256: PASS"
)


# ==============================================================================
# 17. FINAL FROZEN STATE
# ==============================================================================

R2_DENSE_CANDIDATES_FROZEN = True
R2_DENSE_CANDIDATES_LOAD_READY = True
R2_CELL_7_READY = True


print("\n" + "=" * 80)
print("TRACE THE ACE — R2 FINAL FROZEN CANDIDATE VERIFICATION")
print("=" * 80)

print(
    "Frozen artifact:",
    R2_DENSE_CANDIDATES_FINAL,
)

print(
    "Frozen manifest:",
    R2_DENSE_FREEZE_MANIFEST,
)

print(
    "Rows:",
    f"{len(r2_dense_candidates):,}",
)

print(
    "Responses:",
    f"{unique_response_count:,}",
)

print(
    "Sessions:",
    f"{unique_session_count:,}",
)

print(
    "Cross-session rows:",
    foreign_session_rows,
)

print(
    "Duplicate response/turn pairs:",
    duplicate_pair_count,
)

print(
    "Target used:",
    r2_dense_manifest[
        "target_used"
    ],
)

print(
    "SHA256 verified:",
    True,
)

print(
    "Manifest status:",
    r2_dense_manifest[
        "status"
    ],
)

print(
    "R2_DENSE_CANDIDATES_FROZEN:",
    R2_DENSE_CANDIDATES_FROZEN,
)

print(
    "R2_DENSE_CANDIDATES_LOAD_READY:",
    R2_DENSE_CANDIDATES_LOAD_READY,
)

print(
    "R2_CELL_7_READY:",
    R2_CELL_7_READY,
)

print("=" * 80)
print(
    "R2 CELL 7 — FROZEN CANDIDATE VERIFICATION: PASS"
)
print("=" * 80)


# ==============================================================================
# 18. MEMORY CLEANUP
# ==============================================================================

del r2_dense_candidates
del retrieval_queries_r2
del expected_session
del candidate_counts
del expected_counts
del session_sizes
del dense_scores
del ranks

gc.collect()


print(
    "\nR2 Cell 7 memory cleanup: PASS"
)


TRACE THE ACE — R2 DENSE RETRIEVAL
CELL 7 — FROZEN DENSE CANDIDATE RELOAD + INTEGRITY VERIFICATION

FROZEN ARTIFACT EXISTENCE
Frozen root : True
Candidates  : True
Manifest    : True

FROZEN MANIFEST
JSON valid       : PASS
Status           : FROZEN
Artifact         : r2_dense_candidates
Rows             : 1,752,048
Responses        : 35,072
Sessions         : 22,821
Top-K            : 50
Target used      : False
Cross-session    : False

FROZEN PARQUET SCHEMA
Columns: ['response_id', 'session_id', 'objective_uid', 'fold', 'turn_uid', 'role', 'turn_index', 'dense_score', 'retrieval_rank', 'text_norm']
Schema: PASS

Parquet metadata rows: 1,752,048
Population metadata: PASS

RELOADING FROZEN DENSE CANDIDATES
Reloaded rows: 1,752,048

RESPONSE COVERAGE
Unique responses: 35,072
Response coverage: PASS
Unique sessions: 22,821
Session coverage: PASS

Foreign-session rows: 0
Cross-session contamination: PASS

Dense scores finite: PASS
Dense score range: PASS
Retrieval rank range: PASS

Per-

#  Do not rerun R2 Cells 3–5. If the kernel dies, we reload the frozen parquet and continue.

In [22]:
# ==============================================================================
# TRACE THE ACE — R2 DENSE RETRIEVAL
# CELL 8 — DENSE RETRIEVAL QUALITY DIAGNOSTICS
# ==============================================================================

from pathlib import Path
import json
import gc

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


print("\n" + "=" * 80)
print("TRACE THE ACE — R2 DENSE RETRIEVAL")
print("CELL 8 — DENSE RETRIEVAL QUALITY DIAGNOSTICS")
print("=" * 80)


# ==============================================================================
# 1. PROJECT PATHS
# ==============================================================================

PROJECT_ROOT = Path(
    r"D:\Competition\Trace-the-race-local"
)

SCRATCH_ROOT = (
    PROJECT_ROOT
    / "scratch_mastery_outputs"
)

R0_ROOT = (
    SCRATCH_ROOT
    / "02_retrieval"
    / "R0_input"
)

R2_ROOT = (
    SCRATCH_ROOT
    / "02_retrieval"
    / "R2_dense"
)

R2_FREEZE_ROOT = (
    R2_ROOT
    / "frozen"
)

R2_DIAGNOSTICS_ROOT = (
    R2_ROOT
    / "diagnostics"
)

R2_DIAGNOSTICS_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


R2_DENSE_CANDIDATES_FINAL = (
    R2_FREEZE_ROOT
    / "r2_dense_candidates.parquet"
)

R2_DENSE_FREEZE_MANIFEST = (
    R2_FREEZE_ROOT
    / "r2_dense_freeze_manifest.json"
)

R0_RETRIEVAL_QUERIES = (
    R0_ROOT
    / "retrieval_queries.parquet"
)


# ==============================================================================
# 2. DIAGNOSTIC ARTIFACT PATHS
# ==============================================================================

R2_RESPONSE_DIAGNOSTICS = (
    R2_DIAGNOSTICS_ROOT
    / "response_diagnostics.parquet"
)

R2_ROLE_DIAGNOSTICS = (
    R2_DIAGNOSTICS_ROOT
    / "role_diagnostics.parquet"
)

R2_RANK_DIAGNOSTICS = (
    R2_DIAGNOSTICS_ROOT
    / "rank_diagnostics.parquet"
)

R2_MANUAL_REVIEW_SAMPLE = (
    R2_DIAGNOSTICS_ROOT
    / "manual_review_sample.parquet"
)

R2_DIAGNOSTICS_MANIFEST = (
    R2_DIAGNOSTICS_ROOT
    / "r2_diagnostics_manifest.json"
)


# ==============================================================================
# 3. EXPECTED CONTRACT
# ==============================================================================

EXPECTED_ROWS = 1_752_048
EXPECTED_RESPONSES = 35_072
EXPECTED_SESSIONS = 22_821
EXPECTED_OBJECTIVES = 398
EXPECTED_TOP_K = 50

EXPECTED_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "turn_uid",
    "role",
    "turn_index",
    "dense_score",
    "retrieval_rank",
    "text_norm",
]


# ==============================================================================
# 4. FROZEN DEPENDENCY VERIFICATION
# ==============================================================================

assert R2_DENSE_CANDIDATES_FINAL.exists(), (
    "Frozen R2 dense candidate artifact is missing."
)

assert R2_DENSE_FREEZE_MANIFEST.exists(), (
    "R2 dense freeze manifest is missing."
)

assert R0_RETRIEVAL_QUERIES.exists(), (
    "R0 retrieval queries artifact is missing."
)


with open(
    R2_DENSE_FREEZE_MANIFEST,
    "r",
    encoding="utf-8",
) as f:

    freeze_manifest = json.load(f)


assert isinstance(
    freeze_manifest,
    dict,
)


assert (
    freeze_manifest.get("status")
    == "FROZEN"
), (
    "R2 dense candidate artifact is not frozen."
)


assert (
    freeze_manifest.get("target_used")
    is False
), (
    "R2 dense retrieval must remain target-blind."
)


assert (
    freeze_manifest.get(
        "cross_session_contamination"
    )
    is False
), (
    "Frozen artifact reports cross-session contamination."
)


print("\n" + "=" * 80)
print("R2 FROZEN CANDIDATE DEPENDENCY")
print("=" * 80)

print(
    "Frozen candidate:",
    True,
)

print(
    "Freeze manifest:",
    True,
)

print(
    "Manifest status:",
    freeze_manifest["status"],
)

print(
    "Target used:",
    freeze_manifest["target_used"],
)

print(
    "Cross-session contamination:",
    freeze_manifest[
        "cross_session_contamination"
    ],
)

print(
    "R2 frozen dependency: PASS"
)


# ==============================================================================
# 5. LOAD FROZEN CANDIDATES
# ==============================================================================

r2_candidates = pd.read_parquet(
    R2_DENSE_CANDIDATES_FINAL,
    engine="pyarrow",
)


assert (
    len(r2_candidates)
    == EXPECTED_ROWS
), (
    "Frozen R2 candidate population mismatch."
)


assert (
    list(r2_candidates.columns)
    == EXPECTED_COLUMNS
), (
    "Frozen R2 candidate schema mismatch."
)


print("\n" + "=" * 80)
print("INPUT ARTIFACT")
print("=" * 80)

print(
    "Candidate rows:",
    f"{len(r2_candidates):,}",
)

print(
    "Responses:",
    f"{r2_candidates['response_id'].nunique():,}",
)

print(
    "Sessions:",
    f"{r2_candidates['session_id'].nunique():,}",
)

print(
    "Objectives:",
    f"{r2_candidates['objective_uid'].nunique():,}",
)

print(
    "Folds:",
    sorted(
        r2_candidates[
            "fold"
        ]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    ),
)


assert (
    r2_candidates[
        "response_id"
    ].nunique()
    == EXPECTED_RESPONSES
)

assert (
    r2_candidates[
        "session_id"
    ].nunique()
    == EXPECTED_SESSIONS
)

assert (
    r2_candidates[
        "objective_uid"
    ].nunique()
    == EXPECTED_OBJECTIVES
)


# ==============================================================================
# 6. LOAD R0 QUERY METADATA
# ==============================================================================

query_meta = pd.read_parquet(
    R0_RETRIEVAL_QUERIES,
    engine="pyarrow",
)


assert (
    len(query_meta)
    == EXPECTED_RESPONSES
), (
    "R0 retrieval query population mismatch."
)


query_meta[
    "response_id"
] = (
    query_meta[
        "response_id"
    ].astype(str)
)

query_meta[
    "session_id"
] = (
    query_meta[
        "session_id"
    ].astype(str)
)

query_meta[
    "objective_uid"
] = (
    query_meta[
        "objective_uid"
    ].astype(str)
)


r2_candidates[
    "response_id"
] = (
    r2_candidates[
        "response_id"
    ].astype(str)
)

r2_candidates[
    "session_id"
] = (
    r2_candidates[
        "session_id"
    ].astype(str)
)

r2_candidates[
    "objective_uid"
] = (
    r2_candidates[
        "objective_uid"
    ].astype(str)
)


# ==============================================================================
# 7. DENSE SCORE DISTRIBUTION
# ==============================================================================

dense_scores = (
    r2_candidates[
        "dense_score"
    ]
    .astype(float)
)


assert np.isfinite(
    dense_scores.to_numpy()
).all()


assert (
    dense_scores.min()
    >= -1.0001
)

assert (
    dense_scores.max()
    <= 1.0001
)


score_distribution = pd.DataFrame(
    [
        {
            "metric": "dense_score_mean",
            "value": float(
                dense_scores.mean()
            ),
        },
        {
            "metric": "dense_score_median",
            "value": float(
                dense_scores.median()
            ),
        },
        {
            "metric": "dense_score_std",
            "value": float(
                dense_scores.std()
            ),
        },
        {
            "metric": "dense_score_min",
            "value": float(
                dense_scores.min()
            ),
        },
        {
            "metric": "dense_score_max",
            "value": float(
                dense_scores.max()
            ),
        },
        {
            "metric": "dense_score_q25",
            "value": float(
                dense_scores.quantile(0.25)
            ),
        },
        {
            "metric": "dense_score_q75",
            "value": float(
                dense_scores.quantile(0.75)
            ),
        },
        {
            "metric": "dense_score_q95",
            "value": float(
                dense_scores.quantile(0.95)
            ),
        },
        {
            "metric": "dense_score_q99",
            "value": float(
                dense_scores.quantile(0.99)
            ),
        },
    ]
)


print("\n" + "=" * 80)
print("DENSE SCORE DISTRIBUTION")
print("=" * 80)

print(
    score_distribution.to_string(
        index=False
    )
)


# ==============================================================================
# 8. RESPONSE-LEVEL DIAGNOSTICS
# ==============================================================================

response_score_summary = (
    r2_candidates
    .groupby(
        "response_id",
        sort=False,
    )[
        "dense_score"
    ]
    .agg(
        [
            "max",
            "mean",
            "median",
            "std",
            "min",
        ]
    )
    .reset_index()
)


response_score_summary = (
    response_score_summary
    .rename(
        columns={
            "max":
                "top_dense_score",
            "mean":
                "mean_candidate_score",
            "median":
                "median_candidate_score",
            "std":
                "candidate_score_std",
            "min":
                "min_candidate_score",
        }
    )
)


response_score_summary[
    "candidate_count"
] = (
    r2_candidates
    .groupby(
        "response_id",
        sort=False,
    )
    .size()
    .reindex(
        response_score_summary[
            "response_id"
        ]
    )
    .to_numpy()
)


response_diagnostics = (
    response_score_summary
    .merge(
        query_meta[
            [
                "response_id",
                "session_id",
                "objective_uid",
                "fold",
            ]
        ],
        on="response_id",
        how="left",
        validate="one_to_one",
    )
)


assert (
    len(response_diagnostics)
    == EXPECTED_RESPONSES
)


assert (
    response_diagnostics[
        "session_id"
    ].notna().all()
)


assert (
    response_diagnostics[
        "objective_uid"
    ].notna().all()
)


response_diagnostics.to_parquet(
    R2_RESPONSE_DIAGNOSTICS,
    engine="pyarrow",
    index=False,
)


# ==============================================================================
# 9. ROLE DISTRIBUTION
# ==============================================================================

role_diagnostics = (
    r2_candidates
    .groupby(
        "role",
        dropna=False,
    )
    .size()
    .reset_index(
        name="rows"
    )
)


role_diagnostics[
    "share"
] = (
    role_diagnostics[
        "rows"
    ]
    / len(r2_candidates)
)


print("\n" + "=" * 80)
print("ROLE DISTRIBUTION")
print("=" * 80)

print(
    role_diagnostics.to_string(
        index=False
    )
)


role_diagnostics.to_parquet(
    R2_ROLE_DIAGNOSTICS,
    engine="pyarrow",
    index=False,
)


# ==============================================================================
# 10. RANK-WISE SCORE SUMMARY
# ==============================================================================

rank_diagnostics = (
    r2_candidates
    .groupby(
        "retrieval_rank",
        sort=True,
    )[
        "dense_score"
    ]
    .agg(
        [
            "count",
            "mean",
            "median",
            "min",
            "max",
        ]
    )
    .reset_index()
)


rank_diagnostics = (
    rank_diagnostics
    .rename(
        columns={
            "count": "rows",
            "mean": "mean_dense",
            "median": "median_dense",
            "min": "min_dense",
            "max": "max_dense",
        }
    )
)


rank_diagnostics[
    "responses"
] = (
    r2_candidates
    .groupby(
        "retrieval_rank"
    )[
        "response_id"
    ]
    .nunique()
    .reindex(
        rank_diagnostics[
            "retrieval_rank"
        ]
    )
    .to_numpy()
)


print("\n" + "=" * 80)
print("RANK-WISE SCORE SUMMARY")
print("=" * 80)

print(
    rank_diagnostics.head(
        10
    ).to_string(
        index=False
    )
)


rank_diagnostics.to_parquet(
    R2_RANK_DIAGNOSTICS,
    engine="pyarrow",
    index=False,
)


# ==============================================================================
# 11. TOP-K CENSUS
# ==============================================================================

candidate_counts = (
    r2_candidates
    .groupby(
        "response_id",
        sort=False,
    )
    .size()
)


assert (
    len(candidate_counts)
    == EXPECTED_RESPONSES
)


assert (
    candidate_counts.max()
    <= EXPECTED_TOP_K
)


assert (
    candidate_counts.sum()
    == EXPECTED_ROWS
)


print("\n" + "=" * 80)
print("TOP-K CENSUS")
print("=" * 80)

print(
    "Responses:",
    f"{len(candidate_counts):,}",
)

print(
    "Maximum candidates/response:",
    int(
        candidate_counts.max()
    ),
)

print(
    "Minimum candidates/response:",
    int(
        candidate_counts.min()
    ),
)

print(
    "Candidate census: PASS"
)


# ==============================================================================
# 12. RANK MONOTONICITY
# ==============================================================================

rank_order_failures = 0


for (
    response_id,
    group,
) in r2_candidates.groupby(
    "response_id",
    sort=False,
):

    ordered = (
        group
        .sort_values(
            "retrieval_rank"
        )[
            "dense_score"
        ]
        .to_numpy()
    )


    if len(ordered) > 1:

        if not np.all(
            ordered[:-1]
            >=
            ordered[1:] - 1e-7
        ):

            rank_order_failures += 1


assert (
    rank_order_failures
    == 0
), (
    "Dense retrieval rank monotonicity failed."
)


print("\n" + "=" * 80)
print("RANK MONOTONICITY")
print("=" * 80)

print(
    "Rank-order failures:",
    rank_order_failures,
)

print(
    "Rank monotonicity: PASS"
)


# ==============================================================================
# 13. TOP-10 SUMMARY
# ==============================================================================

top_rank_summary = pd.DataFrame(
    [
        {
            "retrieval_rank": rank,
            "rows":
                int(
                    (
                        r2_candidates[
                            "retrieval_rank"
                        ]
                        == rank
                    ).sum()
                ),
            "responses":
                int(
                    r2_candidates.loc[
                        r2_candidates[
                            "retrieval_rank"
                        ]
                        == rank,
                        "response_id",
                    ].nunique()
                ),
            "mean_dense":
                float(
                    r2_candidates.loc[
                        r2_candidates[
                            "retrieval_rank"
                        ]
                        == rank,
                        "dense_score",
                    ].mean()
                ),
            "median_dense":
                float(
                    r2_candidates.loc[
                        r2_candidates[
                            "retrieval_rank"
                        ]
                        == rank,
                        "dense_score",
                    ].median()
                ),
        }
        for rank in range(
            10
        )
    ]
)


print("\n" + "=" * 80)
print("TOP-10 DENSE RETRIEVAL SUMMARY")
print("=" * 80)

print(
    top_rank_summary.to_string(
        index=False
    )
)


# ==============================================================================
# 14. TOP RETRIEVED TEXT REPETITION
# ==============================================================================

top10 = (
    r2_candidates[
        r2_candidates[
            "retrieval_rank"
        ]
        < 10
    ]
    .copy()
)


text_counts = (
    top10[
        "text_norm"
    ]
    .fillna("")
    .astype(str)
    .value_counts()
    .head(30)
    .reset_index()
)


text_counts.columns = [
    "text_norm",
    "count",
]


print("\n" + "=" * 80)
print("TOP RETRIEVED TEXT REPETITION")
print("=" * 80)

print(
    text_counts.to_string(
        index=False
    )
)


# ==============================================================================
# 15. MANUAL REVIEW SAMPLE
# ==============================================================================

MANUAL_REVIEW_RESPONSES = 100
MANUAL_REVIEW_TOP_K = 10


manual_review_response_ids = (
    response_diagnostics[
        "response_id"
    ]
    .sort_values()
    .drop_duplicates()
    .head(
        MANUAL_REVIEW_RESPONSES
    )
    .tolist()
)


manual_review = (
    r2_candidates[
        r2_candidates[
            "response_id"
        ].isin(
            manual_review_response_ids
        )
        &
        (
            r2_candidates[
                "retrieval_rank"
            ]
            <
            MANUAL_REVIEW_TOP_K
        )
    ]
    .copy()
)


manual_review = (
    manual_review
    .sort_values(
        [
            "response_id",
            "retrieval_rank",
        ]
    )
)


assert (
    manual_review[
        "response_id"
    ].nunique()
    <= MANUAL_REVIEW_RESPONSES
)


assert (
    len(manual_review)
    <=
    (
        MANUAL_REVIEW_RESPONSES
        *
        MANUAL_REVIEW_TOP_K
    )
)


manual_review.to_parquet(
    R2_MANUAL_REVIEW_SAMPLE,
    engine="pyarrow",
    index=False,
)


print("\n" + "=" * 80)
print("MANUAL REVIEW SAMPLE")
print("=" * 80)

print(
    "Responses represented:",
    manual_review[
        "response_id"
    ].nunique(),
)

print(
    "Rows:",
    len(manual_review),
)

print(
    manual_review.head(
        20
    ).to_string(
        index=False
    )
)


# ==============================================================================
# 16. SESSION CONTAMINATION
# ==============================================================================

response_to_session = dict(
    zip(
        query_meta[
            "response_id"
        ],
        query_meta[
            "session_id"
        ],
    )
)


expected_sessions = (
    r2_candidates[
        "response_id"
    ]
    .map(
        response_to_session
    )
)


foreign_session_rows = int(
    (
        r2_candidates[
            "session_id"
        ]
        !=
        expected_sessions
    ).sum()
)


assert (
    foreign_session_rows
    == 0
), (
    "Cross-session contamination detected."
)


responses_with_multiple_sessions = (
    r2_candidates
    .groupby(
        "response_id"
    )[
        "session_id"
    ]
    .nunique()
)


multi_session_responses = int(
    (
        responses_with_multiple_sessions
        > 1
    ).sum()
)


assert (
    multi_session_responses
    == 0
), (
    "Some responses contain multiple sessions."
)


print("\n" + "=" * 80)
print("SESSION CONTAMINATION")
print("=" * 80)

print(
    "Foreign-session rows:",
    foreign_session_rows,
)

print(
    "Responses with multiple candidate sessions:",
    multi_session_responses,
)

print(
    "Cross-session contamination: PASS"
)


# ==============================================================================
# 17. TARGET ISOLATION
# ==============================================================================

assert (
    "target"
    not in r2_candidates.columns
)

assert (
    "target"
    not in manual_review.columns
)

assert (
    "target"
    not in response_diagnostics.columns
)


print("\n" + "=" * 80)
print("TARGET ISOLATION")
print("=" * 80)

print(
    "Target column present:",
    False,
)

print(
    "Target used:",
    False,
)

print(
    "Target isolation: PASS"
)


# ==============================================================================
# 18. OBJECTIVE COVERAGE
# ==============================================================================

candidate_objectives = set(
    r2_candidates[
        "objective_uid"
    ]
)

query_objectives = set(
    query_meta[
        "objective_uid"
    ]
)


missing_objectives = (
    query_objectives
    -
    candidate_objectives
)


assert (
    len(missing_objectives)
    == 0
), (
    "Some query objectives have no dense candidates."
)


print("\n" + "=" * 80)
print("OBJECTIVE COVERAGE")
print("=" * 80)

print(
    "Query objectives:",
    len(query_objectives),
)

print(
    "Candidate objectives:",
    len(candidate_objectives),
)

print(
    "Missing objectives:",
    len(missing_objectives),
)

print(
    "Objective coverage: PASS"
)


# ==============================================================================
# 19. RESPONSE IDENTITY
# ==============================================================================

query_responses = set(
    query_meta[
        "response_id"
    ]
)

candidate_responses = set(
    r2_candidates[
        "response_id"
    ]
)


missing_responses = (
    query_responses
    -
    candidate_responses
)

unexpected_responses = (
    candidate_responses
    -
    query_responses
)


assert (
    len(missing_responses)
    == 0
)

assert (
    len(unexpected_responses)
    == 0
)


print("\n" + "=" * 80)
print("RESPONSE IDENTITY")
print("=" * 80)

print(
    "Missing responses:",
    len(missing_responses),
)

print(
    "Unexpected responses:",
    len(unexpected_responses),
)

print(
    "Response identity: PASS"
)


# ==============================================================================
# 20. DIAGNOSTIC MANIFEST
# ==============================================================================

diagnostic_manifest = {
    "artifact":
        "r2_dense_retrieval_diagnostics",

    "status":
        "READY",

    "source_artifact":
        "r2_dense_candidates",

    "candidate_rows":
        int(
            len(r2_candidates)
        ),

    "responses":
        int(
            r2_candidates[
                "response_id"
            ].nunique()
        ),

    "sessions":
        int(
            r2_candidates[
                "session_id"
            ].nunique()
        ),

    "objectives":
        int(
            r2_candidates[
                "objective_uid"
            ].nunique()
        ),

    "folds":
        sorted(
            r2_candidates[
                "fold"
            ]
            .dropna()
            .astype(int)
            .unique()
            .tolist()
        ),

    "manual_review_responses":
        int(
            manual_review[
                "response_id"
            ].nunique()
        ),

    "manual_review_rows":
        int(
            len(manual_review)
        ),

    "foreign_session_rows":
        int(
            foreign_session_rows
        ),

    "multi_session_responses":
        int(
            multi_session_responses
        ),

    "target_used":
        False,

    "retrieval_recomputed":
        False,

    "rank_monotonicity_failures":
        int(
            rank_order_failures
        ),

    "diagnostic_artifacts_written":
        True,
}


with open(
    R2_DIAGNOSTICS_MANIFEST,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        diagnostic_manifest,
        f,
        indent=2,
        ensure_ascii=False,
    )


assert (
    R2_DIAGNOSTICS_MANIFEST.exists()
)


# ==============================================================================
# 21. FINAL STATUS
# ==============================================================================

R2_DIAGNOSTICS_READY = True
R2_CELL_8_READY = True


print("\n" + "=" * 80)
print("R2 CELL 8 STATUS")
print("=" * 80)

print(
    "Frozen candidate rows:",
    f"{len(r2_candidates):,}",
)

print(
    "Responses diagnosed:",
    f"{r2_candidates['response_id'].nunique():,}",
)

print(
    "Objectives represented:",
    f"{r2_candidates['objective_uid'].nunique():,}",
)

print(
    "Folds:",
    sorted(
        r2_candidates[
            "fold"
        ]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    ),
)

print(
    "Manual review responses:",
    manual_review[
        "response_id"
    ].nunique(),
)

print(
    "Manual review rows:",
    len(manual_review),
)

print(
    "Target used:",
    False,
)

print(
    "Retrieval recomputed:",
    False,
)

print(
    "Cross-session contamination:",
    foreign_session_rows,
)

print(
    "Rank monotonicity failures:",
    rank_order_failures,
)

print(
    "Diagnostic artifacts written:",
    True,
)

print(
    "R2_DIAGNOSTICS_READY:",
    R2_DIAGNOSTICS_READY,
)

print(
    "R2_CELL_8_READY:",
    R2_CELL_8_READY,
)

print("=" * 80)
print(
    "R2 CELL 8 — RETRIEVAL DIAGNOSTICS: PASS"
)
print("=" * 80)


# ==============================================================================
# 22. MEMORY CLEANUP
# ==============================================================================

del r2_candidates
del query_meta
del response_score_summary
del response_diagnostics
del role_diagnostics
del rank_diagnostics
del manual_review
del top10
del dense_scores
del expected_sessions
del candidate_counts
del response_to_session
del missing_objectives
del missing_responses
del unexpected_responses
del candidate_objectives
del query_objectives
del responses_with_multiple_sessions

gc.collect()


print(
    "\nR2 Cell 8 memory cleanup: PASS"
)


TRACE THE ACE — R2 DENSE RETRIEVAL
CELL 8 — DENSE RETRIEVAL QUALITY DIAGNOSTICS

R2 FROZEN CANDIDATE DEPENDENCY
Frozen candidate: True
Freeze manifest: True
Manifest status: FROZEN
Target used: False
Cross-session contamination: False
R2 frozen dependency: PASS

INPUT ARTIFACT
Candidate rows: 1,752,048
Responses: 35,072
Sessions: 22,821
Objectives: 398
Folds: [0, 1, 2, 3, 4]

DENSE SCORE DISTRIBUTION
            metric     value
  dense_score_mean  0.403632
dense_score_median  0.395071
   dense_score_std  0.143576
   dense_score_min -0.154894
   dense_score_max  1.000000
   dense_score_q25  0.302311
   dense_score_q75  0.491252
   dense_score_q95  0.657265
   dense_score_q99  0.787096

ROLE DISTRIBUTION
      role    rows    share
background   58011 0.033110
   student  612258 0.349453
     tutor 1081779 0.617437

RANK-WISE SCORE SUMMARY
 retrieval_rank  rows  mean_dense  median_dense  min_dense  max_dense  responses
              0 35072    0.750177      0.747486   0.014754   1.00000

In [24]:
# ==============================================================================
# TRACE THE ACE — R2 DENSE RETRIEVAL
# CELL 9 — FINAL AUDIT + MANIFEST + FREEZE GATE
# ==============================================================================

from pathlib import Path
import json
import hashlib
import gc

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


print("\n" + "=" * 80)
print("TRACE THE ACE — R2 DENSE RETRIEVAL")
print("CELL 9 — FINAL AUDIT + MANIFEST + FREEZE GATE")
print("=" * 80)


# ==============================================================================
# 1. PROJECT PATHS
# ==============================================================================

PROJECT_ROOT = Path(
    r"D:\Competition\Trace-the-race-local"
)

SCRATCH_ROOT = (
    PROJECT_ROOT
    / "scratch_mastery_outputs"
)

R0_ROOT = (
    SCRATCH_ROOT
    / "02_retrieval"
    / "R0_input"
)

R1_ROOT = (
    SCRATCH_ROOT
    / "02_retrieval"
    / "R1_sparse"
)

R2_ROOT = (
    SCRATCH_ROOT
    / "02_retrieval"
    / "R2_dense"
)

R2_FREEZE_ROOT = (
    R2_ROOT
    / "frozen"
)

R2_DIAGNOSTICS_ROOT = (
    R2_ROOT
    / "diagnostics"
)


# ==============================================================================
# 2. R2 ARTIFACT PATHS
# ==============================================================================

R2_DENSE_CANDIDATES_FINAL = (
    R2_FREEZE_ROOT
    / "r2_dense_candidates.parquet"
)

R2_DENSE_FREEZE_MANIFEST = (
    R2_FREEZE_ROOT
    / "r2_dense_freeze_manifest.json"
)

R2_DIAGNOSTICS_MANIFEST = (
    R2_DIAGNOSTICS_ROOT
    / "r2_diagnostics_manifest.json"
)

R2_RESPONSE_DIAGNOSTICS = (
    R2_DIAGNOSTICS_ROOT
    / "response_diagnostics.parquet"
)

R2_ROLE_DIAGNOSTICS = (
    R2_DIAGNOSTICS_ROOT
    / "role_diagnostics.parquet"
)

R2_RANK_DIAGNOSTICS = (
    R2_DIAGNOSTICS_ROOT
    / "rank_diagnostics.parquet"
)

R2_MANUAL_REVIEW_SAMPLE = (
    R2_DIAGNOSTICS_ROOT
    / "manual_review_sample.parquet"
)

R0_RETRIEVAL_QUERIES = (
    R0_ROOT
    / "retrieval_queries.parquet"
)

R0_SESSION_TURN_INDEX = (
    R0_ROOT
    / "session_turn_index.parquet"
)

R0_OBJECTIVE_CATALOGUE = (
    R0_ROOT
    / "objective_catalogue.parquet"
)

R2_FINAL_MANIFEST = (
    R2_ROOT
    / "r2_manifest.json"
)


# ==============================================================================
# 3. EXPECTED CONTRACT
# ==============================================================================

EXPECTED_CANDIDATE_ROWS = 1_752_048
EXPECTED_RESPONSES = 35_072
EXPECTED_SESSIONS = 22_821
EXPECTED_OBJECTIVES = 398
EXPECTED_TOP_K = 50

EXPECTED_R2_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "turn_uid",
    "role",
    "turn_index",
    "dense_score",
    "retrieval_rank",
    "text_norm",
]


# ==============================================================================
# 4. ARTIFACT EXISTENCE
# ==============================================================================

artifact_paths = {
    "r2_dense_candidates":
        R2_DENSE_CANDIDATES_FINAL,

    "r2_freeze_manifest":
        R2_DENSE_FREEZE_MANIFEST,

    "r2_diagnostics_manifest":
        R2_DIAGNOSTICS_MANIFEST,

    "r2_response_diagnostics":
        R2_RESPONSE_DIAGNOSTICS,

    "r2_role_diagnostics":
        R2_ROLE_DIAGNOSTICS,

    "r2_rank_diagnostics":
        R2_RANK_DIAGNOSTICS,

    "r2_manual_review_sample":
        R2_MANUAL_REVIEW_SAMPLE,

    "r0_retrieval_queries":
        R0_RETRIEVAL_QUERIES,

    "r0_session_turn_index":
        R0_SESSION_TURN_INDEX,

    "r0_objective_catalogue":
        R0_OBJECTIVE_CATALOGUE,
}


print("\n" + "=" * 80)
print("R2 ARTIFACT EXISTENCE")
print("=" * 80)


artifact_existence = {}

for name, path in artifact_paths.items():

    exists = path.exists()

    artifact_existence[name] = exists

    print(
        f"{name:<30}: {exists}"
    )

    assert exists, (
        f"Missing required artifact:\n{path}"
    )


# ==============================================================================
# 5. LOAD AND VERIFY R2 FREEZE MANIFEST
# ==============================================================================

with open(
    R2_DENSE_FREEZE_MANIFEST,
    "r",
    encoding="utf-8",
) as f:

    freeze_manifest = json.load(f)


assert isinstance(
    freeze_manifest,
    dict,
)

assert (
    freeze_manifest.get(
        "status"
    )
    == "FROZEN"
)

assert (
    freeze_manifest.get(
        "target_used"
    )
    is False
)

assert (
    freeze_manifest.get(
        "cross_session_contamination"
    )
    is False
)


print("\n" + "=" * 80)
print("R2 FREEZE MANIFEST")
print("=" * 80)

print(
    "JSON valid       : PASS"
)

print(
    "Status           :",
    freeze_manifest.get(
        "status"
    ),
)

print(
    "Artifact         :",
    freeze_manifest.get(
        "artifact"
    ),
)

print(
    "Target used      :",
    freeze_manifest.get(
        "target_used"
    ),
)

print(
    "Cross-session    :",
    freeze_manifest.get(
        "cross_session_contamination"
    ),
)


# ==============================================================================
# 6. LOAD AND VERIFY DIAGNOSTIC MANIFEST
# ==============================================================================

with open(
    R2_DIAGNOSTICS_MANIFEST,
    "r",
    encoding="utf-8",
) as f:

    diagnostics_manifest = json.load(
        f
    )


assert isinstance(
    diagnostics_manifest,
    dict,
)

assert (
    diagnostics_manifest.get(
        "status"
    )
    == "READY"
)

assert (
    diagnostics_manifest.get(
        "target_used"
    )
    is False
)

assert (
    diagnostics_manifest.get(
        "retrieval_recomputed"
    )
    is False
)

assert (
    diagnostics_manifest.get(
        "foreign_session_rows"
    )
    == 0
)

assert (
    diagnostics_manifest.get(
        "multi_session_responses"
    )
    == 0
)

assert (
    diagnostics_manifest.get(
        "rank_monotonicity_failures"
    )
    == 0
)


print("\n" + "=" * 80)
print("R2 DIAGNOSTIC MANIFEST")
print("=" * 80)

print(
    "JSON valid       : PASS"
)

print(
    "Status           :",
    diagnostics_manifest.get(
        "status"
    ),
)

print(
    "Target used      :",
    diagnostics_manifest.get(
        "target_used"
    ),
)

print(
    "Retrieval recomputed:",
    diagnostics_manifest.get(
        "retrieval_recomputed"
    ),
)

print(
    "Foreign-session rows:",
    diagnostics_manifest.get(
        "foreign_session_rows"
    ),
)

print(
    "Multi-session responses:",
    diagnostics_manifest.get(
        "multi_session_responses"
    ),
)

print(
    "Rank monotonicity failures:",
    diagnostics_manifest.get(
        "rank_monotonicity_failures"
    ),
)


# ==============================================================================
# 7. RELOAD FROZEN CANDIDATES
# ==============================================================================

r2_candidates = pd.read_parquet(
    R2_DENSE_CANDIDATES_FINAL,
    engine="pyarrow",
)


print("\n" + "=" * 80)
print("R2 FROZEN CANDIDATE RELOAD")
print("=" * 80)

print(
    "Rows:",
    f"{len(r2_candidates):,}",
)

print(
    "Columns:",
    len(
        r2_candidates.columns
    ),
)


# ==============================================================================
# 8. POPULATION AUDIT
# ==============================================================================

assert (
    len(r2_candidates)
    == EXPECTED_CANDIDATE_ROWS
), (
    "R2 candidate row count mismatch."
)


response_count = (
    r2_candidates[
        "response_id"
    ].nunique()
)

session_count = (
    r2_candidates[
        "session_id"
    ].nunique()
)

objective_count = (
    r2_candidates[
        "objective_uid"
    ].nunique()
)


assert (
    response_count
    == EXPECTED_RESPONSES
)

assert (
    session_count
    == EXPECTED_SESSIONS
)

assert (
    objective_count
    == EXPECTED_OBJECTIVES
)


print("\n" + "=" * 80)
print("R2 POPULATION AUDIT")
print("=" * 80)

print(
    "Candidate rows :",
    f"{len(r2_candidates):,}",
)

print(
    "Expected        :",
    f"{EXPECTED_CANDIDATE_ROWS:,}",
)

print(
    "Responses       :",
    f"{response_count:,}",
)

print(
    "Expected        :",
    f"{EXPECTED_RESPONSES:,}",
)

print(
    "Sessions        :",
    f"{session_count:,}",
)

print(
    "Expected        :",
    f"{EXPECTED_SESSIONS:,}",
)

print(
    "Objectives      :",
    f"{objective_count:,}",
)

print(
    "Expected        :",
    f"{EXPECTED_OBJECTIVES:,}",
)


# ==============================================================================
# 9. SCHEMA AUDIT
# ==============================================================================

assert (
    list(r2_candidates.columns)
    == EXPECTED_R2_COLUMNS
), (
    "R2 frozen candidate schema mismatch."
)


print("\n" + "=" * 80)
print("R2 SCHEMA AUDIT")
print("=" * 80)

print(
    "Schema columns:",
    list(
        r2_candidates.columns
    ),
)

print(
    "Schema valid: PASS"
)


# ==============================================================================
# 10. NULL / IDENTITY AUDIT
# ==============================================================================

identity_columns = [
    "response_id",
    "session_id",
    "objective_uid",
    "turn_uid",
    "role",
    "turn_index",
    "retrieval_rank",
]


null_failures = {}

for column in identity_columns:

    null_count = int(
        r2_candidates[
            column
        ].isna().sum()
    )

    null_failures[
        column
    ] = null_count

    assert (
        null_count == 0
    ), (
        f"Null values found in {column}."
    )


print("\n" + "=" * 80)
print("R2 IDENTITY / NULL AUDIT")
print("=" * 80)

print(
    "Null identity failures:",
    sum(
        null_failures.values()
    ),
)

print(
    "Identity null audit: PASS"
)


# ==============================================================================
# 11. RESPONSE IDENTITY AGAINST R0
# ==============================================================================

r0_queries = pd.read_parquet(
    R0_RETRIEVAL_QUERIES,
    engine="pyarrow",
)


assert (
    len(r0_queries)
    == EXPECTED_RESPONSES
)


r0_response_ids = set(
    r0_queries[
        "response_id"
    ]
    .astype(str)
)

r2_response_ids = set(
    r2_candidates[
        "response_id"
    ]
    .astype(str)
)


missing_response_ids = (
    r0_response_ids
    -
    r2_response_ids
)

unexpected_response_ids = (
    r2_response_ids
    -
    r0_response_ids
)


assert (
    len(missing_response_ids)
    == 0
)

assert (
    len(unexpected_response_ids)
    == 0
)


print("\n" + "=" * 80)
print("RESPONSE COVERAGE")
print("=" * 80)

print(
    "R0 responses:",
    len(r0_response_ids),
)

print(
    "R2 responses:",
    len(r2_response_ids),
)

print(
    "Missing responses:",
    len(missing_response_ids),
)

print(
    "Unexpected responses:",
    len(unexpected_response_ids),
)

print(
    "Response coverage: PASS"
)


# ==============================================================================
# 12. TOP-K / RANK AUDIT
# ==============================================================================

candidate_counts = (
    r2_candidates
    .groupby(
        "response_id",
        sort=False,
    )
    .size()
)


assert (
    len(candidate_counts)
    == EXPECTED_RESPONSES
)

assert (
    candidate_counts.min()
    >= 1
)

assert (
    candidate_counts.max()
    <= EXPECTED_TOP_K
)


rank_min = int(
    r2_candidates[
        "retrieval_rank"
    ].min()
)

rank_max = int(
    r2_candidates[
        "retrieval_rank"
    ].max()
)


assert (
    rank_min
    >= 0
)

assert (
    rank_max
    <
    EXPECTED_TOP_K
)


rank_order_failures = 0


for (
    response_id,
    group,
) in r2_candidates.groupby(
    "response_id",
    sort=False,
):

    ordered = (
        group
        .sort_values(
            "retrieval_rank"
        )[
            "dense_score"
        ]
        .to_numpy()
    )

    if len(ordered) > 1:

        if not np.all(
            ordered[:-1]
            >=
            ordered[1:]
            - 1e-7
        ):

            rank_order_failures += 1


assert (
    rank_order_failures
    == 0
)


print("\n" + "=" * 80)
print("R2 TOP-K / RANK AUDIT")
print("=" * 80)

print(
    "Responses:",
    len(candidate_counts),
)

print(
    "Minimum candidates:",
    int(
        candidate_counts.min()
    ),
)

print(
    "Maximum candidates:",
    int(
        candidate_counts.max()
    ),
)

print(
    "Rank range:",
    f"{rank_min} → {rank_max}",
)

print(
    "Rank ordering failures:",
    rank_order_failures,
)

print(
    "Top-K / rank audit: PASS"
)


# ==============================================================================
# 13. SCORE VALIDITY
# ==============================================================================

dense_scores = (
    r2_candidates[
        "dense_score"
    ]
    .astype(float)
)


assert np.isfinite(
    dense_scores.to_numpy()
).all()


assert (
    dense_scores.min()
    >= -1.0001
)

assert (
    dense_scores.max()
    <= 1.0001
)


print("\n" + "=" * 80)
print("R2 SCORE AUDIT")
print("=" * 80)

print(
    "Finite scores: PASS"
)

print(
    "Score range:",
    float(
        dense_scores.min()
    ),
    "→",
    float(
        dense_scores.max()
    ),
)

print(
    "Score validity: PASS"
)


# ==============================================================================
# 14. SESSION CONTAMINATION AUDIT
# ==============================================================================

r0_queries[
    "response_id"
] = (
    r0_queries[
        "response_id"
    ].astype(str)
)

r0_queries[
    "session_id"
] = (
    r0_queries[
        "session_id"
    ].astype(str)
)


response_to_session = dict(
    zip(
        r0_queries[
            "response_id"
        ],
        r0_queries[
            "session_id"
        ],
    )
)


expected_sessions = (
    r2_candidates[
        "response_id"
    ]
    .map(
        response_to_session
    )
)


foreign_session_rows = int(
    (
        r2_candidates[
            "session_id"
        ]
        !=
        expected_sessions
    ).sum()
)


assert (
    foreign_session_rows
    == 0
)


multi_session_responses = int(
    (
        r2_candidates
        .groupby(
            "response_id"
        )[
            "session_id"
        ]
        .nunique()
        > 1
    ).sum()
)


assert (
    multi_session_responses
    == 0
)


print("\n" + "=" * 80)
print("SESSION ISOLATION")
print("=" * 80)

print(
    "Foreign-session rows:",
    foreign_session_rows,
)

print(
    "Multi-session responses:",
    multi_session_responses,
)

print(
    "Session isolation: PASS"
)


# ==============================================================================
# 15. TARGET ISOLATION
# ==============================================================================

assert (
    "target"
    not in r2_candidates.columns
)

assert (
    "target"
    not in r0_queries.columns
)


print("\n" + "=" * 80)
print("TARGET ISOLATION")
print("=" * 80)

print(
    "Target present in R2 candidates:",
    False,
)

print(
    "Target used:",
    False,
)

print(
    "Target isolation: PASS"
)


# ==============================================================================
# 16. DIAGNOSTIC ARTIFACT VERIFICATION
# ==============================================================================

diagnostic_paths = [
    R2_RESPONSE_DIAGNOSTICS,
    R2_ROLE_DIAGNOSTICS,
    R2_RANK_DIAGNOSTICS,
    R2_MANUAL_REVIEW_SAMPLE,
]


for path in diagnostic_paths:

    assert path.exists(), (
        f"Missing diagnostic artifact:\n{path}"
    )


diagnostic_response_rows = len(
    pd.read_parquet(
        R2_RESPONSE_DIAGNOSTICS,
        engine="pyarrow",
    )
)


assert (
    diagnostic_response_rows
    == EXPECTED_RESPONSES
)


diagnostic_role_rows = len(
    pd.read_parquet(
        R2_ROLE_DIAGNOSTICS,
        engine="pyarrow",
    )
)


diagnostic_rank_rows = len(
    pd.read_parquet(
        R2_RANK_DIAGNOSTICS,
        engine="pyarrow",
    )
)


diagnostic_manual_rows = len(
    pd.read_parquet(
        R2_MANUAL_REVIEW_SAMPLE,
        engine="pyarrow",
    )
)


print("\n" + "=" * 80)
print("DIAGNOSTIC ARTIFACT AUDIT")
print("=" * 80)

print(
    "Response diagnostics:",
    f"{diagnostic_response_rows:,}",
)

print(
    "Role diagnostics rows:",
    f"{diagnostic_role_rows:,}",
)

print(
    "Rank diagnostics rows:",
    f"{diagnostic_rank_rows:,}",
)

print(
    "Manual review rows:",
    f"{diagnostic_manual_rows:,}",
)

print(
    "Diagnostic artifacts: PASS"
)


# ==============================================================================
# 17. SHA256
# ==============================================================================

def sha256_file(path, chunk_size=8 * 1024 * 1024):

    sha256 = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            sha256.update(
                chunk
            )

    return sha256.hexdigest()


r2_candidates_sha256 = (
    sha256_file(
        R2_DENSE_CANDIDATES_FINAL
    )
)


print("\n" + "=" * 80)
print("R2 FROZEN CANDIDATE SHA256")
print("=" * 80)

print(
    "SHA256:",
    r2_candidates_sha256,
)


# ==============================================================================
# 18. FINAL R2 GATE MATRIX
# ==============================================================================

r2_gate_matrix = {
    "frozen_candidate_exists":
        True,

    "freeze_manifest_valid":
        True,

    "diagnostic_manifest_valid":
        True,

    "exact_candidate_population":
        len(r2_candidates)
        == EXPECTED_CANDIDATE_ROWS,

    "exact_response_population":
        response_count
        == EXPECTED_RESPONSES,

    "exact_session_population":
        session_count
        == EXPECTED_SESSIONS,

    "exact_objective_population":
        objective_count
        == EXPECTED_OBJECTIVES,

    "schema_valid":
        list(
            r2_candidates.columns
        )
        == EXPECTED_R2_COLUMNS,

    "identity_null_free":
        sum(
            null_failures.values()
        )
        == 0,

    "response_coverage_valid":
        len(missing_response_ids)
        == 0
        and
        len(unexpected_response_ids)
        == 0,

    "top_k_valid":
        candidate_counts.max()
        <= EXPECTED_TOP_K,

    "rank_valid":
        rank_order_failures
        == 0,

    "score_valid":
        bool(
            np.isfinite(
                dense_scores.to_numpy()
            ).all()
        ),

    "target_isolated":
        "target"
        not in r2_candidates.columns,

    "session_isolation_valid":
        foreign_session_rows
        == 0
        and
        multi_session_responses
        == 0,

    "diagnostics_written":
        all(
            path.exists()
            for path in diagnostic_paths
        ),

    "diagnostics_not_recomputed":
        diagnostics_manifest.get(
            "retrieval_recomputed"
        )
        is False,
}


assert all(
    r2_gate_matrix.values()
), (
    "R2 final gate failed."
)


print("\n" + "=" * 80)
print("R2 FINAL GATE MATRIX")
print("=" * 80)

for (
    check_name,
    passed,
) in r2_gate_matrix.items():

    print(
        f"{check_name:<35}: {passed}"
    )


print(
    "\nR2 final gate matrix: PASS"
)


# ==============================================================================
# 19. WRITE FINAL R2 MANIFEST
# ==============================================================================

r2_final_manifest = {
    "artifact":
        "r2_dense_retrieval",

    "status":
        "FROZEN",

    "ready":
        True,

    "candidate_artifact":
        str(
            R2_DENSE_CANDIDATES_FINAL
        ),

    "candidate_sha256":
        r2_candidates_sha256,

    "candidate_rows":
        int(
            len(r2_candidates)
        ),

    "responses":
        int(
            response_count
        ),

    "sessions":
        int(
            session_count
        ),

    "objectives":
        int(
            objective_count
        ),

    "top_k":
        int(
            EXPECTED_TOP_K
        ),

    "target_used":
        False,

    "cross_session_contamination":
        False,

    "foreign_session_rows":
        int(
            foreign_session_rows
        ),

    "multi_session_responses":
        int(
            multi_session_responses
        ),

    "rank_monotonicity_failures":
        int(
            rank_order_failures
        ),

    "retrieval_recomputed":
        False,

    "diagnostics_recomputed":
        False,

    "diagnostics_manifest":
        str(
            R2_DIAGNOSTICS_MANIFEST
        ),

    "freeze_manifest":
        str(
            R2_DENSE_FREEZE_MANIFEST
        ),

    "gate_matrix":
        {
            key: bool(value)
            for key, value
            in r2_gate_matrix.items()
        },
}


with open(
    R2_FINAL_MANIFEST,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        r2_final_manifest,
        f,
        indent=2,
        ensure_ascii=False,
    )


assert R2_FINAL_MANIFEST.exists()


# ==============================================================================
# 20. RELOAD FINAL MANIFEST — SERIALIZATION TEST
# ==============================================================================

with open(
    R2_FINAL_MANIFEST,
    "r",
    encoding="utf-8",
) as f:

    manifest_reload = json.load(
        f
    )


assert (
    manifest_reload[
        "status"
    ]
    == "FROZEN"
)

assert (
    manifest_reload[
        "ready"
    ]
    is True
)

assert (
    manifest_reload[
        "candidate_sha256"
    ]
    == r2_candidates_sha256
)

assert (
    manifest_reload[
        "candidate_rows"
    ]
    == EXPECTED_CANDIDATE_ROWS
)


print("\n" + "=" * 80)
print("R2 MANIFEST SERIALIZATION TEST")
print("=" * 80)

print(
    "Manifest:",
    R2_FINAL_MANIFEST,
)

print(
    "JSON:",
    "VALID",
)

print(
    "SHA256:",
    manifest_reload[
        "candidate_sha256"
    ],
)

print(
    "Serialization test: PASS"
)


# ==============================================================================
# 21. FINAL R2 FLAGS
# ==============================================================================

R2_DENSE_RETRIEVAL_READY = True
R2_FROZEN = True
R2_FINAL_AUDIT_READY = True
R2_CELL_9_READY = True


print("\n" + "=" * 80)
print("TRACE THE ACE — R2 FINAL STATUS")
print("=" * 80)

print(
    "Dense candidates:",
    f"{EXPECTED_CANDIDATE_ROWS:,}",
)

print(
    "Responses:",
    f"{EXPECTED_RESPONSES:,}",
)

print(
    "Sessions:",
    f"{EXPECTED_SESSIONS:,}",
)

print(
    "Objectives:",
    f"{EXPECTED_OBJECTIVES:,}",
)

print(
    "R2 manifest:",
    R2_FINAL_MANIFEST.exists(),
)

print(
    "Target leakage:",
    False,
)

print(
    "Cross-session contamination:",
    foreign_session_rows,
)

print(
    "Rank ordering failures:",
    rank_order_failures,
)

print(
    "R2_DENSE_RETRIEVAL_READY:",
    R2_DENSE_RETRIEVAL_READY,
)

print(
    "R2_FROZEN:",
    R2_FROZEN,
)

print(
    "R2_FINAL_AUDIT_READY:",
    R2_FINAL_AUDIT_READY,
)

print(
    "R2_CELL_9_READY:",
    R2_CELL_9_READY,
)

print("=" * 80)
print("R2 FINAL FREEZE: PASS")
print("=" * 80)


# ==============================================================================
# 22. MEMORY CLEANUP
# ==============================================================================

_cleanup_names = [
    "r2_candidates",
    "r0_queries",
    "dense_scores",
    "candidate_counts",
    "response_to_session",
    "expected_sessions",
    "response_score_summary",
    "diagnostic_paths",
    "diagnostics_manifest",
    "freeze_manifest",
    "r2_gate_matrix",
    "r2_final_manifest",
    "manifest_reload",
    "missing_response_ids",
    "unexpected_response_ids",
]

for _name in _cleanup_names:
    if _name in globals():
        del globals()[_name]

del _cleanup_names
del _name

gc.collect()

print(
    "\nR2 Cell 9 memory cleanup: PASS"
)


TRACE THE ACE — R2 DENSE RETRIEVAL
CELL 9 — FINAL AUDIT + MANIFEST + FREEZE GATE

R2 ARTIFACT EXISTENCE
r2_dense_candidates           : True
r2_freeze_manifest            : True
r2_diagnostics_manifest       : True
r2_response_diagnostics       : True
r2_role_diagnostics           : True
r2_rank_diagnostics           : True
r2_manual_review_sample       : True
r0_retrieval_queries          : True
r0_session_turn_index         : True
r0_objective_catalogue        : True

R2 FREEZE MANIFEST
JSON valid       : PASS
Status           : FROZEN
Artifact         : r2_dense_candidates
Target used      : False
Cross-session    : False

R2 DIAGNOSTIC MANIFEST
JSON valid       : PASS
Status           : READY
Target used      : False
Retrieval recomputed: False
Foreign-session rows: 0
Multi-session responses: 0
Rank monotonicity failures: 0

R2 FROZEN CANDIDATE RELOAD
Rows: 1,752,048
Columns: 10

R2 POPULATION AUDIT
Candidate rows : 1,752,048
Expected        : 1,752,048
Responses       : 35,072
Ex